# HaluRISC — Full Training Pipeline (Colab) — corrected grouped-split protocol

Runs the complete experiment protocol on a Colab GPU (T4 or better):

1. HaluEval download + prepare with a **GROUP-AWARE 70/15/15 split** (both answers of one question stay in the same partition; a leakage report is generated and asserted)
2. Full feature extraction (length, lexical, entity/NER, **NLI**, numeric, hedging, semantic) + NLI checkpoint provenance
3. XGBoost tuning (30 iters, 5-fold CV) + baselines + 3-seed protocol + early stopping
4. Platt vs isotonic calibration, ECE/Brier, McNemar, bootstrap CIs, Wilcoxon, 7-group ablations
5. SHAP global + local explanations
6. RAGTruth zero-shot external validation
7. Error analysis (10 FP + 10 FN, auto-tagged for manual review)
8. Latency/efficiency analysis
9. Optional LLM-as-judge comparison (needs OPENAI_API_KEY)
10. Artifact manifest generation (hashes, versions, hardware, split report)
11. Checkpoint all phases to **Google Drive** for crash-safe resume and download

**Before starting:** upload only this notebook. Cell 3 contains the complete runtime source tree, writes it to `/content/HaluRISC/`, and verifies SHA-256 hashes. No source zip is required.

**Runtime:** enable GPU (Runtime > Change runtime type). Use L4 for the full fresh run; cell 2 automatically selects batch 512 on L4/A100 and 256 on T4. Drive checkpoints make restarts resume without repeating completed phases. Keep only one Colab tab open.

**After the run:** download the Drive artifact package, unzip it at the repository root, run cell 7i again locally with `& .venv\Scripts\python.exe src\models\verify_artifacts.py`, then start the API if needed.


In [ ]:
# 1) Mount Google Drive (artifacts persist here across sessions)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/HaluRISC'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted at', DRIVE_DIR)


In [ ]:
# 2) Environment check: GPU must be enabled + adaptive batch size
# L4/A100 -> batch 512 (22.5+ GB VRAM); T4 -> batch 256 (16 GB, stable).
# T4 is the recommended runtime: far fewer quota disconnects than L4.
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('!! No GPU detected - enable GPU in Runtime > Change runtime type')
    raise SystemExit(1)
GPU_NAME = torch.cuda.get_device_name(0)
BATCH_SIZE = 512 if any(k in GPU_NAME for k in ('L4', 'A100', 'V100', 'L40')) else 256
print(f'GPU: {GPU_NAME} -> feature batch size {BATCH_SIZE}')


In [ ]:
# 3) SELF-CONTAINED: write the HaluRISC source from this cell (NO zip upload)
# Regenerate with: python colab/build_self_contained.py
# To patch a single file later: edit its EMBEDDED entry below and rerun
# this cell, or paste a small cell that rewrites just that file.
import base64, hashlib, os

ROOT = '/content/HaluRISC'
os.makedirs(ROOT, exist_ok=True)

EMBEDDED = {
 "src/__init__.py": "IiIiCkhhbHVSSVNDOiBDYWxpYnJhdGVkIGFuZCBFeHBsYWluYWJsZSBIYWxsdWNpbmF0aW9uIFJpc2sgUHJlZGljdGlvbiBGcmFtZXdvcmsKIiIiCg==",
 "src/api/main.py": "IiIiCkhhbHVSSVNDIEZhc3RBUEkgaW5mZXJlbmNlIHNlcnZlci4KCkVuZHBvaW50czoKICBHRVQgIC9oZWFsdGggICAtPiBzdGF0dXMsIG1vZGVsL2ZlYXR1cmUgdmVyc2lvbnMsIGFydGlmYWN0cyBsb2FkZWQKICBQT1NUIC9wcmVkaWN0ICAtPiBjYWxpYnJhdGVkIGhhbGx1Y2luYXRpb24tcmlzayBwcmVkaWN0aW9uIGZvciB7cXVlc3Rpb24sIGNvbnRleHQsIGFuc3dlcn0KICBQT1NUIC9leHBsYWluICAtPiBTSEFQIHRvcC1mZWF0dXJlIGV4cGxhbmF0aW9uIGZvciB0aGUgc2FtZSBpbnB1dHMKICBQT1NUIC9qdWRnZSAgICAtPiBMTE0tYXMtanVkZ2UgKEdQVCA1LjYgTHVuYSkgY29tcGFyaXNvbiBiYXNlbGluZQoKQm91bmRhcnkgcnVsZTogdGhlIEFQSSBMT0FEUyBhcnRpZmFjdHMgYW5kIGZlYXR1cmUgbW9kZWxzIGF0IHN0YXJ0dXA7IGl0IE5FVkVSIHRyYWlucy4KClN0YWJpbGl0eTogaGVhdnkgbW9kZWxzIChzcGFDeSwgTkxJLCBTQkVSVCkgYXJlIHByZWxvYWRlZCBhdCBzdGFydHVwLiBTZXQKSEFMVV9BUElfREVWSUNFPWNwdSAoZGVmYXVsdCkgdG8gYXZvaWQgVlJBTSBPT00gLyBkcml2ZXIgY3Jhc2hlcyBvbiBzbWFsbCBHUFVzOwpIQUxVX0FQSV9ERVZJQ0U9Y3VkYSBvcHRzIGludG8gR1BVIGluZmVyZW5jZS4gUHJlZmVyIHJ1bm5pbmcgV0lUSE9VVCAtLXJlbG9hZAoodXZpY29ybidzIGZpbGUgd2F0Y2hlciBjYW4gcmVzdGFydCB0aGUgc2VydmVyIHdoZW4gcmVwbyBmaWxlcyBjaGFuZ2UpLgoKUnVuIChyZXBvIHJvb3QsIC52ZW52KToKICBweXRob24gLW0gdXZpY29ybiBzcmMuYXBpLm1haW46YXBwIC0tcG9ydCA4MDAwCiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCB0aHJlYWRpbmcKaW1wb3J0IHRpbWUKZnJvbSBjb250ZXh0bGliIGltcG9ydCBhc3luY2NvbnRleHRtYW5hZ2VyCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgRGljdCwgTGlzdCwgT3B0aW9uYWwKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gZG90ZW52IGltcG9ydCBsb2FkX2RvdGVudgpmcm9tIGZhc3RhcGkgaW1wb3J0IEZhc3RBUEksIEhUVFBFeGNlcHRpb24KZnJvbSBmYXN0YXBpLm1pZGRsZXdhcmUuY29ycyBpbXBvcnQgQ09SU01pZGRsZXdhcmUKZnJvbSBweWRhbnRpYyBpbXBvcnQgQmFzZU1vZGVsLCBGaWVsZAoKIyBNdXN0IGJlIHNldCBiZWZvcmUgYW55IENVREEgY29udGV4dCBpcyBjcmVhdGVkIChtb2RlbCBwcmVsb2FkIGJlbG93KS4KIyBleHBhbmRhYmxlX3NlZ21lbnRzIGZpZ2h0cyBWUkFNIGZyYWdtZW50YXRpb24gb24gc21hbGwgR1BVcyAoUlRYIDMwNjAgNiBHQik7CiMgVE9LRU5JWkVSU19QQVJBTExFTElTTT1mYWxzZSBhdm9pZHMgdG9rZW5pemVyIHRocmVhZCBkZWFkbG9ja3Mgb24gV2luZG93cy4Kb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJQWVRPUkNIX0NVREFfQUxMT0NfQ09ORiIsICJleHBhbmRhYmxlX3NlZ21lbnRzOlRydWUiKQpvcy5lbnZpcm9uLnNldGRlZmF1bHQoIlRPS0VOSVpFUlNfUEFSQUxMRUxJU00iLCAiZmFsc2UiKQoKbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8sIGZvcm1hdD0iJShhc2N0aW1lKXMgLSAlKGxldmVsbmFtZSlzIC0gJShtZXNzYWdlKXMiKQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigiaGFsdXJpc2NfYXBpIikKCmxvYWRfZG90ZW52KCkgICMgcm9vdCAuZW52IChGQVNUQVBJXyosIE9QRU5BSV9BUElfS0VZLCBPUEVOQUlfTU9ERUwpCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KTU9ERUxTX0RJUiA9IFJPT1QgLyAiYXJ0aWZhY3RzIiAvICJtb2RlbHMiCgpNQVhfQU5TV0VSX0NIQVJTID0gMjAwMDAKTUFYX0NPTlRFWFRfQ0hBUlMgPSAyMDAwMAoKTU9ERUxfVkVSU0lPTiA9ICJ4Z2Jvb3N0LXYxLjAiCkZFQVRVUkVfVkVSU0lPTiA9ICJjb3Vyc2UtdjEuMCIKCiMgSGVhdnkgbW9kZWxzIChzcGFDeSArIE5MSSArIFNCRVJUKSBhcmUgcHJlbG9hZGVkIGF0IHN0YXJ0dXAgYW5kIGxvYWRlZCBsYXppbHkKIyBvbiBmaXJzdCByZXF1ZXN0IG9ubHkgaWYgc3RhcnR1cCBmYWlsZWQuIFRoZSBsb2NrIHByZXZlbnRzIGNvbmN1cnJlbnQKIyBkb3VibGUtbG9hZGluZywgd2hpY2ggcHJldmlvdXNseSBjYXVzZWQgbWVtb3J5IHNwaWtlcyBhbmQgcHJvY2VzcyBleGl0cy4KRkVBVFVSRV9NT0RFTF9MT0FEX0xPQ0sgPSB0aHJlYWRpbmcuTG9jaygpCgojIHNlbnRlbmNlLXRyYW5zZm9ybWVycyBpcyBub3QgZnVsbHkgdGhyZWFkLXNhZmUgYW5kIGNvbmN1cnJlbnQgQ1VEQSBpbmZlcmVuY2UKIyBmcm9tIHV2aWNvcm4ncyB0aHJlYWRwb29sIGNyYXNoZWQgdGhlIHByb2Nlc3MgKHNpbGVudCBleGl0KS4gQWxsIEdQVSBmZWF0dXJlCiMgZXh0cmFjdGlvbiBpcyBzZXJpYWxpemVkIHRocm91Z2ggdGhpcyBsb2NrLgpJTkZFUkVOQ0VfTE9DSyA9IHRocmVhZGluZy5Mb2NrKCkKClNUQVRFID0geyJtb2RlbCI6IE5vbmUsICJleHBsYWluZXIiOiBOb25lLCAiZmVhdHVyZV9tb2RlbHMiOiBOb25lLCAiZmVhdHVyZV9jb2xzIjogTm9uZSwgInBhcmFtcyI6IE5vbmV9CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgU2NoZW1hcyAoc3RhYmxlIEFQSSBjb250cmFjdCwgc2VlIEFHRU5UUy5tZCDCpzgpCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpjbGFzcyBBbmFseXNpc1JlcXVlc3QoQmFzZU1vZGVsKToKICAgIHF1ZXN0aW9uOiBzdHIgPSBGaWVsZCgiIiwgbWF4X2xlbmd0aD01MDAwLCBkZXNjcmlwdGlvbj0iVGhlIHF1ZXN0aW9uIHRoYXQgd2FzIGFza2VkIikKICAgIGNvbnRleHQ6IE9wdGlvbmFsW3N0cl0gPSBGaWVsZCgiIiwgbWF4X2xlbmd0aD1NQVhfQ09OVEVYVF9DSEFSUywgZGVzY3JpcHRpb249IlJlZmVyZW5jZSBjb250ZXh0L2V2aWRlbmNlIikKICAgIGFuc3dlcjogc3RyID0gRmllbGQoLi4uLCBtYXhfbGVuZ3RoPU1BWF9BTlNXRVJfQ0hBUlMsIGRlc2NyaXB0aW9uPSJDYW5kaWRhdGUgTExNIGFuc3dlciB0byBzY29yZSIpCiAgICBkb21haW46IE9wdGlvbmFsW3N0cl0gPSAicWEiCgoKY2xhc3MgRmVhdHVyZUltcGFjdChCYXNlTW9kZWwpOgogICAgZmVhdHVyZTogc3RyCiAgICB2YWx1ZTogZmxvYXQKICAgIGltcGFjdDogZmxvYXQKCgpjbGFzcyBQcmVkaWN0aW9uUmVzcG9uc2UoQmFzZU1vZGVsKToKICAgIHJpc2tfc2NvcmU6IGZsb2F0CiAgICBjYWxpYnJhdGVkX3Njb3JlOiBmbG9hdAogICAgbGFiZWw6IHN0cgogICAgdGhyZXNob2xkczogRGljdFtzdHIsIGZsb2F0XQogICAgbGF0ZW5jeV9tczogZmxvYXQKICAgIG1vZGVsX3ZlcnNpb246IHN0cgogICAgZmVhdHVyZV92ZXJzaW9uOiBzdHIKICAgIHdhcm5pbmc6IHN0cgogICAgZmVhdHVyZXM6IERpY3Rbc3RyLCBmbG9hdF0KCgpjbGFzcyBFeHBsYW5hdGlvblJlc3BvbnNlKEJhc2VNb2RlbCk6CiAgICB0b3BfZmVhdHVyZXM6IExpc3RbRmVhdHVyZUltcGFjdF0KICAgIGJhc2VfdmFsdWU6IGZsb2F0CgoKY2xhc3MgSnVkZ2VSZXF1ZXN0KEJhc2VNb2RlbCk6CiAgICBxdWVzdGlvbjogc3RyID0gIiIKICAgIGNvbnRleHQ6IE9wdGlvbmFsW3N0cl0gPSAiIgogICAgYW5zd2VyOiBzdHIgPSBGaWVsZCguLi4sIG1heF9sZW5ndGg9TUFYX0FOU1dFUl9DSEFSUykKCgpjbGFzcyBKdWRnZVJlc3BvbnNlKEJhc2VNb2RlbCk6CiAgICBqdWRnbWVudDogc3RyCiAgICBjb25maWRlbmNlOiBmbG9hdAogICAgcmVhc29uaW5nOiBzdHIKICAgIG1vZGVsOiBzdHIKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBTdGFydHVwIC8gYXJ0aWZhY3QgbG9hZGluZwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KZGVmIF9sb2FkX2NhbGlicmF0ZWRfbW9kZWwoKToKICAgICIiIkxvYWQgdGhlIHhnYitwbGF0dCBhcnRpZmFjdDsgcHJlZGljdF9wcm9iYSA9IHBsYXR0KHJhdy5wcmVkaWN0X3Byb2JhKS4iIiIKICAgIGltcG9ydCBqb2JsaWIKCiAgICBidW5kbGUgPSBqb2JsaWIubG9hZChNT0RFTFNfRElSIC8gIm1vZGVsX3hnYm9vc3RfY2FsaWJyYXRlZC5qb2JsaWIiKQogICAgaWYgaXNpbnN0YW5jZShidW5kbGUsIGRpY3QpIGFuZCBidW5kbGUuZ2V0KCJraW5kIikgPT0gInhnYitwbGF0dCI6CiAgICAgICAgcmF3LCBwbGF0dCA9IGJ1bmRsZVsibW9kZWwiXSwgYnVuZGxlWyJjYWxpYnJhdG9yIl0KCiAgICAgICAgZGVmIHByZWRpY3RfcHJvYmEoWCk6CiAgICAgICAgICAgIHAgPSByYXcucHJlZGljdF9wcm9iYShYKVs6LCAxXQogICAgICAgICAgICByZXR1cm4gcGxhdHQucHJlZGljdF9wcm9iYShwLnJlc2hhcGUoLTEsIDEpKQoKICAgICAgICByZXR1cm4geyJyYXciOiByYXcsICJwcmVkaWN0X3Byb2JhIjogcHJlZGljdF9wcm9iYX0KICAgIHJldHVybiB7InJhdyI6IGJ1bmRsZSwgInByZWRpY3RfcHJvYmEiOiBsYW1iZGEgWDogYnVuZGxlLnByZWRpY3RfcHJvYmEoWCl9CgoKZGVmIGxvYWRfYXJ0aWZhY3RzKCk6CiAgICBkZWYgX21pc3NpbmcobmFtZTogc3RyKSAtPiBib29sOgogICAgICAgIHJldHVybiBub3QgKE1PREVMU19ESVIgLyBuYW1lKS5leGlzdHMoKQoKICAgIG1pc3NpbmcgPSBbbiBmb3IgbiBpbiBbIm1vZGVsX3hnYm9vc3RfY2FsaWJyYXRlZC5qb2JsaWIiLCAibW9kZWxfeGdib29zdF9yYXcuam9ibGliIiwgImZlYXR1cmVfbmFtZXMuanNvbiIsICJwYXJhbXMuanNvbiJdIGlmIF9taXNzaW5nKG4pXQogICAgaWYgbWlzc2luZzoKICAgICAgICBsb2dnZXIud2FybmluZyhmIk1pc3NpbmcgYXJ0aWZhY3RzOiB7bWlzc2luZ30gLSBydW4gdHJhaW5pbmcgZmlyc3QgKGNvbGFiL0hhbHVSSVNDX1RyYWluaW5nX1ZlcnNpb25fQi5pcHluYikiKQogICAgICAgIHJldHVybiBGYWxzZQoKICAgIGltcG9ydCBqb2JsaWIKCiAgICBTVEFURVsibW9kZWwiXSA9IF9sb2FkX2NhbGlicmF0ZWRfbW9kZWwoKQogICAgU1RBVEVbInBhcmFtcyJdID0ganNvbi5sb2FkcygoTU9ERUxTX0RJUiAvICJwYXJhbXMuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgU1RBVEVbImZlYXR1cmVfY29scyJdID0ganNvbi5sb2FkcygoTU9ERUxTX0RJUiAvICJmZWF0dXJlX25hbWVzLmpzb24iKS5yZWFkX3RleHQoKSkKCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IGpvYmxpYgoKICAgICAgICBleHBsYWluZXJfcGF0aCA9IE1PREVMU19ESVIgLyAic2hhcF9leHBsYWluZXIuam9ibGliIgogICAgICAgIGlmIGV4cGxhaW5lcl9wYXRoLmV4aXN0cygpOgogICAgICAgICAgICBTVEFURVsiZXhwbGFpbmVyIl0gPSBqb2JsaWIubG9hZChleHBsYWluZXJfcGF0aCkKICAgICAgICAgICAgbG9nZ2VyLmluZm8oIkxvYWRlZCBzYXZlZCBTSEFQIGV4cGxhaW5lciIpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgaW1wb3J0IHNoYXAKCiAgICAgICAgICAgIHJhdyA9IFNUQVRFWyJtb2RlbCJdWyJyYXciXQogICAgICAgICAgICBTVEFURVsiZXhwbGFpbmVyIl0gPSBzaGFwLlRyZWVFeHBsYWluZXIocmF3KQogICAgICAgICAgICBsb2dnZXIuaW5mbygiQnVpbHQgU0hBUCBleHBsYWluZXIgZnJvbSByYXcgbW9kZWwiKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIGxvZ2dlci53YXJuaW5nKGYiU0hBUCBleHBsYWluZXIgbm90IGxvYWRlZDoge2V9IikKCiAgICBsb2dnZXIuaW5mbygiQXJ0aWZhY3RzIGxvYWRlZC4iKQogICAgcmV0dXJuIFRydWUKCgpkZWYgbG9hZF9mZWF0dXJlX21vZGVscygpOgogICAgIiIiVGhyZWFkLXNhZmUgbGF6eSBsb2FkIG9mIE5FUiArIE5MSSArIGVtYmVkZGluZyBtb2RlbHMuCgogICAgRWFnZXJseSBwcmVsb2FkZWQgYXQgc3RhcnR1cCAoc2VlIGxpZmVzcGFuKTsgdGhpcyBpcyBhIGZhbGxiYWNrIHRoYXQgbXVzdAogICAgbmV2ZXIgcnVuIGNvbmN1cnJlbnRseSBmcm9tIG11bHRpcGxlIHJlcXVlc3QgdGhyZWFkcy4KICAgICIiIgogICAgaWYgU1RBVEVbImZlYXR1cmVfbW9kZWxzIl0gaXMgbm90IE5vbmU6CiAgICAgICAgcmV0dXJuIFNUQVRFWyJmZWF0dXJlX21vZGVscyJdCgogICAgd2l0aCBGRUFUVVJFX01PREVMX0xPQURfTE9DSzoKICAgICAgICBpZiBTVEFURVsiZmVhdHVyZV9tb2RlbHMiXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmV0dXJuIFNUQVRFWyJmZWF0dXJlX21vZGVscyJdCgogICAgICAgIGZyb20gc3JjLmZlYXR1cmVzLmV4dHJhY3RfZmVhdHVyZXMgaW1wb3J0IGxvYWRfaGVhdnlfbW9kZWxzCgogICAgICAgIGRldmljZSA9IG9zLmVudmlyb24uZ2V0KCJIQUxVX0FQSV9ERVZJQ0UiLCAiY3B1IikKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBTVEFURVsiZmVhdHVyZV9tb2RlbHMiXSA9IGxvYWRfaGVhdnlfbW9kZWxzKGRldmljZT1kZXZpY2UpCiAgICAgICAgICAgIGxvZ2dlci5pbmZvKGYiRmVhdHVyZSBtb2RlbHMgbG9hZGVkIGluIHt0aW1lLnRpbWUoKSAtIHQwOi4xZn1zIChkZXZpY2U9e2RldmljZX0pIikKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGxvZ2dlci5lcnJvcihmIkZlYXR1cmUgbW9kZWxzIGZhaWxlZCB0byBsb2FkIChkZXZpY2U9e2RldmljZX0pOiB7ZX0iKQogICAgICAgICAgICByYWlzZQogICAgcmV0dXJuIFNUQVRFWyJmZWF0dXJlX21vZGVscyJdCgoKQGFzeW5jY29udGV4dG1hbmFnZXIKYXN5bmMgZGVmIGxpZmVzcGFuKGFwcDogRmFzdEFQSSk6CiAgICBsb2FkX2FydGlmYWN0cygpCiAgICBpZiBvcy5lbnZpcm9uLmdldCgiSEFMVV9BUElfUFJFTE9BRCIsICIxIikgIT0gIjAiOgogICAgICAgIHRyeToKICAgICAgICAgICAgbG9hZF9mZWF0dXJlX21vZGVscygpCiAgICAgICAgICAgIFNUQVRFWyJtb2RlbHNfcmVhZHkiXSA9IFRydWUKICAgICAgICAgICAgbG9nZ2VyLmluZm8oIkFsbCBtb2RlbHMgcHJlbG9hZGVkLiBBUEkgcmVhZHkuIikKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIFNUQVRFWyJtb2RlbHNfcmVhZHkiXSA9IEZhbHNlCiAgICAgICAgICAgIGxvZ2dlci53YXJuaW5nKGYiUHJlbG9hZCBvZiBoZWF2eSBmZWF0dXJlIG1vZGVscyBmYWlsZWQgKHtlfSk7IEFQSSBzdGlsbCBzZXJ2aW5nICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgZiIvaGVhbHRoIGFuZCAvanVkZ2UsIC9wcmVkaWN0IHdpbGwgcmV0cnkgb24gZmlyc3QgcmVxdWVzdC4iKQogICAgZWxzZToKICAgICAgICBsb2dnZXIuaW5mbygiSEFMVV9BUElfUFJFTE9BRD0wIC0+IGhlYXZ5IG1vZGVscyB3aWxsIGxvYWQgbGF6aWx5IG9uIGZpcnN0IC9wcmVkaWN0LiIpCiAgICB5aWVsZAogICAgU1RBVEUuY2xlYXIoKQoKCmFwcCA9IEZhc3RBUEkoCiAgICB0aXRsZT0iSGFsdVJJU0MgQVBJIiwKICAgIGRlc2NyaXB0aW9uPSJDYWxpYnJhdGVkICYgZXhwbGFpbmFibGUgaGFsbHVjaW5hdGlvbi1yaXNrIGVzdGltYXRpb24gKFZlcnNpb24gQSkiLAogICAgdmVyc2lvbj0iMS4wLjAiLAogICAgbGlmZXNwYW49bGlmZXNwYW4sCikKCmFwcC5hZGRfbWlkZGxld2FyZSgKICAgIENPUlNNaWRkbGV3YXJlLAogICAgYWxsb3dfb3JpZ2lucz1bImh0dHA6Ly9sb2NhbGhvc3Q6MzAwMCIsICJodHRwOi8vMTI3LjAuMC4xOjMwMDAiXSwKICAgIGFsbG93X2NyZWRlbnRpYWxzPVRydWUsCiAgICBhbGxvd19tZXRob2RzPVsiKiJdLAogICAgYWxsb3dfaGVhZGVycz1bIioiXSwKKQoKCmRlZiBfZmVhdHVyZV92ZWN0b3IocmVxOiBBbmFseXNpc1JlcXVlc3QpIC0+IERpY3Rbc3RyLCBmbG9hdF06CiAgICBmcm9tIHNyYy5mZWF0dXJlcy5leHRyYWN0X2ZlYXR1cmVzIGltcG9ydCBleHRyYWN0X2FsbF9mZWF0dXJlc19zaW5nbGUKCiAgICB0cnk6CiAgICAgICAgbW9kZWxzID0gbG9hZF9mZWF0dXJlX21vZGVscygpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcmFpc2UgSFRUUEV4Y2VwdGlvbihzdGF0dXNfY29kZT01MDMsIGRldGFpbD1mIkZlYXR1cmUgbW9kZWxzIHVuYXZhaWxhYmxlOiB7ZX0iKQogICAgd2l0aCBJTkZFUkVOQ0VfTE9DSzoKICAgICAgICBmZWF0cyA9IGV4dHJhY3RfYWxsX2ZlYXR1cmVzX3NpbmdsZShyZXEucXVlc3Rpb24gb3IgIiIsIHJlcS5jb250ZXh0IG9yICIiLCByZXEuYW5zd2VyLCBtb2RlbHMpCiAgICBtaXNzaW5nID0gW2MgZm9yIGMgaW4gU1RBVEVbImZlYXR1cmVfY29scyJdIGlmIGMgbm90IGluIGZlYXRzXQogICAgaWYgbWlzc2luZzoKICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKHN0YXR1c19jb2RlPTUwMCwgZGV0YWlsPWYiRmVhdHVyZSBleHRyYWN0b3IgbWlzc2luZyBjb2x1bW5zOiB7bWlzc2luZ30iKQogICAgcmV0dXJuIGZlYXRzCgoKZGVmIF9yaXNrX2xhYmVsKHA6IGZsb2F0KSAtPiBzdHI6CiAgICBpZiBwID49IDAuNzA6CiAgICAgICAgcmV0dXJuICJoaWdoX3Jpc2siCiAgICBpZiBwID49IDAuMzA6CiAgICAgICAgcmV0dXJuICJtZWRpdW1fcmlzayIKICAgIHJldHVybiAibG93X3Jpc2siCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgRW5kcG9pbnRzCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpAYXBwLmdldCgiL2hlYWx0aCIpCmRlZiBoZWFsdGhfY2hlY2soKToKICAgIGFydGlmYWN0c19vayA9IFNUQVRFWyJtb2RlbCJdIGlzIG5vdCBOb25lCiAgICByZXR1cm4gewogICAgICAgICJzdGF0dXMiOiAib2siIGlmIGFydGlmYWN0c19vayBhbmQgU1RBVEVbImZlYXR1cmVfbW9kZWxzIl0gaXMgbm90IE5vbmUgZWxzZSAiZGVncmFkZWQiLAogICAgICAgICJtb2RlbCI6IE1PREVMX1ZFUlNJT04sCiAgICAgICAgImZlYXR1cmVfdmVyc2lvbiI6IEZFQVRVUkVfVkVSU0lPTiwKICAgICAgICAiYXJ0aWZhY3RzX2xvYWRlZCI6IGFydGlmYWN0c19vaywKICAgICAgICAiZmVhdHVyZV9tb2RlbHNfcmVhZHkiOiBTVEFURVsiZmVhdHVyZV9tb2RlbHMiXSBpcyBub3QgTm9uZSwKICAgICAgICAiZXhwbGFpbmVyX3JlYWR5IjogU1RBVEVbImV4cGxhaW5lciJdIGlzIG5vdCBOb25lLAogICAgICAgICJuX2ZlYXR1cmVzIjogbGVuKFNUQVRFWyJmZWF0dXJlX2NvbHMiXSkgaWYgU1RBVEVbImZlYXR1cmVfY29scyJdIGVsc2UgMCwKICAgIH0KCgpAYXBwLnBvc3QoIi9wcmVkaWN0IiwgcmVzcG9uc2VfbW9kZWw9UHJlZGljdGlvblJlc3BvbnNlKQpkZWYgcHJlZGljdF9yaXNrKHJlcTogQW5hbHlzaXNSZXF1ZXN0KToKICAgIGlmIFNUQVRFWyJtb2RlbCJdIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgSFRUUEV4Y2VwdGlvbihzdGF0dXNfY29kZT01MDMsIGRldGFpbD0iTW9kZWwgYXJ0aWZhY3RzIG5vdCBsb2FkZWQuIFJ1biB0cmFpbmluZyAoY29sYWIvSGFsdVJJU0NfVHJhaW5pbmdfVmVyc2lvbl9CLmlweW5iKSBhbmQgcGxhY2UgYXJ0aWZhY3RzLyBpbiB0aGUgcmVwbyByb290LiIpCiAgICBpZiBub3QgcmVxLmFuc3dlci5zdHJpcCgpOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NDAwLCBkZXRhaWw9IkFuc3dlciBzdHJpbmcgY2Fubm90IGJlIGVtcHR5LiIpCgogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgZmVhdHMgPSBfZmVhdHVyZV92ZWN0b3IocmVxKQogICAgWCA9IG5wLmFycmF5KFtbZmVhdHNbY10gZm9yIGMgaW4gU1RBVEVbImZlYXR1cmVfY29scyJdXV0sIGR0eXBlPW5wLmZsb2F0NjQpCgogICAgcCA9IGZsb2F0KFNUQVRFWyJtb2RlbCJdWyJwcmVkaWN0X3Byb2JhIl0oWClbMCwgMV0pCiAgICBwID0gbWluKDAuOTk5LCBtYXgoMC4wMDEsIHApKQoKICAgIHRocmVzaG9sZHMgPSB7ImxvdyI6IDAuMzAsICJtZWRpdW0iOiAwLjcwLCAiaGlnaCI6IDEuMH0KICAgIGxhdGVuY3kgPSByb3VuZCgodGltZS50aW1lKCkgLSB0MCkgKiAxMDAwLCAyKQoKICAgIHJldHVybiBQcmVkaWN0aW9uUmVzcG9uc2UoCiAgICAgICAgcmlza19zY29yZT1yb3VuZChwLCA0KSwKICAgICAgICBjYWxpYnJhdGVkX3Njb3JlPXJvdW5kKHAsIDQpLAogICAgICAgIGxhYmVsPV9yaXNrX2xhYmVsKHApLAogICAgICAgIHRocmVzaG9sZHM9dGhyZXNob2xkcywKICAgICAgICBsYXRlbmN5X21zPWxhdGVuY3ksCiAgICAgICAgbW9kZWxfdmVyc2lvbj1NT0RFTF9WRVJTSU9OLAogICAgICAgIGZlYXR1cmVfdmVyc2lvbj1GRUFUVVJFX1ZFUlNJT04sCiAgICAgICAgd2FybmluZz0iVHJhaW5lZCBvbiBIYWx1RXZhbCBzeW50aGV0aWMgZGF0YS4gUmVzdWx0cyBtYXkgbm90IGdlbmVyYWxpemUgdG8gcmVhbC13b3JsZCBMTE0gb3V0cHV0cy4iLAogICAgICAgIGZlYXR1cmVzPXtrOiBmbG9hdCh2KSBmb3IgaywgdiBpbiBmZWF0cy5pdGVtcygpfSwKICAgICkKCgpAYXBwLnBvc3QoIi9leHBsYWluIiwgcmVzcG9uc2VfbW9kZWw9RXhwbGFuYXRpb25SZXNwb25zZSkKZGVmIGV4cGxhaW5fcmlzayhyZXE6IEFuYWx5c2lzUmVxdWVzdCk6CiAgICBpZiBTVEFURVsibW9kZWwiXSBpcyBOb25lIG9yIFNUQVRFWyJleHBsYWluZXIiXSBpcyBOb25lOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NTAzLCBkZXRhaWw9IkV4cGxhaW5lciBub3QgbG9hZGVkLiBSdW4gdHJhaW5pbmcgZmlyc3QuIikKCiAgICBmZWF0cyA9IF9mZWF0dXJlX3ZlY3RvcihyZXEpCiAgICBYID0gbnAuYXJyYXkoW1tmZWF0c1tjXSBmb3IgYyBpbiBTVEFURVsiZmVhdHVyZV9jb2xzIl1dXSwgZHR5cGU9bnAuZmxvYXQ2NCkKCiAgICBzaGFwX3ZhbHVlcyA9IFNUQVRFWyJleHBsYWluZXIiXS5zaGFwX3ZhbHVlcyhYKVswXQogICAgYmFzZV92YWx1ZSA9IGZsb2F0KFNUQVRFWyJleHBsYWluZXIiXS5leHBlY3RlZF92YWx1ZSkKICAgIG9yZGVyID0gbnAuYXJnc29ydChucC5hYnMoc2hhcF92YWx1ZXMpKVs6Oi0xXVs6NV0KCiAgICB0b3BfZmVhdHVyZXMgPSBbCiAgICAgICAgRmVhdHVyZUltcGFjdCgKICAgICAgICAgICAgZmVhdHVyZT1TVEFURVsiZmVhdHVyZV9jb2xzIl1baV0sCiAgICAgICAgICAgIHZhbHVlPXJvdW5kKGZsb2F0KFhbMCwgaV0pLCA2KSwKICAgICAgICAgICAgaW1wYWN0PXJvdW5kKGZsb2F0KHNoYXBfdmFsdWVzW2ldKSwgNiksCiAgICAgICAgKQogICAgICAgIGZvciBpIGluIG9yZGVyCiAgICBdCiAgICByZXR1cm4gRXhwbGFuYXRpb25SZXNwb25zZSh0b3BfZmVhdHVyZXM9dG9wX2ZlYXR1cmVzLCBiYXNlX3ZhbHVlPXJvdW5kKGJhc2VfdmFsdWUsIDYpKQoKCkBhcHAucG9zdCgiL2p1ZGdlIiwgcmVzcG9uc2VfbW9kZWw9SnVkZ2VSZXNwb25zZSkKZGVmIGp1ZGdlX2Fuc3dlcihyZXE6IEp1ZGdlUmVxdWVzdCk6CiAgICAiIiJMTE0tYXMtanVkZ2UgYmFzZWxpbmUgKEdQVCA1LjYgTHVuYSkuIFVzZXMgT1BFTkFJX0FQSV9LRVkgZnJvbSAuZW52LiIiIgogICAgYXBpX2tleSA9IG9zLmVudmlyb24uZ2V0KCJPUEVOQUlfQVBJX0tFWSIpCiAgICBpZiBub3QgYXBpX2tleToKICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKHN0YXR1c19jb2RlPTUwMywgZGV0YWlsPSJPUEVOQUlfQVBJX0tFWSBub3QgY29uZmlndXJlZCBpbiAuZW52IikKCiAgICB0cnk6CiAgICAgICAgZnJvbSBvcGVuYWkgaW1wb3J0IE9wZW5BSQogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NTAzLCBkZXRhaWw9Im9wZW5haSBwYWNrYWdlIG5vdCBpbnN0YWxsZWQiKQoKICAgIG1vZGVsX25hbWUgPSBvcy5lbnZpcm9uLmdldCgiT1BFTkFJX01PREVMIiwgImdwdC01LjYtbHVuYSIpCiAgICBjbGllbnQgPSBPcGVuQUkoYXBpX2tleT1hcGlfa2V5KQoKICAgIHN5c3RlbSA9ICgKICAgICAgICAiWW91IGFyZSBhbiBleHBlcnQgaGFsbHVjaW5hdGlvbi1qdWRnZS4gR2l2ZW4gYSBxdWVzdGlvbiwgYSByZWZlcmVuY2UgY29udGV4dCwgYW5kIGFuIGFuc3dlciwgIgogICAgICAgICJkZWNpZGUgd2hldGhlciB0aGUgYW5zd2VyIGNvbnRhaW5zIGhhbGx1Y2luYXRlZCBjb250ZW50ICh1bnN1cHBvcnRlZCwgY29udHJhZGljdG9yeSwgb3IgZmFicmljYXRlZCAiCiAgICAgICAgImluZm9ybWF0aW9uIHJlbGF0aXZlIHRvIHRoZSBjb250ZXh0KS4gUmVzcG9uZCB3aXRoIEpTT04gb25seTogIgogICAgICAgICd7Imp1ZGdtZW50IjogImhhbGx1Y2luYXRlZCJ8Imdyb3VuZGVkIiwgImNvbmZpZGVuY2UiOiAwLjAtMS4wLCAicmVhc29uaW5nIjogIjxzaG9ydCBleHBsYW5hdGlvbj4ifS4nCiAgICApCiAgICB1c2VyID0gKAogICAgICAgIGYiUXVlc3Rpb246IHtyZXEucXVlc3Rpb259XG4iCiAgICAgICAgZiJDb250ZXh0OiB7cmVxLmNvbnRleHQgb3IgJyhub25lKSd9XG4iCiAgICAgICAgZiJBbnN3ZXI6IHtyZXEuYW5zd2VyfSIKICAgICkKCiAgICB0cnk6CiAgICAgICAgcmVzcCA9IGNsaWVudC5jaGF0LmNvbXBsZXRpb25zLmNyZWF0ZSgKICAgICAgICAgICAgbW9kZWw9bW9kZWxfbmFtZSwKICAgICAgICAgICAgbWVzc2FnZXM9WwogICAgICAgICAgICAgICAgeyJyb2xlIjogInN5c3RlbSIsICJjb250ZW50Ijogc3lzdGVtfSwKICAgICAgICAgICAgICAgIHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiB1c2VyfSwKICAgICAgICAgICAgXSwKICAgICAgICAgICAgdGVtcGVyYXR1cmU9MCwKICAgICAgICAgICAgbWF4X3Rva2Vucz0yNTAsCiAgICAgICAgKQogICAgICAgIGNvbnRlbnQgPSByZXNwLmNob2ljZXNbMF0ubWVzc2FnZS5jb250ZW50LnN0cmlwKCkKICAgICAgICBkYXRhID0ganNvbi5sb2Fkcyhjb250ZW50W2NvbnRlbnQuZmluZCgieyIpIDogY29udGVudC5yZmluZCgifSIpICsgMV0pCiAgICAgICAgcmV0dXJuIEp1ZGdlUmVzcG9uc2UoCiAgICAgICAgICAgIGp1ZGdtZW50PWRhdGEuZ2V0KCJqdWRnbWVudCIsICJncm91bmRlZCIpLAogICAgICAgICAgICBjb25maWRlbmNlPWZsb2F0KGRhdGEuZ2V0KCJjb25maWRlbmNlIiwgMC4wKSksCiAgICAgICAgICAgIHJlYXNvbmluZz1kYXRhLmdldCgicmVhc29uaW5nIiwgIiIpLAogICAgICAgICAgICBtb2RlbD1tb2RlbF9uYW1lLAogICAgICAgICkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKHN0YXR1c19jb2RlPTUwMiwgZGV0YWlsPWYiTExNIGp1ZGdlIGZhaWxlZDoge2V9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaW1wb3J0IHV2aWNvcm4KCiAgICB1dmljb3JuLnJ1bihhcHAsIGhvc3Q9b3MuZW52aXJvbi5nZXQoIkZBU1RBUElfSE9TVCIsICIxMjcuMC4wLjEiKSwgcG9ydD1pbnQob3MuZW52aXJvbi5nZXQoIkZBU1RBUElfUE9SVCIsICI4MDAwIikpKQo=",
 "src/data/download.py": "IiIiCkRhdGEgYWNxdWlzaXRpb24gc2NyaXB0IGZvciBIYWx1UklTQy4KRG93bmxvYWRzIEhhbHVFdmFsIFFBIGRhdGFzZXQgKHFhX2RhdGEuanNvbiAtIEpTT05MIGZvcm1hdCkgZnJvbSBHaXRIdWIuCiIiIgoKaW1wb3J0IG9zCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCB1cmxsaWIucmVxdWVzdAoKbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8sIGZvcm1hdD0iJShhc2N0aW1lKXMgLSAlKGxldmVsbmFtZSlzIC0gJShtZXNzYWdlKXMiKQoKSEFMVUVWQUxfUUFfVVJMID0gImh0dHBzOi8vcmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbS9SVUNBSUJveC9IYWx1RXZhbC9tYWluL2RhdGEvcWFfZGF0YS5qc29uIgpPVVRQVVRfRElSID0gb3MucGF0aC5qb2luKCJkYXRhIiwgInJhdyIsICJoYWx1ZXZhbCIpCk9VVFBVVF9GSUxFID0gb3MucGF0aC5qb2luKE9VVFBVVF9ESVIsICJxYV9kYXRhLmpzb24iKQoKZGVmIGRvd25sb2FkX2hhbHVldmFsX3FhKCk6CiAgICAiIiJEb3dubG9hZCBIYWx1RXZhbCBxYV9kYXRhLmpzb24gaWYgbm90IGFscmVhZHkgcHJlc2VudC4iIiIKICAgIG9zLm1ha2VkaXJzKE9VVFBVVF9ESVIsIGV4aXN0X29rPVRydWUpCiAgICAKICAgIGxvZ2dpbmcuaW5mbyhmIkRvd25sb2FkaW5nIEhhbHVFdmFsIFFBIGRhdGFzZXQgZnJvbSB7SEFMVUVWQUxfUUFfVVJMfS4uLiIpCiAgICB0cnk6CiAgICAgICAgdXJsbGliLnJlcXVlc3QudXJscmV0cmlldmUoSEFMVUVWQUxfUUFfVVJMLCBPVVRQVVRfRklMRSkKICAgICAgICBzaXplX21iID0gb3MucGF0aC5nZXRzaXplKE9VVFBVVF9GSUxFKSAvICgxMDI0ICogMTAyNCkKICAgICAgICBsb2dnaW5nLmluZm8oZiJTdWNjZXNzZnVsbHkgZG93bmxvYWRlZCBIYWx1RXZhbCBRQSBkYXRhc2V0ICh7c2l6ZV9tYjouMmZ9IE1CKSB0byB7T1VUUFVUX0ZJTEV9IikKICAgICAgICAKICAgICAgICAjIFZlcmlmeSBKU09OTCB2YWxpZGl0eQogICAgICAgIGNvdW50ID0gMAogICAgICAgIHdpdGggb3BlbihPVVRQVVRfRklMRSwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICBmb3IgbGluZSBpbiBmOgogICAgICAgICAgICAgICAgaWYgbGluZS5zdHJpcCgpOgogICAgICAgICAgICAgICAgICAgIGpzb24ubG9hZHMobGluZSkKICAgICAgICAgICAgICAgICAgICBjb3VudCArPSAxCiAgICAgICAgbG9nZ2luZy5pbmZvKGYiVmVyaWZpZWQgZGF0YXNldDoge2NvdW50fSBKU09OTCBpdGVtcyBsb2FkZWQgc3VjY2Vzc2Z1bGx5LiIpCiAgICAgICAgcmV0dXJuIE9VVFBVVF9GSUxFCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgbG9nZ2luZy5lcnJvcihmIkZhaWxlZCB0byBkb3dubG9hZCBvciB2ZXJpZnkgSGFsdUV2YWwgUUEgZGF0YXNldDoge2V9IikKICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhPVVRQVVRfRklMRSk6CiAgICAgICAgICAgIG9zLnJlbW92ZShPVVRQVVRfRklMRSkKICAgICAgICByYWlzZQoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGRvd25sb2FkX2hhbHVldmFsX3FhKCkK",
 "src/data/download_faithbench.py": "IiIiCkZhaXRoQmVuY2ggKE5BQUNMIDIwMjUpIGFjcXVpc2l0aW9uIGZvciBIYWx1UklTQyBWZXJzaW9uIEIgKEIxKS4KCkRvd25sb2FkcyB0aGUgb2ZmaWNpYWwgaHVtYW4tYW5ub3RhdGVkIHJlbGVhc2UgYmF0Y2hlcyBmcm9tIHZlY3RhcmEvRmFpdGhCZW5jaAooZGF0YV9mb3JfcmVsZWFzZS9iYXRjaF97MS4uMTZ9Lmpzb247IGJhdGNoIDEzIGRvZXMgbm90IGV4aXN0IHVwc3RyZWFtKS4KCkZhaXRoQmVuY2ggaXMgQ0MgQlktTkMtU0EgNC4wOiB0aGUgcmF3IGZpbGVzIHN0YXkgdW5kZXIgdGhlIGdpdGlnbm9yZWQKYGRhdGEvcmF3L2ZhaXRoYmVuY2gvYCBhbmQgYXJlIE5FVkVSIGJ1bmRsZWQgaW4gdGhlIHJlcG9zaXRvcnkuIE9ubHkKZG93bmxvYWQgaW5zdHJ1Y3Rpb25zLCBjaXRhdGlvbnMsIGhhc2hlcywgYW5kIGxpY2Vuc2Ugbm90ZXMgYXJlIHNoaXBwZWQuCgpSdW4gKHJlcG8gcm9vdCwgLnZlbnYpOgogIHB5dGhvbiBzcmMvZGF0YS9kb3dubG9hZF9mYWl0aGJlbmNoLnB5CiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCB1cmxsaWIucmVxdWVzdApmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZXpvbmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcyIpCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJkb3dubG9hZF9mYWl0aGJlbmNoIikKClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1syXQpPVVRfRElSID0gUk9PVCAvICJkYXRhIiAvICJyYXciIC8gImZhaXRoYmVuY2giCgpSRVBPID0gInZlY3RhcmEvRmFpdGhCZW5jaCIKQlJBTkNIID0gIm1haW4iClJBV19CQVNFID0gZiJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20ve1JFUE99L3tCUkFOQ0h9IgpBUElfQkFTRSA9IGYiaHR0cHM6Ly9hcGkuZ2l0aHViLmNvbS9yZXBvcy97UkVQT30iCgojIGJhdGNoXzEzIGRvZXMgbm90IGV4aXN0IGluIHRoZSBvZmZpY2lhbCByZWxlYXNlCkJBVENIX0lEUyA9IFtpIGZvciBpIGluIHJhbmdlKDEsIDE3KSBpZiBpICE9IDEzXQpCQVRDSF9GSUxFID0gImJhdGNoX3tpZH0uanNvbiIKCgpkZWYgX2h0dHBfZ2V0X2pzb24odXJsOiBzdHIpIC0+IGRpY3Q6CiAgICByZXEgPSB1cmxsaWIucmVxdWVzdC5SZXF1ZXN0KHVybCwgaGVhZGVycz17IlVzZXItQWdlbnQiOiAiaGFsdXJpc2MtYjEifSkKICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3BlbihyZXEsIHRpbWVvdXQ9NjApIGFzIHJlc3A6CiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMocmVzcC5yZWFkKCkuZGVjb2RlKCJ1dGYtOCIpKQoKCmRlZiBfc2hhMjU2KHBhdGg6IFBhdGgpIC0+IHN0cjoKICAgIGltcG9ydCBoYXNobGliCgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggb3BlbihwYXRoLCAicmIiKSBhcyBmOgogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogZi5yZWFkKDEgPDwgMjApLCBiIiIpOgogICAgICAgICAgICBoLnVwZGF0ZShjaHVuaykKICAgIHJldHVybiBoLmhleGRpZ2VzdCgpCgoKZGVmIF9mZXRjaF9jb21taXRfc2hhKCkgLT4gc3RyOgogICAgdHJ5OgogICAgICAgIGluZm8gPSBfaHR0cF9nZXRfanNvbihmIntBUElfQkFTRX0vY29tbWl0cy97QlJBTkNIfSIpCiAgICAgICAgcmV0dXJuIHN0cihpbmZvLmdldCgic2hhIiwgInVua25vd24iKSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBsb2dnZXIud2FybmluZyhmIkNvdWxkIG5vdCBmZXRjaCBjb21taXQgcmV2aXNpb246IHtlfSIpCiAgICAgICAgcmV0dXJuICJ1bmtub3duIgoKCmRlZiBfdmFsaWRhdGVfYmF0Y2hlcygpIC0+IGludDoKICAgICIiIkVhY2ggYmF0Y2ggcGFyc2VzIGFuZCBldmVyeSBzYW1wbGUgaGFzIHNvdXJjZS9zdW1tYXJ5L21ldGFkYXRhLiIiIgogICAgdG90YWwgPSAwCiAgICBmb3IgYmF0Y2hfaWQgaW4gQkFUQ0hfSURTOgogICAgICAgIHBhdGggPSBPVVRfRElSIC8gQkFUQ0hfRklMRS5mb3JtYXQoaWQ9YmF0Y2hfaWQpCiAgICAgICAgYmF0Y2ggPSBqc29uLmxvYWRzKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIHNhbXBsZXMgPSBiYXRjaFsic2FtcGxlcyJdCiAgICAgICAgZm9yIHMgaW4gc2FtcGxlczoKICAgICAgICAgICAgYXNzZXJ0IGlzaW5zdGFuY2Uocy5nZXQoInNvdXJjZSIpLCBzdHIpIGFuZCBzWyJzb3VyY2UiXS5zdHJpcCgpCiAgICAgICAgICAgIGFzc2VydCBpc2luc3RhbmNlKHMuZ2V0KCJzdW1tYXJ5IiksIHN0cikgYW5kIHNbInN1bW1hcnkiXS5zdHJpcCgpCiAgICAgICAgICAgIGFzc2VydCAibWV0YWRhdGEiIGluIHMgYW5kICJzdW1tYXJpemVyIiBpbiBzWyJtZXRhZGF0YSJdCiAgICAgICAgdG90YWwgKz0gbGVuKHNhbXBsZXMpCiAgICBsb2dnZXIuaW5mbyhmIlZhbGlkYXRlZCB7bGVuKEJBVENIX0lEUyl9IEZhaXRoQmVuY2ggYmF0Y2hlcywge3RvdGFsfSBzYW1wbGVzIHRvdGFsLiIpCiAgICByZXR1cm4gdG90YWwKCgpkZWYgZG93bmxvYWRfZmFpdGhiZW5jaChmb3JjZTogYm9vbCA9IEZhbHNlKSAtPiBkaWN0OgogICAgIiIiRG93bmxvYWQgdGhlIG9mZmljaWFsIGJhdGNoZXMgaWYgbWlzc2luZzsgcmV0dXJucyB7YmF0Y2hfaWQ6IHBhdGh9LiIiIgogICAgb3MubWFrZWRpcnMoT1VUX0RJUiwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGhzID0ge30KICAgIG1pc3NpbmcgPSBbYiBmb3IgYiBpbiBCQVRDSF9JRFMgaWYgbm90IChPVVRfRElSIC8gQkFUQ0hfRklMRS5mb3JtYXQoaWQ9YikpLmV4aXN0cygpXQoKICAgIGlmIG1pc3Npbmcgb3IgZm9yY2U6CiAgICAgICAgaWYgZm9yY2UgYW5kIG5vdCBtaXNzaW5nOgogICAgICAgICAgICBsb2dnZXIuaW5mbygiZm9yY2U9VHJ1ZTogcmUtZG93bmxvYWRpbmcgRmFpdGhCZW5jaCBiYXRjaGVzIikKICAgICAgICBjb21taXRfc2hhID0gX2ZldGNoX2NvbW1pdF9zaGEoKQogICAgICAgIGZvciBiYXRjaF9pZCBpbiBCQVRDSF9JRFM6CiAgICAgICAgICAgIG5hbWUgPSBCQVRDSF9GSUxFLmZvcm1hdChpZD1iYXRjaF9pZCkKICAgICAgICAgICAgZGVzdCA9IE9VVF9ESVIgLyBuYW1lCiAgICAgICAgICAgIGlmIGRlc3QuZXhpc3RzKCkgYW5kIG5vdCBmb3JjZToKICAgICAgICAgICAgICAgIHBhdGhzW2JhdGNoX2lkXSA9IHN0cihkZXN0KQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgdXJsID0gZiJ7UkFXX0JBU0V9L2RhdGFfZm9yX3JlbGVhc2Uve25hbWV9IgogICAgICAgICAgICBsb2dnZXIuaW5mbyhmIiAge3VybH0iKQogICAgICAgICAgICB1cmxsaWIucmVxdWVzdC51cmxyZXRyaWV2ZSh1cmwsIGRlc3QpCiAgICAgICAgICAgIGxvZ2dlci5pbmZvKGYiICBzYXZlZCB7bmFtZX0gKHtkZXN0LnN0YXQoKS5zdF9zaXplIC8gMTAyNDouMGZ9IEtCKSIpCiAgICAgICAgX3ZhbGlkYXRlX2JhdGNoZXMoKQogICAgICAgIGhhc2hlcyA9IHsKICAgICAgICAgICAgbmFtZTogeyJzaGEyNTYiOiBfc2hhMjU2KE9VVF9ESVIgLyBuYW1lKSwgImJ5dGVzIjogKE9VVF9ESVIgLyBuYW1lKS5zdGF0KCkuc3Rfc2l6ZX0KICAgICAgICAgICAgZm9yIG5hbWUgaW4gc29ydGVkKHAubmFtZSBmb3IgcCBpbiBPVVRfRElSLmdsb2IoImJhdGNoXyouanNvbiIpKQogICAgICAgIH0KICAgICAgICAoT1VUX0RJUiAvICJyZXZpc2lvbi5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICAgICAganNvbi5kdW1wcygKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAicmVwbyI6IFJFUE8sCiAgICAgICAgICAgICAgICAgICAgImJyYW5jaCI6IEJSQU5DSCwKICAgICAgICAgICAgICAgICAgICAiY29tbWl0X3NoYSI6IGNvbW1pdF9zaGEsCiAgICAgICAgICAgICAgICAgICAgImZldGNoZWRfYXRfdXRjIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KHRpbWVzcGVjPSJzZWNvbmRzIiksCiAgICAgICAgICAgICAgICAgICAgImZpbGVzIjogaGFzaGVzLAogICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgIGluZGVudD0yLAogICAgICAgICAgICApLAogICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICkKICAgICAgICBsb2dnZXIuaW5mbyhmIlJldmlzaW9uICsgaGFzaGVzIHdyaXR0ZW4gdG8ge09VVF9ESVIgLyAncmV2aXNpb24uanNvbid9IikKICAgIGVsc2U6CiAgICAgICAgbG9nZ2VyLmluZm8oIkZhaXRoQmVuY2ggYmF0Y2hlcyBhbHJlYWR5IHByZXNlbnQsIHNraXBwaW5nLiIpCgogICAgZm9yIGJhdGNoX2lkIGluIEJBVENIX0lEUzoKICAgICAgICBwYXRoc1tiYXRjaF9pZF0gPSBzdHIoT1VUX0RJUiAvIEJBVENIX0ZJTEUuZm9ybWF0KGlkPWJhdGNoX2lkKSkKICAgIHJldHVybiBwYXRocwoKCmRlZiBsb2FkX2ZhaXRoYmVuY2hfc2FtcGxlcygpOgogICAgIiIiUmV0dXJucyBhIGxpc3Qgb2YgKGJhdGNoX2lkLCBzYW1wbGVfZGljdCkgYWNyb3NzIGFsbCBvZmZpY2lhbCBiYXRjaGVzLiIiIgogICAgcGF0aHMgPSBkb3dubG9hZF9mYWl0aGJlbmNoKCkKICAgIHNhbXBsZXMgPSBbXQogICAgZm9yIGJhdGNoX2lkLCBwYXRoIGluIHBhdGhzLml0ZW1zKCk6CiAgICAgICAgYmF0Y2ggPSBqc29uLmxvYWRzKFBhdGgocGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIGZvciBzIGluIGJhdGNoWyJzYW1wbGVzIl06CiAgICAgICAgICAgIHNhbXBsZXMuYXBwZW5kKChpbnQoYmF0Y2hfaWQpLCBzKSkKICAgIHJldHVybiBzYW1wbGVzCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGRvd25sb2FkX2ZhaXRoYmVuY2goKQo=",
 "src/data/download_ragtruth.py": "IiIiClJBR1RydXRoIChBQ0wgMjAyNCkgT0ZGSUNJQUwgYWNxdWlzaXRpb24gZm9yIEhhbHVSSVNDIFZlcnNpb24gQiAoQjEpLgoKRG93bmxvYWRzIHRoZSB0d28gb2ZmaWNpYWwgZmlsZXMgZnJvbSBQYXJ0aWNsZU1lZGlhL1JBR1RydXRoOgogIGRhdGFzZXQvcmVzcG9uc2UuanNvbmwgICAgKHJlc3BvbnNlcyB3aXRoIHdvcmQtbGV2ZWwgaGFsbHVjaW5hdGlvbiBzcGFucykKICBkYXRhc2V0L3NvdXJjZV9pbmZvLmpzb25sIChzb3VyY2VzLCB0YXNrIHR5cGVzLCBwcm9tcHRzKQoKVGhpcyByZXBsYWNlcyB0aGUgbG9zc3kgSHVnZ2luZ0ZhY2UgbWlycm9yIHVzZWQgaW4gVmVyc2lvbiBBICh3aGljaCBkcm9wcGVkCnNvdXJjZV9pZCwgc3BhbnMsIHRhc2tfdHlwZSwgc3BsaXQsIGFuZCBxdWFsaXR5KS4gQjEga2VlcHMgZXZlcnkgZmllbGQuCgpQZXIgZG93bmxvYWQgaXQgcmVjb3JkcyB0aGUgcmVwbyBjb21taXQgcmV2aXNpb24gYW5kIHBlci1maWxlIFNIQS0yNTYgaGFzaGVzCmluIGRhdGEvcmF3L3JhZ3RydXRoX29mZmljaWFsL3JldmlzaW9uLmpzb24gZm9yIHByb3ZlbmFuY2UuCgpSdW4gKHJlcG8gcm9vdCwgLnZlbnYpOgogIHB5dGhvbiBzcmMvZGF0YS9kb3dubG9hZF9yYWd0cnV0aC5weQoiIiIKCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBvcwppbXBvcnQgdXJsbGliLnJlcXVlc3QKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8sIGZvcm1hdD0iJShhc2N0aW1lKXMgLSAlKGxldmVsbmFtZSlzIC0gJShtZXNzYWdlKXMiKQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigiZG93bmxvYWRfcmFndHJ1dGgiKQoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdCk9VVF9ESVIgPSBST09UIC8gImRhdGEiIC8gInJhdyIgLyAicmFndHJ1dGhfb2ZmaWNpYWwiCgpSRVBPID0gIlBhcnRpY2xlTWVkaWEvUkFHVHJ1dGgiCkJSQU5DSCA9ICJtYWluIgpSQVdfQkFTRSA9IGYiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL3tSRVBPfS97QlJBTkNIfSIKQVBJX0JBU0UgPSBmImh0dHBzOi8vYXBpLmdpdGh1Yi5jb20vcmVwb3Mve1JFUE99IgoKIyBuYW1lIC0+IHBhdGggaW5zaWRlIHRoZSByZXBvCkZJTEVTID0gewogICAgInJlc3BvbnNlLmpzb25sIjogImRhdGFzZXQvcmVzcG9uc2UuanNvbmwiLAogICAgInNvdXJjZV9pbmZvLmpzb25sIjogImRhdGFzZXQvc291cmNlX2luZm8uanNvbmwiLAp9CgoKZGVmIF9odHRwX2dldF9qc29uKHVybDogc3RyKSAtPiBkaWN0OgogICAgcmVxID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdCh1cmwsIGhlYWRlcnM9eyJVc2VyLUFnZW50IjogImhhbHVyaXNjLWIxIn0pCiAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4ocmVxLCB0aW1lb3V0PTYwKSBhcyByZXNwOgogICAgICAgIHJldHVybiBqc29uLmxvYWRzKHJlc3AucmVhZCgpLmRlY29kZSgidXRmLTgiKSkKCgpkZWYgX3NoYTI1NihwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICBpbXBvcnQgaGFzaGxpYgoKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIG9wZW4ocGF0aCwgInJiIikgYXMgZjoKICAgICAgICBmb3IgY2h1bmsgaW4gaXRlcihsYW1iZGE6IGYucmVhZCgxIDw8IDIwKSwgYiIiKToKICAgICAgICAgICAgaC51cGRhdGUoY2h1bmspCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiBfZmV0Y2hfY29tbWl0X3NoYSgpIC0+IHN0cjoKICAgIHRyeToKICAgICAgICBpbmZvID0gX2h0dHBfZ2V0X2pzb24oZiJ7QVBJX0JBU0V9L2NvbW1pdHMve0JSQU5DSH0iKQogICAgICAgIHJldHVybiBzdHIoaW5mby5nZXQoInNoYSIsICJ1bmtub3duIikpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAjIHByb3ZlbmFuY2Ugc2hvdWxkIG5vdCBibG9jayB0aGUgZG93bmxvYWQKICAgICAgICBsb2dnZXIud2FybmluZyhmIkNvdWxkIG5vdCBmZXRjaCBjb21taXQgcmV2aXNpb246IHtlfSIpCiAgICAgICAgcmV0dXJuICJ1bmtub3duIgoKCmRlZiBfdmFsaWRhdGVfb2ZmaWNpYWwoKToKICAgICIiIlN0cnVjdHVyYWwgdmFsaWRhdGlvbiBvZiB0aGUgb2ZmaWNpYWwgZmlsZXMgKEIxOiByZXByb2R1Y2libGUgZG93bmxvYWQpLiIiIgogICAgcmVzcF9wYXRoID0gT1VUX0RJUiAvICJyZXNwb25zZS5qc29ubCIKICAgIHNyY19wYXRoID0gT1VUX0RJUiAvICJzb3VyY2VfaW5mby5qc29ubCIKCiAgICByZXNwb25zZXMgPSBbXQogICAgc291cmNlX2lkcyA9IHNldCgpCiAgICB3aXRoIG9wZW4ocmVzcF9wYXRoLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgaWYgbGluZS5zdHJpcCgpOgogICAgICAgICAgICAgICAgcmVzcG9uc2VzLmFwcGVuZChqc29uLmxvYWRzKGxpbmUpKQogICAgd2l0aCBvcGVuKHNyY19wYXRoLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgaWYgbGluZS5zdHJpcCgpOgogICAgICAgICAgICAgICAgc3JjID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgICAgICAgICAgc291cmNlX2lkcy5hZGQoc3JjWyJzb3VyY2VfaWQiXSkKCiAgICBpZHMgPSBbclsiaWQiXSBmb3IgciBpbiByZXNwb25zZXNdCiAgICBhc3NlcnQgbGVuKGlkcykgPT0gbGVuKHNldChpZHMpKSwgInJlc3BvbnNlIGlkcyBhcmUgbm90IHVuaXF1ZSIKICAgIGFzc2VydCBsZW4oc291cmNlX2lkcykgPT0gbGVuKHtzdHIocykgZm9yIHMgaW4gc291cmNlX2lkc30pLCAic291cmNlIGlkcyBhcmUgbm90IHVuaXF1ZSIKICAgIG1pc3NpbmcgPSBzb3J0ZWQoe3JbInNvdXJjZV9pZCJdIGZvciByIGluIHJlc3BvbnNlc30gLSB7c3RyKHMpIGZvciBzIGluIHNvdXJjZV9pZHN9KQogICAgYXNzZXJ0IG5vdCBtaXNzaW5nLCBmInJlc3BvbnNlcyByZWZlcmVuY2UgdW5rbm93biBzb3VyY2VzOiB7bWlzc2luZ1s6NV19IgogICAgbG9nZ2VyLmluZm8oCiAgICAgICAgZiJWYWxpZGF0ZWQgb2ZmaWNpYWwgUkFHVHJ1dGg6IHtsZW4ocmVzcG9uc2VzKX0gcmVzcG9uc2VzLCAiCiAgICAgICAgZiJ7bGVuKHNvdXJjZV9pZHMpfSBzb3VyY2VzLCBubyBvcnBoYW4gcmVzcG9uc2VzLiIKICAgICkKICAgIHJldHVybiBsZW4ocmVzcG9uc2VzKSwgbGVuKHNvdXJjZV9pZHMpCgoKZGVmIGRvd25sb2FkX3JhZ3RydXRoX29mZmljaWFsKGZvcmNlOiBib29sID0gRmFsc2UpIC0+IGRpY3Q6CiAgICAiIiJEb3dubG9hZCB0aGUgb2ZmaWNpYWwgZmlsZXMgaWYgbWlzc2luZzsgcmV0dXJucyB7bmFtZTogcGF0aH0uIiIiCiAgICBvcy5tYWtlZGlycyhPVVRfRElSLCBleGlzdF9vaz1UcnVlKQogICAgcGF0aHMgPSB7fQogICAgbWlzc2luZyA9IFtuYW1lIGZvciBuYW1lIGluIEZJTEVTIGlmIG5vdCAoT1VUX0RJUiAvIG5hbWUpLmV4aXN0cygpXQoKICAgIGlmIG1pc3Npbmcgb3IgZm9yY2U6CiAgICAgICAgaWYgZm9yY2UgYW5kIG5vdCBtaXNzaW5nOgogICAgICAgICAgICBsb2dnZXIuaW5mbygiZm9yY2U9VHJ1ZTogcmUtZG93bmxvYWRpbmcgb2ZmaWNpYWwgUkFHVHJ1dGggZmlsZXMiKQogICAgICAgIGNvbW1pdF9zaGEgPSBfZmV0Y2hfY29tbWl0X3NoYSgpCiAgICAgICAgZm9yIG5hbWUsIHJlcG9fcGF0aCBpbiBGSUxFUy5pdGVtcygpOgogICAgICAgICAgICB1cmwgPSBmIntSQVdfQkFTRX0ve3JlcG9fcGF0aH0iCiAgICAgICAgICAgIGRlc3QgPSBPVVRfRElSIC8gbmFtZQogICAgICAgICAgICBsb2dnZXIuaW5mbyhmIiAge3VybH0iKQogICAgICAgICAgICB1cmxsaWIucmVxdWVzdC51cmxyZXRyaWV2ZSh1cmwsIGRlc3QpCiAgICAgICAgICAgIHNpemVfbWIgPSBkZXN0LnN0YXQoKS5zdF9zaXplIC8gKDEwMjQgKiAxMDI0KQogICAgICAgICAgICBsb2dnZXIuaW5mbyhmIiAgc2F2ZWQge2Rlc3QubmFtZX0gKHtzaXplX21iOi4yZn0gTUIpIikKICAgICAgICBfdmFsaWRhdGVfb2ZmaWNpYWwoKQogICAgICAgIGhhc2hlcyA9IHtuYW1lOiB7InNoYTI1NiI6IF9zaGEyNTYoT1VUX0RJUiAvIG5hbWUpLCAiYnl0ZXMiOiAoT1VUX0RJUiAvIG5hbWUpLnN0YXQoKS5zdF9zaXplfSBmb3IgbmFtZSBpbiBGSUxFU30KICAgICAgICAoT1VUX0RJUiAvICJyZXZpc2lvbi5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICAgICAganNvbi5kdW1wcygKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAicmVwbyI6IFJFUE8sCiAgICAgICAgICAgICAgICAgICAgImJyYW5jaCI6IEJSQU5DSCwKICAgICAgICAgICAgICAgICAgICAiY29tbWl0X3NoYSI6IGNvbW1pdF9zaGEsCiAgICAgICAgICAgICAgICAgICAgImZldGNoZWRfYXRfdXRjIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KHRpbWVzcGVjPSJzZWNvbmRzIiksCiAgICAgICAgICAgICAgICAgICAgImZpbGVzIjogaGFzaGVzLAogICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgIGluZGVudD0yLAogICAgICAgICAgICApLAogICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICkKICAgICAgICBsb2dnZXIuaW5mbyhmIlJldmlzaW9uICsgaGFzaGVzIHdyaXR0ZW4gdG8ge09VVF9ESVIgLyAncmV2aXNpb24uanNvbid9IikKICAgIGVsc2U6CiAgICAgICAgbG9nZ2VyLmluZm8oIk9mZmljaWFsIFJBR1RydXRoIGZpbGVzIGFscmVhZHkgcHJlc2VudCwgc2tpcHBpbmcuIikKCiAgICBmb3IgbmFtZSBpbiBGSUxFUzoKICAgICAgICBwYXRoc1tuYW1lXSA9IHN0cihPVVRfRElSIC8gbmFtZSkKICAgIHJldHVybiBwYXRocwoKCmRlZiBsb2FkX3JhZ3RydXRoX29mZmljaWFsKCk6CiAgICAiIiJSZXR1cm5zIChyZXNwb25zZXM6IGxpc3RbZGljdF0sIHNvdXJjZXM6IGRpY3Rbc291cmNlX2lkIC0+IGRpY3RdKS4iIiIKICAgIHBhdGhzID0gZG93bmxvYWRfcmFndHJ1dGhfb2ZmaWNpYWwoKQogICAgcmVzcG9uc2VzID0gW10KICAgIHdpdGggb3BlbihwYXRoc1sicmVzcG9uc2UuanNvbmwiXSwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGZvciBsaW5lIGluIGY6CiAgICAgICAgICAgIGlmIGxpbmUuc3RyaXAoKToKICAgICAgICAgICAgICAgIHJlc3BvbnNlcy5hcHBlbmQoanNvbi5sb2FkcyhsaW5lKSkKICAgIHNvdXJjZXMgPSB7fQogICAgd2l0aCBvcGVuKHBhdGhzWyJzb3VyY2VfaW5mby5qc29ubCJdLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgaWYgbGluZS5zdHJpcCgpOgogICAgICAgICAgICAgICAgc3JjID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgICAgICAgICAgc291cmNlc1tzdHIoc3JjWyJzb3VyY2VfaWQiXSldID0gc3JjCiAgICByZXR1cm4gcmVzcG9uc2VzLCBzb3VyY2VzCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGRvd25sb2FkX3JhZ3RydXRoX29mZmljaWFsKCkK",
 "src/data/mappings.py": "IiIiClB1cmUgbGFiZWwtbWFwcGluZyBmdW5jdGlvbnMgZm9yIHRoZSBCMSB1bmlmaWVkIHNjaGVtYSAocm9hZG1hcCDCpzE0IEIxLjEvQjEuNCkuCgpFdmVyeSBtYXBwaW5nIGlzIGEgZGV0ZXJtaW5pc3RpYywgdW5pdC10ZXN0ZWQgZnVuY3Rpb24uIFRoZSBiaW5hcnkgYGxhYmVsYAppcyBhIGRvY3VtZW50ZWQgaW50ZXJmYWNlOyBvcmlnaW5hbCB0YXhvbm9teSAoUkFHVHJ1dGggc3BhbnMsIEZhaXRoQmVuY2gKYW5ub3RhdGlvbiBsYWJlbHMpIGlzIHByZXNlcnZlZCBsb3NzbGVzc2x5IGluIGBzcGFuX2Fubm90YXRpb25zYC4KCk1hcHBpbmdzOgogIC0gSGFsdUV2YWw6ICAgIGNvcnJlY3QgYW5zd2VyIC0+IDAsIGhhbGx1Y2luYXRlZCBhbnN3ZXIgLT4gMSAoZml4ZWQgYXQgYnVpbGQpLgogIC0gUkFHVHJ1dGg6ICAgIDEgaWYgdGhlIHJlc3BvbnNlIGNhcnJpZXMgYW55IGhhbGx1Y2luYXRpb24gc3BhbiwgZWxzZSAwCiAgICAgICAgICAgICAgICAgKG9mZmljaWFsIHNwYW5zIGFyZSB0aGUgaHVtYW4gYW5ub3RhdGlvbiBvZiBoYWxsdWNpbmF0aW9uKS4KICAtIEZhaXRoQmVuY2g6ICBzZXZlcml0eSBhZ2dyZWdhdGlvbiBvdmVyIGFubm90YXRpb24gbGFiZWxzOgogICAgICAgICAgICAgICAgICAgbm8gYW5ub3RhdGlvbnMgLT4gMAogICAgICAgICAgICAgICAgICAgQmVuaWduIC0+IDAKICAgICAgICAgICAgICAgICAgIFF1ZXN0aW9uYWJsZSAtPiAxIChwcmltYXJ5IG1hcHBpbmcpCiAgICAgICAgICAgICAgICAgICBVbndhbnRlZCAvIFVud2FudGVkLkludHJpbnNpYyAvIFVud2FudGVkLkV4dHJpbnNpYyAtPiAxCiAgICAgICAgICAgICAgICAgUHJpbWFyeSBhZ2dyZWdhdGlvbiA9IHdvcnN0IHNldmVyaXR5IChvZmZpY2lhbCBzY3JpcHQgZGVmYXVsdCkuCiIiIgoKUkFHVFJVVEhfTEFCRUxfTUFQUElORyA9ICJyYWd0cnV0aC1zcGFuLXYxIgpGQUlUSEJFTkNIX0xBQkVMX01BUFBJTkdfUFJJTUFSWSA9ICJmYWl0aGJlbmNoLXdvcnN0LXErdW53YW50ZWQtdjEiCgojIFNldmVyaXR5IHVzZWQgYnkgdGhlIG9mZmljaWFsIEZhaXRoQmVuY2ggYmluYXJpemUucHkgKDEgPSBsZWFzdCwgMyA9IG1vc3Qgc2V2ZXJlKQpGQUlUSEJFTkNIX1NFVkVSSVRZID0gewogICAgIkJlbmlnbiI6IDEsCiAgICAiUXVlc3Rpb25hYmxlIjogMiwKICAgICJVbndhbnRlZCI6IDMsCiAgICAiVW53YW50ZWQuSW50cmluc2ljIjogMywKICAgICJVbndhbnRlZC5FeHRyaW5zaWMiOiAzLAp9CgojIFRoZSBvZmZpY2lhbCBSRUFETUUgZXhhbXBsZSBjb250YWlucyBhIHR5cG8gKCJJbnN0cmluc2ljIik7IHRoZSBzY2hlbWEgYW5kCiMgcmVsZWFzZWQgZGF0YSB1c2UgIkludHJpbnNpYyIuIE5vcm1hbGl6ZSBzbyBib3RoIHNwZWxsaW5ncyBtYXAgaWRlbnRpY2FsbHkuCl9GQUlUSEJFTkNIX1RZUE9fTUFQID0geyJVbndhbnRlZC5JbnN0cmluc2ljIjogIlVud2FudGVkLkludHJpbnNpYyJ9CgpGQUlUSEJFTkNIX1BSSU1BUllfQ0xBU1NFUyA9IGZyb3plbnNldCgKICAgIHsiUXVlc3Rpb25hYmxlIiwgIlVud2FudGVkIiwgIlVud2FudGVkLkludHJpbnNpYyIsICJVbndhbnRlZC5FeHRyaW5zaWMifQopCkZBSVRIQkVOQ0hfU1RSSUNUX0NMQVNTRVMgPSBmcm96ZW5zZXQoeyJVbndhbnRlZCIsICJVbndhbnRlZC5JbnRyaW5zaWMiLCAiVW53YW50ZWQuRXh0cmluc2ljIn0pCgojIFNlbnNpdGl2aXR5IGNvbmZpZ3VyYXRpb25zIHJlcG9ydGVkIGluIHRoZSBtYXBwaW5nIHJlcG9ydCAoQjEuNikKRkFJVEhCRU5DSF9TRU5TSVRJVklUWV9DT05GSUdTID0gewogICAgInByaW1hcnlfd29yc3RfcV9wbHVzX3Vud2FudGVkIjogewogICAgICAgICJhZ2dyZWdhdGlvbiI6ICJ3b3JzdCIsCiAgICAgICAgImhhbGx1Y2luYXRlZF9jbGFzc2VzIjogRkFJVEhCRU5DSF9QUklNQVJZX0NMQVNTRVMsCiAgICB9LAogICAgIm1ham9yaXR5X3FfcGx1c191bndhbnRlZCI6IHsKICAgICAgICAiYWdncmVnYXRpb24iOiAibWFqb3JpdHkiLAogICAgICAgICJoYWxsdWNpbmF0ZWRfY2xhc3NlcyI6IEZBSVRIQkVOQ0hfUFJJTUFSWV9DTEFTU0VTLAogICAgfSwKICAgICJzdHJpY3Rfd29yc3RfdW53YW50ZWRfb25seSI6IHsKICAgICAgICAiYWdncmVnYXRpb24iOiAid29yc3QiLAogICAgICAgICJoYWxsdWNpbmF0ZWRfY2xhc3NlcyI6IEZBSVRIQkVOQ0hfU1RSSUNUX0NMQVNTRVMsCiAgICB9LAp9CgoKZGVmIG5vcm1hbGl6ZV9mYWl0aGJlbmNoX2xhYmVsKGxhYmVsOiBzdHIpIC0+IHN0cjoKICAgIHJldHVybiBfRkFJVEhCRU5DSF9UWVBPX01BUC5nZXQobGFiZWwsIGxhYmVsKQoKCmRlZiByYWd0cnV0aF9sYWJlbF9mcm9tX3NwYW5zKHNwYW5zKSAtPiBpbnQ6CiAgICAiIiJBbnkgaHVtYW4tYW5ub3RhdGVkIGhhbGx1Y2luYXRpb24gc3BhbiA9PiAxOyBlbXB0eSBhbm5vdGF0aW9uIGxpc3QgPT4gMC4iIiIKICAgIHJldHVybiAxIGlmIHNwYW5zIGVsc2UgMAoKCmRlZiBmYWl0aGJlbmNoX2Fubm90YXRpb25fbGFiZWxzKGFubm90YXRpb25zKSAtPiBzZXQ6CiAgICAiIiJTZXQgb2Ygbm9ybWFsaXplZCBsYWJlbCBzdHJpbmdzIGFjcm9zcyBhbGwgYW5ub3RhdGlvbnMgb2YgYSBzYW1wbGUuIiIiCiAgICBsYWJlbHMgPSBzZXQoKQogICAgZm9yIGFubiBpbiBhbm5vdGF0aW9ucyBvciBbXToKICAgICAgICBmb3IgbGFiIGluIGFubi5nZXQoImxhYmVsIikgb3IgW106CiAgICAgICAgICAgIGxhYmVscy5hZGQobm9ybWFsaXplX2ZhaXRoYmVuY2hfbGFiZWwobGFiKSkKICAgIHJldHVybiBsYWJlbHMKCgpkZWYgZmFpdGhiZW5jaF9zZXZlcml0eShsYWJlbDogc3RyKSAtPiBpbnQ6CiAgICBub3JtYWxpemVkID0gbm9ybWFsaXplX2ZhaXRoYmVuY2hfbGFiZWwobGFiZWwpCiAgICBpZiBub3JtYWxpemVkIG5vdCBpbiBGQUlUSEJFTkNIX1NFVkVSSVRZOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bmtub3duIEZhaXRoQmVuY2ggbGFiZWw6IHtsYWJlbCFyfSIpCiAgICByZXR1cm4gRkFJVEhCRU5DSF9TRVZFUklUWVtub3JtYWxpemVkXQoKCmRlZiBmYWl0aGJlbmNoX2xhYmVsKAogICAgYW5ub3RhdGlvbnMsCiAgICBhZ2dyZWdhdGlvbjogc3RyID0gIndvcnN0IiwKICAgIGhhbGx1Y2luYXRlZF9jbGFzc2VzPUZBSVRIQkVOQ0hfUFJJTUFSWV9DTEFTU0VTLAopIC0+IGludDoKICAgICIiIkFnZ3JlZ2F0ZSBhbm5vdGF0aW9uIGxhYmVscyB0byBhIGJpbmFyeSBsYWJlbC4KCiAgICBhZ2dyZWdhdGlvbj0id29yc3QiOiBtb3N0IHNldmVyZSBsYWJlbCB3aW5zIChvZmZpY2lhbCBzY3JpcHQgZGVmYXVsdCkuCiAgICBhZ2dyZWdhdGlvbj0ibWFqb3JpdHkiOiBtb3N0IGZyZXF1ZW50IHNldmVyaXR5IHdpbnMgKHRpZXMgLT4gbW9zdCBzZXZlcmUpLgogICAgIiIiCiAgICBsYWJlbHMgPSBmYWl0aGJlbmNoX2Fubm90YXRpb25fbGFiZWxzKGFubm90YXRpb25zKQogICAgaWYgbm90IGxhYmVsczoKICAgICAgICByZXR1cm4gMAogICAgc2V2ZXJpdGllcyA9IFtmYWl0aGJlbmNoX3NldmVyaXR5KGwpIGZvciBsIGluIGxhYmVsc10KICAgIGlmIGFnZ3JlZ2F0aW9uID09ICJ3b3JzdCI6CiAgICAgICAgY2hvc2VuX3NldiA9IG1heChzZXZlcml0aWVzKQogICAgZWxpZiBhZ2dyZWdhdGlvbiA9PSAibWFqb3JpdHkiOgogICAgICAgIGNob3Nlbl9zZXYgPSBtYXgoc2V0KHNldmVyaXRpZXMpLCBrZXk9c2V2ZXJpdGllcy5jb3VudCkKICAgIGVsc2U6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gYWdncmVnYXRpb24gc3RyYXRlZ3k6IHthZ2dyZWdhdGlvbiFyfSIpCiAgICBjaG9zZW4gPSBuZXh0KGwgZm9yIGwgaW4gc2V0KGxhYmVscykgaWYgZmFpdGhiZW5jaF9zZXZlcml0eShsKSA9PSBjaG9zZW5fc2V2KQogICAgcmV0dXJuIDEgaWYgY2hvc2VuIGluIGhhbGx1Y2luYXRlZF9jbGFzc2VzIGVsc2UgMAoKCmRlZiBmYWl0aGJlbmNoX2xhYmVsX3NlbnNpdGl2aXR5KGFubm90YXRpb25zKSAtPiBkaWN0OgogICAgIiIiQWxsIGNvbmZpZ3VyZWQgRmFpdGhCZW5jaCBsYWJlbGluZ3MgZm9yIHRoZSBzZW5zaXRpdml0eSByZXBvcnQuIiIiCiAgICByZXR1cm4gewogICAgICAgIG5hbWU6IGludChmYWl0aGJlbmNoX2xhYmVsKGFubm90YXRpb25zLCAqKmNmZykpCiAgICAgICAgZm9yIG5hbWUsIGNmZyBpbiBGQUlUSEJFTkNIX1NFTlNJVElWSVRZX0NPTkZJR1MuaXRlbXMoKQogICAgfQo=",
 "src/data/prepare.py": "IiIiDQpEYXRhIHByZXBhcmF0aW9uIHNjcmlwdCBmb3IgSGFsdVJJU0MuDQpQYXJzZXMgcWFfZGF0YS5qc29uIChKU09OTCkgaW50byBhIGJpbmFyeSBjbGFzc2lmaWNhdGlvbiBkYXRhc2V0ICh0d28gcm93cyBwZXIgZW50cnk6IGNvcnJlY3QgJiBoYWxsdWNpbmF0ZWQpLA0KcGVyZm9ybXMgYSBHUk9VUC1BV0FSRSB0cmFpbi92YWwvdGVzdCBzcGxpdCAoNzAvMTUvMTUpIHNvIHRoYXQgYm90aCBhbnN3ZXIgdmFyaWFudHMgb2Ygb25lIG9yaWdpbmFsIHF1ZXN0aW9uDQpzdGF5IGluIHRoZSBzYW1lIHBhcnRpdGlvbiwgc2F2ZXMgc3BsaXQgaW5kaWNlcywgYSBsZWFrYWdlIHJlcG9ydCwgYW5kIGFuIGF1dG8tc2FtcGxlZCBhdWRpdCBmaWxlLg0KIiIiDQoNCmltcG9ydCBvcw0KaW1wb3J0IGpzb24NCmltcG9ydCBsb2dnaW5nDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBwYW5kYXMgYXMgcGQNCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXQNCg0KbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8sIGZvcm1hdD0iJShhc2N0aW1lKXMgLSAlKGxldmVsbmFtZSlzIC0gJShtZXNzYWdlKXMiKQ0KDQpSQVdfREFUQV9QQVRIID0gb3MucGF0aC5qb2luKCJkYXRhIiwgInJhdyIsICJoYWx1ZXZhbCIsICJxYV9kYXRhLmpzb24iKQ0KUFJPQ0VTU0VEX0RJUiA9IG9zLnBhdGguam9pbigiZGF0YSIsICJwcm9jZXNzZWQiKQ0KQVJUSUZBQ1RTX0RJUiA9IG9zLnBhdGguam9pbigiYXJ0aWZhY3RzIikNClBST0NFU1NFRF9QQVJRVUVUID0gb3MucGF0aC5qb2luKFBST0NFU1NFRF9ESVIsICJxYV9jbGVhbi5wYXJxdWV0IikNCkFVRElUX0pTT04gPSBvcy5wYXRoLmpvaW4oUFJPQ0VTU0VEX0RJUiwgImF1ZGl0XzUwX3NhbXBsZXMuanNvbiIpDQpTUExJVF9JTkRJQ0VTX05QWSA9IG9zLnBhdGguam9pbihBUlRJRkFDVFNfRElSLCAic3BsaXRfaW5kaWNlcy5ucHkiKQ0KU1BMSVRfSU5ESUNFU19KU09OID0gb3MucGF0aC5qb2luKEFSVElGQUNUU19ESVIsICJzcGxpdF9pbmRpY2VzLmpzb24iKQ0KU1BMSVRfUkVQT1JUX0pTT04gPSBvcy5wYXRoLmpvaW4oQVJUSUZBQ1RTX0RJUiwgInNwbGl0X2ludGVncml0eV9yZXBvcnQuanNvbiIpDQoNClNQTElUX1NFRUQgPSA0Mg0KDQoNCmRlZiBsb2FkX2FuZF9wYXJzZV9yYXdfZGF0YShyYXdfcGF0aDogc3RyID0gUkFXX0RBVEFfUEFUSCkgLT4gcGQuRGF0YUZyYW1lOg0KICAgICIiIkxvYWRzIEhhbHVFdmFsIFFBIEpTT05MIGFuZCBleHBhbmRzIGludG8gMiByb3dzIHBlciBxdWVzdGlvbiAoY29ycmVjdD0wLCBoYWxsdWNpbmF0ZWQ9MSkuIiIiDQogICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKHJhd19wYXRoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJSYXcgZGF0YSBub3QgZm91bmQgYXQge3Jhd19wYXRofS4gUnVuIHNyYy9kYXRhL2Rvd25sb2FkLnB5IGZpcnN0LiIpDQoNCiAgICByYXdfaXRlbXMgPSBbXQ0KICAgIHdpdGggb3BlbihyYXdfcGF0aCwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOg0KICAgICAgICBmb3IgbGluZSBpbiBmOg0KICAgICAgICAgICAgaWYgbGluZS5zdHJpcCgpOg0KICAgICAgICAgICAgICAgIHJhd19pdGVtcy5hcHBlbmQoanNvbi5sb2FkcyhsaW5lKSkNCg0KICAgIGxvZ2dpbmcuaW5mbyhmIkxvYWRlZCB7bGVuKHJhd19pdGVtcyl9IHJhdyBRQSBpdGVtcy4iKQ0KICAgIHJvd3MgPSBbXQ0KDQogICAgZm9yIGlkeCwgaXRlbSBpbiBlbnVtZXJhdGUocmF3X2l0ZW1zKToNCiAgICAgICAgcXVlc3Rpb24gPSBpdGVtLmdldCgicXVlc3Rpb24iLCAiIikuc3RyaXAoKQ0KICAgICAgICBjb250ZXh0ID0gaXRlbS5nZXQoImtub3dsZWRnZSIsICIiKS5zdHJpcCgpDQogICAgICAgIHJpZ2h0X2FucyA9IGl0ZW0uZ2V0KCJyaWdodF9hbnN3ZXIiLCBpdGVtLmdldCgiYW5zd2VyIiwgIiIpKS5zdHJpcCgpDQogICAgICAgIGhhbGx1Y2luYXRlZF9hbnMgPSBpdGVtLmdldCgiaGFsbHVjaW5hdGVkX2Fuc3dlciIsICIiKS5zdHJpcCgpDQoNCiAgICAgICAgaWYgbm90IHF1ZXN0aW9uIG9yIG5vdCByaWdodF9hbnM6DQogICAgICAgICAgICBjb250aW51ZQ0KDQogICAgICAgICMgQ29ycmVjdCBzYW1wbGUgKGxhYmVsID0gMCkNCiAgICAgICAgcm93cy5hcHBlbmQoew0KICAgICAgICAgICAgInNhbXBsZV9pZCI6IGYicV97aWR4fV9jb3JyZWN0IiwNCiAgICAgICAgICAgICJpdGVtX2lkeCI6IGlkeCwNCiAgICAgICAgICAgICJxdWVzdGlvbiI6IHF1ZXN0aW9uLA0KICAgICAgICAgICAgImNvbnRleHQiOiBjb250ZXh0LA0KICAgICAgICAgICAgImFuc3dlciI6IHJpZ2h0X2FucywNCiAgICAgICAgICAgICJsYWJlbCI6IDANCiAgICAgICAgfSkNCg0KICAgICAgICAjIEhhbGx1Y2luYXRlZCBzYW1wbGUgKGxhYmVsID0gMSkNCiAgICAgICAgaWYgaGFsbHVjaW5hdGVkX2FuczoNCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsNCiAgICAgICAgICAgICAgICAic2FtcGxlX2lkIjogZiJxX3tpZHh9X2hhbGx1Y2luYXRlZCIsDQogICAgICAgICAgICAgICAgIml0ZW1faWR4IjogaWR4LA0KICAgICAgICAgICAgICAgICJxdWVzdGlvbiI6IHF1ZXN0aW9uLA0KICAgICAgICAgICAgICAgICJjb250ZXh0IjogY29udGV4dCwNCiAgICAgICAgICAgICAgICAiYW5zd2VyIjogaGFsbHVjaW5hdGVkX2FucywNCiAgICAgICAgICAgICAgICAibGFiZWwiOiAxDQogICAgICAgICAgICB9KQ0KDQogICAgZGYgPSBwZC5EYXRhRnJhbWUocm93cykNCiAgICBsb2dnaW5nLmluZm8oZiJDcmVhdGVkIGRhdGFzZXQgd2l0aCB7bGVuKGRmKX0gcm93cyAoe2RmWydsYWJlbCddLnZhbHVlX2NvdW50cygpLnRvX2RpY3QoKX0pLiIpDQogICAgcmV0dXJuIGRmDQoNCg0KZGVmIGJ1aWxkX2ludGVncml0eV9yZXBvcnQoZGY6IHBkLkRhdGFGcmFtZSkgLT4gZGljdDoNCiAgICAiIiJMZWFrYWdlIHJlcG9ydDogZXZlcnkgaXRlbV9pZHggKHNvdXJjZSBxdWVzdGlvbikgbXVzdCBtYXAgdG8gZXhhY3RseSBvbmUgc3BsaXQuIiIiDQogICAgcGVyID0gZGYuZ3JvdXBieSgiaXRlbV9pZHgiKVsic3BsaXQiXS5udW5pcXVlKCkNCiAgICBjcm9zcyA9IGludCgocGVyID4gMSkuc3VtKCkpDQogICAgcmVwb3J0ID0gew0KICAgICAgICAic3BsaXQiOiAiZ3JvdXBfYnlfaXRlbV9pZHgiLA0KICAgICAgICAic2VlZCI6IFNQTElUX1NFRUQsDQogICAgICAgICJuX2dyb3Vwc190b3RhbCI6IGludChwZXIuc2l6ZSksDQogICAgICAgICJuX2dyb3Vwc19wZXJfc3BsaXQiOiB7c3RyKGspOiBpbnQodikgZm9yIGssIHYgaW4gZGYuZ3JvdXBieSgic3BsaXQiKVsiaXRlbV9pZHgiXS5udW5pcXVlKCkudG9fZGljdCgpLml0ZW1zKCl9LA0KICAgICAgICAibl9yb3dzX3Blcl9zcGxpdCI6IHtzdHIoayk6IGludCh2KSBmb3IgaywgdiBpbiBkZi5ncm91cGJ5KCJzcGxpdCIpLnNpemUoKS50b19kaWN0KCkuaXRlbXMoKX0sDQogICAgICAgICJsYWJlbF9tZWFuX3Blcl9zcGxpdCI6IHtzdHIoayk6IHJvdW5kKGZsb2F0KHYpLCA0KSBmb3IgaywgdiBpbiBkZi5ncm91cGJ5KCJzcGxpdCIpWyJsYWJlbCJdLm1lYW4oKS50b19kaWN0KCkuaXRlbXMoKX0sDQogICAgICAgICJncm91cHNfc3Bhbm5pbmdfbXVsdGlwbGVfc3BsaXRzIjogY3Jvc3MsDQogICAgICAgICJsZWFrYWdlX2ZyZWUiOiBjcm9zcyA9PSAwLA0KICAgIH0NCiAgICBpZiBjcm9zcyA+IDA6DQogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKGYiR3JvdXAgbGVha2FnZSBkZXRlY3RlZDoge2Nyb3NzfSBpdGVtX2lkeCB2YWx1ZXMgc3BhbiBtdWx0aXBsZSBzcGxpdHMiKQ0KICAgIHJldHVybiByZXBvcnQNCg0KDQpkZWYgZ3JvdXBfc3BsaXRfYnlfaXRlbShkZjogcGQuRGF0YUZyYW1lLCB0ZXN0X3NpemU6IGZsb2F0ID0gMC4zMCwgdmFsX3NoYXJlOiBmbG9hdCA9IDAuNSwgc2VlZDogaW50ID0gU1BMSVRfU0VFRCk6DQogICAgIiIiR3JvdXAtYXdhcmUgNzAvMTUvMTUgc3BsaXQ6IGVhY2ggb3JpZ2luYWwgcXVlc3Rpb24gKGl0ZW1faWR4KSBnb2VzIHRvIGV4YWN0bHkgb25lIHBhcnRpdGlvbi4NCg0KICAgIFJldHVybnMgKGRmX3dpdGhfc3BsaXRfY29sdW1uLCBpbnRlZ3JpdHlfcmVwb3J0KS4NCiAgICAiIiINCiAgICBncnAgPSAoDQogICAgICAgIGRmLmdyb3VwYnkoIml0ZW1faWR4Iiwgc29ydD1GYWxzZSkNCiAgICAgICAgLmFnZyhsYWJlbD0oImxhYmVsIiwgIm1heCIpLCBuX3Jvd3M9KCJsYWJlbCIsICJzaXplIikpDQogICAgICAgIC5yZXNldF9pbmRleCgpDQogICAgKQ0KICAgIHN0cmF0ID0gZ3JwWyJsYWJlbCJdIGlmIGdycFsibGFiZWwiXS5udW5pcXVlKCkgPiAxIGVsc2UgTm9uZQ0KICAgIHRyYWluX2csIHRlbXBfZyA9IHRyYWluX3Rlc3Rfc3BsaXQoZ3JwLCB0ZXN0X3NpemU9dGVzdF9zaXplLCByYW5kb21fc3RhdGU9c2VlZCwgc3RyYXRpZnk9c3RyYXQpDQogICAgc3RyYXRfdCA9IHRlbXBfZ1sibGFiZWwiXSBpZiB0ZW1wX2dbImxhYmVsIl0ubnVuaXF1ZSgpID4gMSBlbHNlIE5vbmUNCiAgICB2YWxfZywgdGVzdF9nID0gdHJhaW5fdGVzdF9zcGxpdCh0ZW1wX2csIHRlc3Rfc2l6ZT12YWxfc2hhcmUsIHJhbmRvbV9zdGF0ZT1zZWVkLCBzdHJhdGlmeT1zdHJhdF90KQ0KDQogICAgc3BsaXRfb2Y6IGRpY3QgPSB7fQ0KICAgIGZvciBnLCBuYW1lIGluICgodHJhaW5fZywgInRyYWluIiksICh2YWxfZywgInZhbCIpLCAodGVzdF9nLCAidGVzdCIpKToNCiAgICAgICAgZm9yIGkgaW4gZ1siaXRlbV9pZHgiXToNCiAgICAgICAgICAgIHNwbGl0X29mW2ludChpKV0gPSBuYW1lDQoNCiAgICBvdXQgPSBkZi5jb3B5KCkNCiAgICBvdXRbInNwbGl0Il0gPSBvdXRbIml0ZW1faWR4Il0ubWFwKHNwbGl0X29mKQ0KICAgIHJlcG9ydCA9IGJ1aWxkX2ludGVncml0eV9yZXBvcnQob3V0KQ0KICAgIGxvZ2dpbmcuaW5mbygNCiAgICAgICAgZiJHcm91cCBzcGxpdDogdHJhaW4ge2xlbih0cmFpbl9nKX0gLyB2YWwge2xlbih2YWxfZyl9IC8gdGVzdCB7bGVuKHRlc3RfZyl9IGdyb3VwczsgIg0KICAgICAgICBmImxlYWthZ2VfZnJlZT17cmVwb3J0WydsZWFrYWdlX2ZyZWUnXX0iDQogICAgKQ0KICAgIHJldHVybiBvdXQsIHJlcG9ydA0KDQoNCmRlZiBwcmVwYXJlX3NwbGl0c19hbmRfc2F2ZShkZjogcGQuRGF0YUZyYW1lKToNCiAgICAiIiJQZXJmb3JtcyB0aGUgZ3JvdXAtYXdhcmUgdHJhaW4vdmFsL3Rlc3Qgc3BsaXQgYW5kIHNhdmVzIGFsbCBhcnRpZmFjdHMuIiIiDQogICAgb3MubWFrZWRpcnMoUFJPQ0VTU0VEX0RJUiwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBvcy5tYWtlZGlycyhBUlRJRkFDVFNfRElSLCBleGlzdF9vaz1UcnVlKQ0KDQogICAgZGYsIHJlcG9ydCA9IGdyb3VwX3NwbGl0X2J5X2l0ZW0oZGYsIHNlZWQ9U1BMSVRfU0VFRCkNCg0KICAgICMgU2F2ZSBwcm9jZXNzZWQgZGF0YXNldCB0byBQYXJxdWV0DQogICAgZGYudG9fcGFycXVldChQUk9DRVNTRURfUEFSUVVFVCwgaW5kZXg9RmFsc2UpDQogICAgbG9nZ2luZy5pbmZvKGYiU2F2ZWQgcHJvY2Vzc2VkIGRhdGFzZXQgdG8ge1BST0NFU1NFRF9QQVJRVUVUfSIpDQoNCiAgICAjIFNhdmUgc3BsaXQgaW5kaWNlcyArIGdyb3VwIGlkcyAoZXhhY3QgcmVwcm9kdWNpYmlsaXR5KQ0KICAgIHNwbGl0X2luZGljZXMgPSB7DQogICAgICAgICJzcGxpdCI6ICJncm91cF9ieV9pdGVtX2lkeCIsDQogICAgICAgICJ0cmFpbiI6IGRmLmluZGV4W2RmWyJzcGxpdCJdID09ICJ0cmFpbiJdLnRvbGlzdCgpLA0KICAgICAgICAidmFsIjogZGYuaW5kZXhbZGZbInNwbGl0Il0gPT0gInZhbCJdLnRvbGlzdCgpLA0KICAgICAgICAidGVzdCI6IGRmLmluZGV4W2RmWyJzcGxpdCJdID09ICJ0ZXN0Il0udG9saXN0KCksDQogICAgICAgICJncm91cF9pZHMiOiB7DQogICAgICAgICAgICAidHJhaW4iOiBzb3J0ZWQoZGYubG9jW2RmWyJzcGxpdCJdID09ICJ0cmFpbiIsICJpdGVtX2lkeCJdLnVuaXF1ZSgpLnRvbGlzdCgpKSwNCiAgICAgICAgICAgICJ2YWwiOiBzb3J0ZWQoZGYubG9jW2RmWyJzcGxpdCJdID09ICJ2YWwiLCAiaXRlbV9pZHgiXS51bmlxdWUoKS50b2xpc3QoKSksDQogICAgICAgICAgICAidGVzdCI6IHNvcnRlZChkZi5sb2NbZGZbInNwbGl0Il0gPT0gInRlc3QiLCAiaXRlbV9pZHgiXS51bmlxdWUoKS50b2xpc3QoKSksDQogICAgICAgIH0sDQogICAgICAgICJzZWVkIjogU1BMSVRfU0VFRCwNCiAgICAgICAgIm5fdHJhaW4iOiBpbnQoKGRmWyJzcGxpdCJdID09ICJ0cmFpbiIpLnN1bSgpKSwNCiAgICAgICAgIm5fdmFsIjogaW50KChkZlsic3BsaXQiXSA9PSAidmFsIikuc3VtKCkpLA0KICAgICAgICAibl90ZXN0IjogaW50KChkZlsic3BsaXQiXSA9PSAidGVzdCIpLnN1bSgpKSwNCiAgICB9DQoNCiAgICB3aXRoIG9wZW4oU1BMSVRfSU5ESUNFU19KU09OLCAidyIpIGFzIGY6DQogICAgICAgIGpzb24uZHVtcChzcGxpdF9pbmRpY2VzLCBmLCBpbmRlbnQ9MikNCg0KICAgIG5wLnNhdmUoU1BMSVRfSU5ESUNFU19OUFksIHNwbGl0X2luZGljZXMsIGFsbG93X3BpY2tsZT1UcnVlKQ0KICAgIGxvZ2dpbmcuaW5mbyhmIlNhdmVkIHNwbGl0IGluZGljZXMgdG8ge1NQTElUX0lORElDRVNfSlNPTn0gYW5kIHtTUExJVF9JTkRJQ0VTX05QWX0iKQ0KDQogICAgd2l0aCBvcGVuKFNQTElUX1JFUE9SVF9KU09OLCAidyIpIGFzIGY6DQogICAgICAgIGpzb24uZHVtcChyZXBvcnQsIGYsIGluZGVudD0yKQ0KICAgIGxvZ2dpbmcuaW5mbyhmIlNhdmVkIHNwbGl0IGludGVncml0eSByZXBvcnQgdG8ge1NQTElUX1JFUE9SVF9KU09OfSIpDQoNCiAgICAjIFNhbXBsZSA1MCByb3dzIGZvciB0aGUgTUFOVUFMIGF1ZGl0IChsYWJlbHMgbXVzdCBiZSByZXZpZXdlZCBieSBhIGh1bWFuIGJlZm9yZSB0aGUgcGFwZXIpDQogICAgYXVkaXRfc2FtcGxlcyA9IGRmLnNhbXBsZShuPW1pbig1MCwgbGVuKGRmKSksIHJhbmRvbV9zdGF0ZT1TUExJVF9TRUVEKS50b19kaWN0KG9yaWVudD0icmVjb3JkcyIpDQogICAgd2l0aCBvcGVuKEFVRElUX0pTT04sICJ3IikgYXMgZjoNCiAgICAgICAganNvbi5kdW1wKGF1ZGl0X3NhbXBsZXMsIGYsIGluZGVudD0yKQ0KICAgIGxvZ2dpbmcuaW5mbyhmIlNhdmVkIDUwIGF1dG8tc2FtcGxlZCBhdWRpdCByb3dzIHRvIHtBVURJVF9KU09OfSAobWFudWFsIHJldmlldyByZXF1aXJlZCkiKQ0KDQoNCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6DQogICAgZGYgPSBsb2FkX2FuZF9wYXJzZV9yYXdfZGF0YSgpDQogICAgcHJlcGFyZV9zcGxpdHNfYW5kX3NhdmUoZGYpDQo=",
 "src/data/prepare_unified.py": "...",
 "src/data/registry.py": "IiIiCkIxIGRhdGFzZXQgcmVnaXN0cnk6IGxpY2Vuc2UsIHByb3ZlbmFuY2UsIGdyb3VwaW5nIHJ1bGVzLCBhbmQgbGFiZWwgcnVsZXMKKHJvYWRtYXAgwqcxNCBCMS42L0IxLjcpLgoKUmVzdHJpY3RlZCBkYXRhc2V0cyAoRmFpdGhCZW5jaCwgQ0MgQlktTkMtU0EpIGFyZSBORVZFUiBidW5kbGVkIGluIHRoZQpyZXBvc2l0b3J5OiByYXcgZmlsZXMgc3RheSB1bmRlciBnaXRpZ25vcmVkIGBkYXRhL3Jhdy9gLCBhbmQgb25seSBkb3dubG9hZAppbnN0cnVjdGlvbnMsIGNpdGF0aW9ucywgaGFzaGVzLCBhbmQgbGljZW5zZSBub3RlcyBhcmUgc2hpcHBlZC4KIiIiCgppbXBvcnQganNvbgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QsIGRhdGFjbGFzcwpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZXpvbmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBEYXRhc2V0UmVjb3JkOgogICAgbmFtZTogc3RyCiAgICBkaXNwbGF5X25hbWU6IHN0cgogICAgdXJsOiBzdHIKICAgIGxpY2Vuc2U6IHN0cgogICAgcmVkaXN0cmlidXRpb25fYWxsb3dlZDogYm9vbAogICAgZ3JvdXBpbmdfcnVsZTogc3RyCiAgICBsYWJlbF9kZWZpbml0aW9uOiBzdHIKICAgIGxhYmVsX21hcHBpbmdfdmVyc2lvbjogc3RyCiAgICBjaXRhdGlvbjogc3RyCiAgICByYXdfZGlyOiBzdHIgICMgcmVsYXRpdmUgdG8gcmVwbyByb290LCB1bmRlciBkYXRhL3Jhdy8KCgpEQVRBU0VUX1JFR0lTVFJZID0gKAogICAgRGF0YXNldFJlY29yZCgKICAgICAgICBuYW1lPSJoYWx1ZXZhbCIsCiAgICAgICAgZGlzcGxheV9uYW1lPSJIYWx1RXZhbCAoUUEpIiwKICAgICAgICB1cmw9Imh0dHBzOi8vZ2l0aHViLmNvbS9SVUNBSUJveC9IYWx1RXZhbCIsCiAgICAgICAgbGljZW5zZT0iTUlUIiwKICAgICAgICByZWRpc3RyaWJ1dGlvbl9hbGxvd2VkPVRydWUsCiAgICAgICAgZ3JvdXBpbmdfcnVsZT0iZ3JvdXAgYnkgaXRlbV9pZHggKG9yaWdpbmFsIHF1ZXN0aW9uKTsgYm90aCBhbnN3ZXIgdmFyaWFudHMgc3RheSBpbiBvbmUgcGFydGl0aW9uIiwKICAgICAgICBsYWJlbF9kZWZpbml0aW9uPSJjb3JyZWN0IGFuc3dlciAtPiAwOyBoYWxsdWNpbmF0ZWQgYW5zd2VyIC0+IDEiLAogICAgICAgIGxhYmVsX21hcHBpbmdfdmVyc2lvbj0iYjEtbGFiZWxzLXYxIiwKICAgICAgICBjaXRhdGlvbj0iTGkgZXQgYWwuLCBBQ0wgMjAyMywgRE9JIDEwLjE4NjUzL3YxLzIwMjMuZW1ubHAtbWFpbi4zOTciLAogICAgICAgIHJhd19kaXI9ImRhdGEvcmF3L2hhbHVldmFsIiwKICAgICksCiAgICBEYXRhc2V0UmVjb3JkKAogICAgICAgIG5hbWU9InJhZ3RydXRoIiwKICAgICAgICBkaXNwbGF5X25hbWU9IlJBR1RydXRoIChvZmZpY2lhbCkiLAogICAgICAgIHVybD0iaHR0cHM6Ly9naXRodWIuY29tL1BhcnRpY2xlTWVkaWEvUkFHVHJ1dGgiLAogICAgICAgIGxpY2Vuc2U9Ik1JVCIsCiAgICAgICAgcmVkaXN0cmlidXRpb25fYWxsb3dlZD1UcnVlLAogICAgICAgIGdyb3VwaW5nX3J1bGU9Imdyb3VwIGJ5IHNvdXJjZV9pZCAob25lIHNvdXJjZSBlbGljaXRzIHNpeCByZXNwb25zZXMpIiwKICAgICAgICBsYWJlbF9kZWZpbml0aW9uPSJhbnkgaHVtYW4tYW5ub3RhdGVkIGhhbGx1Y2luYXRpb24gc3BhbiAtPiAxOyBubyBzcGFucyAtPiAwIiwKICAgICAgICBsYWJlbF9tYXBwaW5nX3ZlcnNpb249ImIxLWxhYmVscy12MSIsCiAgICAgICAgY2l0YXRpb249Ik5pdSBldCBhbC4sIEFDTCAyMDI0LCBET0kgMTAuMTg2NTMvdjEvMjAyNC5hY2wtbG9uZy41ODUiLAogICAgICAgIHJhd19kaXI9ImRhdGEvcmF3L3JhZ3RydXRoX29mZmljaWFsIiwKICAgICksCiAgICBEYXRhc2V0UmVjb3JkKAogICAgICAgIG5hbWU9ImZhaXRoYmVuY2giLAogICAgICAgIGRpc3BsYXlfbmFtZT0iRmFpdGhCZW5jaCAoc3VtbWFyaXphdGlvbikiLAogICAgICAgIHVybD0iaHR0cHM6Ly9naXRodWIuY29tL3ZlY3RhcmEvRmFpdGhCZW5jaCIsCiAgICAgICAgbGljZW5zZT0iQ0MgQlktTkMtU0EgNC4wIiwKICAgICAgICByZWRpc3RyaWJ1dGlvbl9hbGxvd2VkPUZhbHNlLAogICAgICAgIGdyb3VwaW5nX3J1bGU9Imdyb3VwIGJ5IHJhd19zYW1wbGVfaWQgd2hlbiBhdmFpbGFibGUsIGVsc2Ugc3RhYmxlIGhhc2ggb2Ygc291cmNlIHRleHQiLAogICAgICAgIGxhYmVsX2RlZmluaXRpb249IndvcnN0LXNldmVyaXR5IGFnZ3JlZ2F0aW9uOyBCZW5pZ24vZW1wdHkgLT4gMDsgUXVlc3Rpb25hYmxlL1Vud2FudGVkKiAtPiAxIiwKICAgICAgICBsYWJlbF9tYXBwaW5nX3ZlcnNpb249ImIxLWxhYmVscy12MSIsCiAgICAgICAgY2l0YXRpb249IkJhbyBldCBhbC4sIE5BQUNMIDIwMjUsIERPSSAxMC4xODY1My92MS8yMDI1Lm5hYWNsLXNob3J0LjM4IiwKICAgICAgICByYXdfZGlyPSJkYXRhL3Jhdy9mYWl0aGJlbmNoIiwKICAgICksCikKCgpkZWYgcmVhZF9oYXNoZXNfanNvbihwYXRoOiBQYXRoKSAtPiBkaWN0OgogICAgaWYgbm90IHBhdGguZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIHt9CiAgICB0cnk6CiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBleGNlcHQgKFZhbHVlRXJyb3IsIE9TRXJyb3IpOgogICAgICAgIHJldHVybiB7fQoKCmRlZiBsaWNlbnNlX21hbmlmZXN0KCkgLT4gZGljdDoKICAgICIiIkFzc2VtYmxlIHRoZSBCMSBsaWNlbnNlL3Byb3ZlbmFuY2UgbWFuaWZlc3QgZnJvbSB0aGUgcmVnaXN0cnkgKyBkaXNrIGhhc2hlcy4iIiIKICAgIGRhdGFzZXRzID0gW10KICAgIGZvciByZWMgaW4gREFUQVNFVF9SRUdJU1RSWToKICAgICAgICByZXZpc2lvbiA9IHJlYWRfaGFzaGVzX2pzb24oUk9PVCAvIHJlYy5yYXdfZGlyIC8gInJldmlzaW9uLmpzb24iKQogICAgICAgIGZpbGVzID0ge30KICAgICAgICBmb3IgbmFtZSwgbWV0YSBpbiAocmV2aXNpb24uZ2V0KCJmaWxlcyIpIG9yIHt9KS5pdGVtcygpOgogICAgICAgICAgICBmaWxlc1tuYW1lXSA9IHsKICAgICAgICAgICAgICAgICJzaGEyNTYiOiBtZXRhLmdldCgic2hhMjU2IiksCiAgICAgICAgICAgICAgICAiYnl0ZXMiOiBtZXRhLmdldCgiYnl0ZXMiKSwKICAgICAgICAgICAgfQogICAgICAgIGRhdGFzZXRzLmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgKiphc2RpY3QocmVjKSwKICAgICAgICAgICAgICAgICJkb3dubG9hZF9yZXZpc2lvbiI6IHJldmlzaW9uLmdldCgiY29tbWl0X3NoYSIpLAogICAgICAgICAgICAgICAgImRvd25sb2FkZWRfYXRfdXRjIjogcmV2aXNpb24uZ2V0KCJmZXRjaGVkX2F0X3V0YyIpLAogICAgICAgICAgICAgICAgImZpbGVzIjogZmlsZXMsCiAgICAgICAgICAgIH0KICAgICAgICApCiAgICByZXR1cm4gewogICAgICAgICJzY2hlbWEiOiAiYjEtbGljZW5zZS1tYW5pZmVzdC12MSIsCiAgICAgICAgImdlbmVyYXRlZF9hdF91dGMiOiBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQodGltZXNwZWM9InNlY29uZHMiKSwKICAgICAgICAibm90ZSI6ICJSZXN0cmljdGVkIGRhdGFzZXRzIGFyZSBub3QgcmVkaXN0cmlidXRlZDsgZG93bmxvYWQgaW5zdHJ1Y3Rpb25zLCBoYXNoZXMsIGFuZCBsaWNlbnNlIG5vdGVzIGFyZSBzaGlwcGVkIGluc3RlYWQuIiwKICAgICAgICAiZGF0YXNldHMiOiBkYXRhc2V0cywKICAgIH0K",
 "src/data/schema.py": "IiIiCkhhbHVSSVNDIFZlcnNpb24gQiB1bmlmaWVkIGRhdGFzZXQgc2NoZW1hIChyb2FkbWFwIMKnMTQgQjEpLgoKQ2Fub25pY2FsLCB2ZXJzaW9uZWQgcm93IGNvbnRyYWN0IHNoYXJlZCBieSBIYWx1RXZhbCwgUkFHVHJ1dGgsIGFuZCBGYWl0aEJlbmNoLgoKRGVzaWduIHJ1bGVzOgogIC0gVmVyc2lvbiBBIHByZXByb2Nlc3NpbmcgKGBzcmMvZGF0YS9wcmVwYXJlLnB5YCkgaXMgZnJvemVuIGFuZCB1bmNoYW5nZWQuCiAgLSBUaGlzIG1vZHVsZSBpcyBhZGRpdGl2ZTogaXQgZGVmaW5lcyB0aGUgQjEgY29udHJhY3QgYW5kIHZhbGlkYXRpb24gb25seS4KICAtIE9yaWdpbmFsIGFubm90YXRpb25zIGFuZCBtZXRhZGF0YSBhcmUgcHJlc2VydmVkIGxvc3NsZXNzbHkgaW4gSlNPTiBjb2x1bW5zOwogICAgdGhlIGJpbmFyeSBgbGFiZWxgIGlzIGEgZGVyaXZlZCwgZG9jdW1lbnRlZCBpbnRlcmZhY2UsIG5ldmVyIGEgcmVwbGFjZW1lbnQuCiAgLSBgc2FtcGxlX2lkYCBpcyBnbG9iYWxseSB1bmlxdWU7IGBzb3VyY2VfZ3JvdXBfaWRgIGlzIHRoZSBsZWFrYWdlLWNvbnRyb2wKICAgIGdyb3VwIGtleSAoSGFsdUV2YWw6IGl0ZW1faWR4LCBSQUdUcnV0aDogc291cmNlX2lkLCBGYWl0aEJlbmNoOiByYXcgaWQpLgoiIiIKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCgpTQ0hFTUFfVkVSU0lPTiA9ICJiMS1zY2hlbWEtdjEiCkxBQkVMX01BUFBJTkdfVkVSU0lPTiA9ICJiMS1sYWJlbHMtdjEiCgpTT1VSQ0VfREFUQVNFVFMgPSAoImhhbHVldmFsIiwgInJhZ3RydXRoIiwgImZhaXRoYmVuY2giKQpUQVNLUyA9ICgicWEiLCAic3VtbWFyaXphdGlvbiIsICJkYXRhX3RvX3RleHQiKQpTUExJVFMgPSAoInRyYWluIiwgInZhbCIsICJ0ZXN0IikKClVOSUZJRURfQ09MVU1OUyA9IFsKICAgICJzYW1wbGVfaWQiLCAgICAgICAgICAjIGdsb2JhbGx5IHVuaXF1ZSByb3cgaWQKICAgICJzb3VyY2VfZGF0YXNldCIsICAgICAjIGhhbHVldmFsIHwgcmFndHJ1dGggfCBmYWl0aGJlbmNoCiAgICAic291cmNlX2dyb3VwX2lkIiwgICAgIyBsZWFrYWdlLWNvbnRyb2wgZ3JvdXAga2V5CiAgICAidGFzayIsICAgICAgICAgICAgICAgIyBxYSB8IHN1bW1hcml6YXRpb24gfCBkYXRhX3RvX3RleHQKICAgICJkb21haW4iLCAgICAgICAgICAgICAjIHN0YWJsZSBkb21haW4vc291cmNlIGNhdGVnb3J5CiAgICAicXVlc3Rpb24iLCAgICAgICAgICAgIyB1c2VyIHF1ZXN0aW9uOyAiIiBmb3Igc3VtbWFyaXphdGlvbi9kYXRhLXRvLXRleHQKICAgICJjb250ZXh0IiwgICAgICAgICAgICAjIGV2aWRlbmNlIC8gc291cmNlIHRleHQKICAgICJhbnN3ZXIiLCAgICAgICAgICAgICAjIG1vZGVsIHJlc3BvbnNlIC8gc3VtbWFyeSAobXVzdCBtYXRjaCBzcGFuIG9mZnNldHMpCiAgICAibGFiZWwiLCAgICAgICAgICAgICAgIyB1bmlmaWVkIGJpbmFyeSBsYWJlbCAoMCA9IGZhaXRoZnVsLCAxID0gaGFsbHVjaW5hdGVkKQogICAgInNwYW5fYW5ub3RhdGlvbnMiLCAgICMgSlNPTiBzdHJpbmcsIG9yaWdpbmFsIGFubm90YXRpb25zIHByZXNlcnZlZCB2ZXJiYXRpbQogICAgImdlbmVyYXRvcl9tb2RlbCIsICAgICMgb3JpZ2luYWwgbW9kZWwgd2hlbiBhdmFpbGFibGUKICAgICJvZmZpY2lhbF9zcGxpdCIsICAgICAjIG5hdGl2ZSBkYXRhc2V0IHNwbGl0IChSQUdUcnV0aCB0cmFpbi90ZXN0KSBvciAiIgogICAgImV4cGVyaW1lbnRfc3BsaXQiLCAgICMgSGFsdUV2YWwgZ3JvdXBlZCBzcGxpdCAodHJhaW4vdmFsL3Rlc3QpIG9yICIiCiAgICAicXVhbGl0eSIsICAgICAgICAgICAgIyBSQUdUcnV0aCBxdWFsaXR5IGZsYWcgKGdvb2QvdHJ1bmNhdGVkLy4uLikgb3IgIiIKICAgICJuYXRpdmVfcmVjb3JkX2lkIiwgICAjIG9yaWdpbmFsIGRhdGFzZXQgcm93IGlkCiAgICAibmF0aXZlX21ldGFkYXRhIiwgICAgIyBKU09OIHN0cmluZywgb3RoZXIgc291cmNlLXNwZWNpZmljIG1ldGFkYXRhCiAgICAibGFiZWxfbWFwcGluZ192ZXJzaW9uIiwKXQoKRU1QVFlfU1BBTlMgPSAiW10iCkVNUFRZX01FVEEgPSAie30iCgoKZGVmIGpzb25fZHVtcHMob2JqKSAtPiBzdHI6CiAgICAiIiJEZXRlcm1pbmlzdGljIEpTT04gc2VyaWFsaXphdGlvbiBmb3IgbWV0YWRhdGEgY29sdW1ucy4iIiIKICAgIHJldHVybiBqc29uLmR1bXBzKG9iaiwgZW5zdXJlX2FzY2lpPUZhbHNlLCBzb3J0X2tleXM9VHJ1ZSkKCgpkZWYgc2hhMjU2X3RleHQodGV4dDogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYodGV4dC5lbmNvZGUoInV0Zi04IikpLmhleGRpZ2VzdCgpCgoKZGVmIGZyYW1lX2ZpbmdlcnByaW50KGRmKSAtPiBzdHI6CiAgICAiIiJTdGFibGUgY29udGVudCBoYXNoIG9mIGEgY2Fub25pY2FsIGZyYW1lIChkZXRlcm1pbmlzbSBjaGVjaykuIiIiCiAgICByZXR1cm4gc2hhMjU2X3RleHQoanNvbl9kdW1wcyhkZi50b19kaWN0KG9yaWVudD0icmVjb3JkcyIpKSkKCgpkZWYgX2FsbF9qc29uX3N0cmluZ3MoZGYsIGNvbDogc3RyKSAtPiBib29sOgogICAgZGVmIG9rKHYpOgogICAgICAgIGlmIHYgaXMgTm9uZSBvciAoaXNpbnN0YW5jZSh2LCBzdHIpIGFuZCBub3Qgdi5zdHJpcCgpKToKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZSh2LCBzdHIpOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICB0cnk6CiAgICAgICAgICAgIGpzb24ubG9hZHModikKICAgICAgICBleGNlcHQgKFZhbHVlRXJyb3IsIFR5cGVFcnJvcik6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIHJldHVybiBUcnVlCgogICAgcmV0dXJuIGRmW2NvbF0ubWFwKG9rKS5hbGwoKQoKCmRlZiB2YWxpZGF0ZV91bmlmaWVkX2RmKGRmKToKICAgICIiIlJhaXNlIFZhbHVlRXJyb3Igd2l0aCBhIHByZWNpc2UgbWVzc2FnZSBvbiBhbnkgY29udHJhY3QgdmlvbGF0aW9uLiIiIgogICAgbWlzc2luZyA9IFtjIGZvciBjIGluIFVOSUZJRURfQ09MVU1OUyBpZiBjIG5vdCBpbiBkZi5jb2x1bW5zXQogICAgaWYgbWlzc2luZzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYibWlzc2luZyBjYW5vbmljYWwgY29sdW1uczoge21pc3Npbmd9IikKICAgIGV4dHJhID0gW2MgZm9yIGMgaW4gZGYuY29sdW1ucyBpZiBjIG5vdCBpbiBVTklGSUVEX0NPTFVNTlNdCiAgICBpZiBleHRyYToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5leHBlY3RlZCBjb2x1bW5zOiB7ZXh0cmF9IikKCiAgICBpZiBkZlsic2FtcGxlX2lkIl0uZHVwbGljYXRlZCgpLmFueSgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNhbXBsZV9pZCBtdXN0IGJlIGdsb2JhbGx5IHVuaXF1ZSIpCiAgICBibGFuayA9IGRmWyJzYW1wbGVfaWQiXS5pc25hKCkgfCAoZGZbInNhbXBsZV9pZCJdLmFzdHlwZShzdHIpLnN0ci5zdHJpcCgpID09ICIiKQogICAgaWYgYmxhbmsuYW55KCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic2FtcGxlX2lkIG11c3QgYmUgbm9uLWVtcHR5IikKICAgIGJsYW5rID0gZGZbInNvdXJjZV9ncm91cF9pZCJdLmlzbmEoKSB8IChkZlsic291cmNlX2dyb3VwX2lkIl0uYXN0eXBlKHN0cikuc3RyLnN0cmlwKCkgPT0gIiIpCiAgICBpZiBibGFuay5hbnkoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzb3VyY2VfZ3JvdXBfaWQgbXVzdCBiZSBub24tZW1wdHkiKQoKICAgIGlmIG5vdCBzZXQoZGZbImxhYmVsIl0udW5pcXVlKCkpLmlzc3Vic2V0KHswLCAxfSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibGFiZWwgbXVzdCBiZSAwIG9yIDEiKQogICAgYmFkX2RzID0gc2V0KGRmWyJzb3VyY2VfZGF0YXNldCJdLnVuaXF1ZSgpKSAtIHNldChTT1VSQ0VfREFUQVNFVFMpCiAgICBpZiBiYWRfZHM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gc291cmNlX2RhdGFzZXQgdmFsdWVzOiB7c29ydGVkKGJhZF9kcyl9IikKICAgIGJhZF90YXNrID0gc2V0KGRmWyJ0YXNrIl0udW5pcXVlKCkpIC0gc2V0KFRBU0tTKQogICAgaWYgYmFkX3Rhc2s6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gdGFzayB2YWx1ZXM6IHtzb3J0ZWQoYmFkX3Rhc2spfSIpCgogICAgZW1wdHlfYW5zID0gZGZbImFuc3dlciJdLmlzbmEoKSB8IChkZlsiYW5zd2VyIl0uYXN0eXBlKHN0cikuc3RyLnN0cmlwKCkgPT0gIiIpCiAgICBpZiBlbXB0eV9hbnMuYW55KCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYW5zd2VyIG11c3QgYmUgbm9uLWVtcHR5IikKCiAgICBleHBfdmFsdWVzID0gc2V0KGRmWyJleHBlcmltZW50X3NwbGl0Il1bZGZbImV4cGVyaW1lbnRfc3BsaXQiXSAhPSAiIl0udW5pcXVlKCkpCiAgICBiYWRfZXhwID0gZXhwX3ZhbHVlcyAtIHNldChTUExJVFMpCiAgICBpZiBiYWRfZXhwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJleHBlcmltZW50X3NwbGl0IG11c3QgYmUgaW4ge1NQTElUU30gb3IgZW1wdHksIGdvdDoge3NvcnRlZChiYWRfZXhwKX0iKQogICAgb2ZmX3ZhbHVlcyA9IHNldChkZlsib2ZmaWNpYWxfc3BsaXQiXVtkZlsib2ZmaWNpYWxfc3BsaXQiXSAhPSAiIl0udW5pcXVlKCkpCiAgICBmb3IgdiBpbiBvZmZfdmFsdWVzOgogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHYsIHN0cik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJvZmZpY2lhbF9zcGxpdCBtdXN0IGJlIGEgc3RyaW5nLCBnb3Qge3Yhcn0iKQoKICAgIGZvciBjb2wgaW4gKCJzcGFuX2Fubm90YXRpb25zIiwgIm5hdGl2ZV9tZXRhZGF0YSIpOgogICAgICAgIGlmIG5vdCBfYWxsX2pzb25fc3RyaW5ncyhkZiwgY29sKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIntjb2x9IG11c3QgYmUgYSBKU09OIHN0cmluZyAob3IgZW1wdHkpIikKICAgIHJldHVybiBUcnVlCg==",
 "src/explain/shap_analysis.py": "IiIiClNIQVAgZXhwbGFpbmFiaWxpdHkgYW5hbHlzaXMgZm9yIEhhbHVSSVNDIChibHVlcHJpbnQgQTksIHJvYWRtYXAgUGhhc2UgNikuCgpQcm9kdWNlcyAoc2F2ZWQgdG8gYXJ0aWZhY3RzL2ZpZ3VyZXMgKyBhcnRpZmFjdHMvcmVzdWx0cyk6CiAgLSBHbG9iYWw6IFNIQVAgYmVlc3dhcm0gc3VtbWFyeSArIG1lYW58U0hBUHwgYmFyIGNoYXJ0CiAgLSBMb2NhbDogd2F0ZXJmYWxsIHBsb3RzIGZvciAzIGhhbmQtcGlja2VkIHRlc3QgY2FzZXMKICAtIFJPQyAvIFBSIGN1cnZlcyArIHJlbGlhYmlsaXR5IGRpYWdyYW0gKHdpdGggRUNFL0JyaWVyIGFubm90YXRpb24pCiAgLSBzaGFwX3N1bW1hcnkuanNvbiAoZ2xvYmFsIHRvcCBmZWF0dXJlcyArIHBlci1jYXNlIGxvY2FsIFNIQVAgZm9yIHRoZSBkYXNoYm9hcmQpCgpSdW4gKHJlcG8gcm9vdCwgLnZlbnYpOgogIHB5dGhvbiBzcmMvZXhwbGFpbi9zaGFwX2FuYWx5c2lzLnB5CiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdKSkKCmltcG9ydCBqb2JsaWIKaW1wb3J0IG1hdHBsb3RsaWIKCm1hdHBsb3RsaWIudXNlKCJBZ2ciKQppbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcyIpCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJzaGFwX2FuYWx5c2lzIikKCmZyb20gc3JjLm1vZGVscy5jb25maWcgaW1wb3J0IEZFQVRVUkVTX0ZBTExCQUNLLCBGRUFUVVJFU19GVUxMLCBGSUdVUkVTX0RJUiwgTU9ERUxTX0RJUiwgUUFfQ0xFQU4sIFJFU1VMVFNfRElSLCBST09UCgpTSEFQX1NVQlNBTVBMRSA9IDEwMDAgICMga2VwdCBmb3IgcmVmZXJlbmNlOyBmdWxsIHRlc3Qgc2V0IGlzIHVzZWQgKHRyZWUgZXhwbGFpbmVyIGlzIGZhc3QpCk5fVE9QX0ZFQVRVUkVTID0gMTAKCgpkZWYgbG9hZF90ZXN0X3NldCgpOgogICAgcGF0aCA9IEZFQVRVUkVTX0ZVTEwgaWYgRkVBVFVSRVNfRlVMTC5leGlzdHMoKSBlbHNlIEZFQVRVUkVTX0ZBTExCQUNLCiAgICBkZiA9IHBkLnJlYWRfcGFycXVldChwYXRoKQogICAgY2xlYW4gPSBwZC5yZWFkX3BhcnF1ZXQoUUFfQ0xFQU4pCiAgICB0ZXh0X2NvbHMgPSBbYyBmb3IgYyBpbiBbInF1ZXN0aW9uIiwgImFuc3dlciIsICJjb250ZXh0Il0gaWYgYyBpbiBjbGVhbi5jb2x1bW5zXQogICAgaWYgdGV4dF9jb2xzOgogICAgICAgIGRmID0gcGQuY29uY2F0KFtkZiwgY2xlYW5bdGV4dF9jb2xzXV0sIGF4aXM9MSkKICAgIGZlYXR1cmVfY29scyA9IGpzb24ubG9hZHMoKE1PREVMU19ESVIgLyAiZmVhdHVyZV9uYW1lcy5qc29uIikucmVhZF90ZXh0KCkpCiAgICB0ZXN0X2RmID0gZGZbZGZbInNwbGl0Il0gPT0gInRlc3QiXS5jb3B5KCkKICAgIFhfdGVzdCA9IHRlc3RfZGZbZmVhdHVyZV9jb2xzXS52YWx1ZXMKICAgIHlfdGVzdCA9IHRlc3RfZGZbImxhYmVsIl0udmFsdWVzCiAgICByZXR1cm4gdGVzdF9kZiwgWF90ZXN0LCB5X3Rlc3QsIGZlYXR1cmVfY29scwoKCmRlZiBjYXNlX2luZGV4ZXMoeV9wcm9iOiBucC5uZGFycmF5KSAtPiBkaWN0OgogICAgIiIiMyBoYW5kLXBpY2tlZCBjYXNlczogY2xlYXIgaGFsbHVjaW5hdGlvbiwgY2xlYXJseSBjb3JyZWN0LCBib3JkZXJsaW5lLiIiIgogICAgaWR4X2hpZ2ggPSBpbnQobnAuYXJnbWF4KHlfcHJvYikpCiAgICBpZHhfbG93ID0gaW50KG5wLmFyZ21pbih5X3Byb2IpKQogICAgaWR4X2JvcmRlciA9IGludChucC5hcmdtaW4obnAuYWJzKHlfcHJvYiAtIDAuNSkpKQogICAgcmV0dXJuIHsiaGlnaF9yaXNrIjogaWR4X2hpZ2gsICJsb3dfcmlzayI6IGlkeF9sb3csICJib3JkZXJsaW5lIjogaWR4X2JvcmRlcn0KCgpkZWYgX3NhdmVfZmlnKGZpZywgbmFtZTogc3RyKToKICAgICIiIlNhdmUgYSBmaWd1cmUgYXMgUE5HIChkYXNoYm9hcmQpICsgUERGIChwYXBlciwgYmx1ZXByaW50IEExOCB2ZWN0b3IgZm9ybWF0KS4iIiIKICAgIGZvciBleHQgaW4gKCJwbmciLCAicGRmIik6CiAgICAgICAgZmlnLnNhdmVmaWcoRklHVVJFU19ESVIgLyBmIntuYW1lfS57ZXh0fSIsIGRwaT0xNTAsIGJib3hfaW5jaGVzPSJ0aWdodCIpCiAgICBwbHQuY2xvc2UoZmlnKQogICAgbG9nZ2VyLmluZm8oZiJTYXZlZCB7bmFtZX0ucG5nLy5wZGYiKQoKCmRlZiBwbG90X3JvY19wcih5X3Rlc3QsIHlfcHJvYiwgbmFtZTogc3RyKToKICAgIGZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBhdmVyYWdlX3ByZWNpc2lvbl9zY29yZSwgcHJlY2lzaW9uX3JlY2FsbF9jdXJ2ZSwgcm9jX2F1Y19zY29yZSwgcm9jX2N1cnZlCgogICAgZnByLCB0cHIsIF8gPSByb2NfY3VydmUoeV90ZXN0LCB5X3Byb2IpCiAgICBwcmVjLCByZWMsIF8gPSBwcmVjaXNpb25fcmVjYWxsX2N1cnZlKHlfdGVzdCwgeV9wcm9iKQogICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIDIsIGZpZ3NpemU9KDEyLCA0LjUpKQogICAgYXhlc1swXS5wbG90KGZwciwgdHByLCBsdz0yLCBjb2xvcj0iIzhiNWNmNiIpCiAgICBheGVzWzBdLnBsb3QoWzAsIDFdLCBbMCwgMV0sIGxzPSItLSIsIGNvbG9yPSJncmF5IiwgYWxwaGE9MC42KQogICAgYXhlc1swXS5zZXRfdGl0bGUoZiJST0MgKEFVQz17cm9jX2F1Y19zY29yZSh5X3Rlc3QsIHlfcHJvYik6LjRmfSkiKQogICAgYXhlc1swXS5zZXRfeGxhYmVsKCJGYWxzZSBwb3NpdGl2ZSByYXRlIikKICAgIGF4ZXNbMF0uc2V0X3lsYWJlbCgiVHJ1ZSBwb3NpdGl2ZSByYXRlIikKICAgIGF4ZXNbMV0ucGxvdChyZWMsIHByZWMsIGx3PTIsIGNvbG9yPSIjNjM2NmYxIikKICAgIGF4ZXNbMV0uc2V0X3RpdGxlKGYiUFIgY3VydmUgKEFVQz17YXZlcmFnZV9wcmVjaXNpb25fc2NvcmUoeV90ZXN0LCB5X3Byb2IpOi40Zn0pIikKICAgIGF4ZXNbMV0uc2V0X3hsYWJlbCgiUmVjYWxsIikKICAgIGF4ZXNbMV0uc2V0X3lsYWJlbCgiUHJlY2lzaW9uIikKICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgX3NhdmVfZmlnKGZpZywgbmFtZSkKCgpkZWYgcGxvdF9yZWxpYWJpbGl0eSh5X3Rlc3QsIHlfcHJvYiwgbmFtZTogc3RyLCBuX2JpbnM6IGludCA9IDEwKToKICAgIGJpbnMgPSBucC5saW5zcGFjZSgwLCAxLCBuX2JpbnMgKyAxKQogICAgaWR4cyA9IG5wLmNsaXAobnAuc2VhcmNoc29ydGVkKGJpbnMsIHlfcHJvYiwgc2lkZT0icmlnaHQiKSAtIDEsIDAsIG5fYmlucyAtIDEpCiAgICBjb25mcywgYWNjcywgY291bnRzID0gW10sIFtdLCBbXQogICAgZm9yIGIgaW4gcmFuZ2Uobl9iaW5zKToKICAgICAgICBtYXNrID0gaWR4cyA9PSBiCiAgICAgICAgaWYgbWFzay5zdW0oKSA9PSAwOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGNvbmZzLmFwcGVuZCh5X3Byb2JbbWFza10ubWVhbigpKQogICAgICAgIGFjY3MuYXBwZW5kKHlfdGVzdFttYXNrXS5tZWFuKCkpCiAgICAgICAgY291bnRzLmFwcGVuZChtYXNrLnN1bSgpKQoKICAgIGZyb20gc3JjLm1vZGVscy50cmFpbl9waXBlbGluZSBpbXBvcnQgZWNlCiAgICBmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgYnJpZXJfc2NvcmVfbG9zcwoKICAgIGVjZV92YWwgPSBlY2UoeV90ZXN0LCB5X3Byb2IpCiAgICBicmllciA9IGJyaWVyX3Njb3JlX2xvc3MoeV90ZXN0LCB5X3Byb2IpCgogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg2LCA1KSkKICAgIGF4LnBsb3QoWzAsIDFdLCBbMCwgMV0sIGxzPSItLSIsIGNvbG9yPSJncmF5IiwgYWxwaGE9MC43LCBsYWJlbD0iUGVyZmVjdCBjYWxpYnJhdGlvbiIpCiAgICBheC5wbG90KGNvbmZzLCBhY2NzLCBtYXJrZXI9Im8iLCBsdz0yLCBjb2xvcj0iIzhiNWNmNiIsIGxhYmVsPSJNb2RlbCIpCiAgICBmb3IgYywgYSwgbiBpbiB6aXAoY29uZnMsIGFjY3MsIGNvdW50cyk6CiAgICAgICAgYXguYW5ub3RhdGUoc3RyKGludChuKSksIChjLCBhKSwgdGV4dGNvb3Jkcz0ib2Zmc2V0IHBvaW50cyIsIHh5dGV4dD0oNCwgNCksIGZvbnRzaXplPTgsIGFscGhhPTAuNykKICAgIGF4LnNldF94bGltKDAsIDEpCiAgICBheC5zZXRfeWxpbSgwLCAxKQogICAgYXguc2V0X3hsYWJlbCgiQ29uZmlkZW5jZSAocHJlZGljdGVkIHByb2JhYmlsaXR5KSIpCiAgICBheC5zZXRfeWxhYmVsKCJBY2N1cmFjeSAoZW1waXJpY2FsIGZyZXF1ZW5jeSkiKQogICAgYXguc2V0X3RpdGxlKGYiUmVsaWFiaWxpdHkgZGlhZ3JhbVxuRUNFPXtlY2VfdmFsOi40Zn0gfCBCcmllcj17YnJpZXI6LjRmfSIpCiAgICBheC5sZWdlbmQoKQogICAgZmlnLnRpZ2h0X2xheW91dCgpCiAgICBfc2F2ZV9maWcoZmlnLCBuYW1lKQogICAgcmV0dXJuIGVjZV92YWwsIGJyaWVyCgoKZGVmIG1haW4oKToKICAgIG9zLm1ha2VkaXJzKEZJR1VSRVNfRElSLCBleGlzdF9vaz1UcnVlKQoKICAgIGltcG9ydCBzaGFwCgogICAgdGVzdF9kZiwgWF90ZXN0LCB5X3Rlc3QsIGZlYXR1cmVfY29scyA9IGxvYWRfdGVzdF9zZXQoKQogICAgcmF3ID0gam9ibGliLmxvYWQoTU9ERUxTX0RJUiAvICJtb2RlbF94Z2Jvb3N0X3Jhdy5qb2JsaWIiKQogICAgeV9wcm9iID0gcmF3LnByZWRpY3RfcHJvYmEoWF90ZXN0KVs6LCAxXQogICAgbG9nZ2VyLmluZm8oZiJUZXN0IHNldDoge2xlbihYX3Rlc3QpfSBzYW1wbGVzLCB7bGVuKGZlYXR1cmVfY29scyl9IGZlYXR1cmVzIikKCiAgICAjIC0tLS0gR2xvYmFsIFNIQVAgKGZ1bGwgdGVzdCBzZXQ7IHRyZWUgZXhwbGFpbmVyIGlzIGNoZWFwKSAtLS0tCiAgICBleHBsYWluZXIgPSBzaGFwLlRyZWVFeHBsYWluZXIocmF3KQogICAgam9ibGliLmR1bXAoZXhwbGFpbmVyLCBNT0RFTFNfRElSIC8gInNoYXBfZXhwbGFpbmVyLmpvYmxpYiIpICAjIEExODogc2F2ZWQgZXhwbGFpbmVyIGFydGlmYWN0CiAgICBsb2dnZXIuaW5mbygiU2F2ZWQgc2hhcF9leHBsYWluZXIuam9ibGliIikKICAgIHNoYXBfdmFsdWVzID0gZXhwbGFpbmVyLnNoYXBfdmFsdWVzKFhfdGVzdCkKCiAgICAjIEJlZXN3YXJtIHN1bW1hcnkKICAgIHNoYXAuc3VtbWFyeV9wbG90KHNoYXBfdmFsdWVzLCBYX3Rlc3QsIGZlYXR1cmVfbmFtZXM9ZmVhdHVyZV9jb2xzLCBzaG93PUZhbHNlLCBtYXhfZGlzcGxheT0xNSkKICAgIF9zYXZlX2ZpZyhwbHQuZ2NmKCksICJmaWdfc2hhcF9zdW1tYXJ5IikKCiAgICAjIE1lYW4gfFNIQVB8IGJhcgogICAgbWVhbl9hYnMgPSBucC5tZWFuKG5wLmFicyhzaGFwX3ZhbHVlcyksIGF4aXM9MCkKICAgIG9yZGVyID0gbnAuYXJnc29ydChtZWFuX2FicylbOjotMV0KICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oOCwgNikpCiAgICBheC5iYXJoKAogICAgICAgIFtmZWF0dXJlX2NvbHNbaV0gZm9yIGkgaW4gb3JkZXJbOk5fVE9QX0ZFQVRVUkVTXV1bOjotMV0sCiAgICAgICAgbWVhbl9hYnNbb3JkZXJbOk5fVE9QX0ZFQVRVUkVTXV1bOjotMV0sCiAgICAgICAgY29sb3I9IiM4YjVjZjYiLAogICAgKQogICAgYXguc2V0X3RpdGxlKCJNZWFuIHxTSEFQfCBmZWF0dXJlIGltcG9ydGFuY2UiKQogICAgYXguc2V0X3hsYWJlbCgibWVhbiB8U0hBUCB2YWx1ZXwiKQogICAgZmlnLnRpZ2h0X2xheW91dCgpCiAgICBfc2F2ZV9maWcoZmlnLCAiZmlnX3NoYXBfaW1wb3J0YW5jZSIpCgogICAgIyAtLS0tIExvY2FsOiAzIHdhdGVyZmFsbCBjYXNlcyAtLS0tCiAgICBjYXNlcyA9IGNhc2VfaW5kZXhlcyh5X3Byb2IpCiAgICBjYXNlX3NoYXAgPSB7fQogICAgZm9yIGxhYmVsLCBpZHggaW4gY2FzZXMuaXRlbXMoKToKICAgICAgICBzaGFwLndhdGVyZmFsbF9wbG90KAogICAgICAgICAgICBzaGFwLkV4cGxhbmF0aW9uKAogICAgICAgICAgICAgICAgc2hhcF92YWx1ZXNbaWR4XSwKICAgICAgICAgICAgICAgIGJhc2VfdmFsdWVzPWV4cGxhaW5lci5leHBlY3RlZF92YWx1ZSwKICAgICAgICAgICAgICAgIGRhdGE9WF90ZXN0W2lkeF0sCiAgICAgICAgICAgICAgICBmZWF0dXJlX25hbWVzPWZlYXR1cmVfY29scywKICAgICAgICAgICAgKSwKICAgICAgICAgICAgbWF4X2Rpc3BsYXk9MTAsCiAgICAgICAgICAgIHNob3c9RmFsc2UsCiAgICAgICAgKQogICAgICAgIF9zYXZlX2ZpZyhwbHQuZ2NmKCksIGYiZmlnX3NoYXBfd2F0ZXJmYWxsX3tsYWJlbH0iKQogICAgICAgIGxvZ2dlci5pbmZvKGYiQ2FzZSAne2xhYmVsfSc6IGluZGV4PXtpZHh9LCBwcm9iPXt5X3Byb2JbaWR4XTouNGZ9LCB0cnVlX2xhYmVsPXt5X3Rlc3RbaWR4XX0iKQogICAgICAgIGNhc2Vfc2hhcFtsYWJlbF0gPSB7CiAgICAgICAgICAgICJzYW1wbGVfaWQiOiBzdHIodGVzdF9kZi5pbG9jW2lkeF1bInNhbXBsZV9pZCJdKSwKICAgICAgICAgICAgInF1ZXN0aW9uIjogc3RyKHRlc3RfZGYuaWxvY1tpZHhdWyJxdWVzdGlvbiJdKVs6MjAwXSwKICAgICAgICAgICAgImFuc3dlciI6IHN0cih0ZXN0X2RmLmlsb2NbaWR4XVsiYW5zd2VyIl0pWzoyMDBdLAogICAgICAgICAgICAicHJvYmFiaWxpdHkiOiBmbG9hdCh5X3Byb2JbaWR4XSksCiAgICAgICAgICAgICJ0cnVlX2xhYmVsIjogaW50KHlfdGVzdFtpZHhdKSwKICAgICAgICB9CgogICAgIyAtLS0tIENhbGlicmF0aW9uICYgcmFua2luZyBmaWd1cmVzIC0tLS0KICAgIGVjZV92YWwsIGJyaWVyX3ZhbCA9IHBsb3RfcmVsaWFiaWxpdHkoeV90ZXN0LCB5X3Byb2IsICJmaWdfcmVsaWFiaWxpdHkiKQogICAgcGxvdF9yb2NfcHIoeV90ZXN0LCB5X3Byb2IsICJmaWdfcm9jX3ByIikKCiAgICAjIC0tLS0gTWFjaGluZS1yZWFkYWJsZSBzdW1tYXJ5IGZvciB0aGUgZGFzaGJvYXJkIC0tLS0KICAgIHN1bW1hcnkgPSB7CiAgICAgICAgIm1vZGVsX3ZlcnNpb24iOiAieGdib29zdC12MS4wIiwKICAgICAgICAibl90ZXN0IjogaW50KGxlbihYX3Rlc3QpKSwKICAgICAgICAiZWNlIjogZWNlX3ZhbCwKICAgICAgICAiYnJpZXIiOiBicmllcl92YWwsCiAgICAgICAgInRvcF9mZWF0dXJlcyI6IFsKICAgICAgICAgICAgeyJmZWF0dXJlIjogZmVhdHVyZV9jb2xzW2ldLCAibWVhbl9hYnNfc2hhcCI6IGZsb2F0KG1lYW5fYWJzW2ldKX0KICAgICAgICAgICAgZm9yIGkgaW4gb3JkZXJbOk5fVE9QX0ZFQVRVUkVTXQogICAgICAgIF0sCiAgICAgICAgImNhc2VzIjogY2FzZV9zaGFwLAogICAgfQogICAgd2l0aCBvcGVuKFJFU1VMVFNfRElSIC8gInNoYXBfc3VtbWFyeS5qc29uIiwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChzdW1tYXJ5LCBmLCBpbmRlbnQ9MikKICAgIGxvZ2dlci5pbmZvKGYiU2F2ZWQgc2hhcF9zdW1tYXJ5Lmpzb24gYW5kIGZpZ3VyZXMgdG8ge0ZJR1VSRVNfRElSfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoUk9PVCkpCiAgICBtYWluKCkK",
 "src/features/entity_features.py": "IiIiCkVudGl0eSAoTkVSKSBmZWF0dXJlIGV4dHJhY3Rpb24gZm9yIEhhbHVSSVNDLgoKR3JvdXAgMyBmZWF0dXJlcyAocm9hZG1hcCDCpzYpOgogIG5fZW50aXRpZXNfYW5zd2VyLCBuX2VudGl0aWVzX2NvbnRleHQsIGVudGl0eV9vdmVybGFwX3JhdGlvLCBub3ZlbF9lbnRpdHlfcmF0aW8KClVzZXMgc3BhQ3kgYGVuX2NvcmVfd2ViX3NtYCBmb3IgbmFtZWQgZW50aXR5IHJlY29nbml0aW9uLgpFbXB0eSBjb250ZXh0IC0+IGFuc3dlciBlbnRpdGllcyBhcmUgYWxsIG5vdmVsIChvdmVybGFwIDApLiBObyBlbnRpdGllcyBpbgphbnN3ZXIgLT4gbm8gdW5zdXBwb3J0ZWQtZW50aXR5IHNpZ25hbCAob3ZlcmxhcCAxLjAsIG5vdmVsIDAuMCkuCiIiIgoKaW1wb3J0IGxvZ2dpbmcKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCgppbXBvcnQgcGFuZGFzIGFzIHBkCgpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykKCk1PREVMX05BTUUgPSAiZW5fY29yZV93ZWJfc20iCgoKZGVmIGxvYWRfbmVyX21vZGVsKCk6CiAgICAiIiJMb2FkIHRoZSBzcGFDeSBORVIgcGlwZWxpbmUgKGxhenksIGNhY2hlZCBhdCBjYWxsIHNpdGUpLiIiIgogICAgaW1wb3J0IHNwYWN5CgogICAgcmV0dXJuIHNwYWN5LmxvYWQoTU9ERUxfTkFNRSkKCgpkZWYgZXh0cmFjdF9lbnRpdHlfZmVhdHVyZXMocXVlc3Rpb246IHN0ciwgY29udGV4dDogc3RyLCBhbnN3ZXI6IHN0ciwgbmxwKSAtPiBkaWN0OgogICAgIiIiR3JvdXAgMzogbmFtZWQtZW50aXR5IG92ZXJsYXAgYmV0d2VlbiBhbnN3ZXIgYW5kIGNvbnRleHQuIiIiCiAgICBhbnNfZW50aXRpZXMgPSB7ZS50ZXh0Lmxvd2VyKCkgZm9yIGUgaW4gbmxwKGFuc3dlcikuZW50c30KICAgIGN0eF9lbnRpdGllcyA9IHtlLnRleHQubG93ZXIoKSBmb3IgZSBpbiBubHAoY29udGV4dCkuZW50c30gaWYgY29udGV4dC5zdHJpcCgpIGVsc2Ugc2V0KCkKCiAgICBuX2FucyA9IGxlbihhbnNfZW50aXRpZXMpCiAgICBuX2N0eCA9IGxlbihjdHhfZW50aXRpZXMpCgogICAgaWYgbl9hbnMgPT0gMDoKICAgICAgICBlbnRpdHlfb3ZlcmxhcF9yYXRpbyA9IDEuMAogICAgICAgIG5vdmVsX2VudGl0eV9yYXRpbyA9IDAuMAogICAgZWxzZToKICAgICAgICBpbnRlciA9IGFuc19lbnRpdGllcy5pbnRlcnNlY3Rpb24oY3R4X2VudGl0aWVzKQogICAgICAgIGVudGl0eV9vdmVybGFwX3JhdGlvID0gbGVuKGludGVyKSAvIG5fYW5zCiAgICAgICAgbm92ZWxfZW50aXR5X3JhdGlvID0gKG5fYW5zIC0gbGVuKGludGVyKSkgLyBuX2FucwoKICAgIHJldHVybiB7CiAgICAgICAgIm5fZW50aXRpZXNfYW5zd2VyIjogbl9hbnMsCiAgICAgICAgIm5fZW50aXRpZXNfY29udGV4dCI6IG5fY3R4LAogICAgICAgICJlbnRpdHlfb3ZlcmxhcF9yYXRpbyI6IHJvdW5kKGZsb2F0KGVudGl0eV9vdmVybGFwX3JhdGlvKSwgNiksCiAgICAgICAgIm5vdmVsX2VudGl0eV9yYXRpbyI6IHJvdW5kKGZsb2F0KG5vdmVsX2VudGl0eV9yYXRpbyksIDYpLAogICAgfQoKCmRlZiBleHRyYWN0X2VudGl0eV9mZWF0dXJlc19kZihkZjogcGQuRGF0YUZyYW1lLCBubHAsIGJhdGNoX3NpemU6IGludCA9IDY0KSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJCYXRjaCBlbnRpdHkgZmVhdHVyZXMgZm9yIGEgRGF0YUZyYW1lIHdpdGggcXVlc3Rpb24vY29udGV4dC9hbnN3ZXIgY29sdW1ucy4iIiIKICAgIGxvZ2dlci5pbmZvKGYiRXh0cmFjdGluZyBlbnRpdHkgKE5FUikgZmVhdHVyZXMgZm9yIHtsZW4oZGYpfSBzYW1wbGVzLi4uIikKICAgIHJvd3MgPSBbXQogICAgZm9yIF8sIHJvdyBpbiBkZi5pdGVycm93cygpOgogICAgICAgIHJvd3MuYXBwZW5kKGV4dHJhY3RfZW50aXR5X2ZlYXR1cmVzKHJvd1sicXVlc3Rpb24iXSwgcm93WyJjb250ZXh0Il0sIHJvd1siYW5zd2VyIl0sIG5scCkpCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MsIGluZGV4PWRmLmluZGV4KQo=",
 "src/features/extract_features.py": "IiIiDQpGZWF0dXJlIGV4dHJhY3Rpb24gcGlwZWxpbmUgZm9yIEhhbHVSSVNDLg0KQ29tcHV0ZXMgbGlnaHR3ZWlnaHQgbGluZ3Vpc3RpYywgbGV4aWNhbCBvdmVybGFwLCBudW1lcmljIGNvbnNpc3RlbmN5LCBhbmQgaGVkZ2luZyBmZWF0dXJlcy4NClByZXBhcmVzIG1vZHVsYXIgYXJjaGl0ZWN0dXJlIGZvciBORVIsIFNlbWFudGljLCBhbmQgTkxJIGZlYXR1cmVzLg0KIiIiDQoNCmltcG9ydCBvcw0KaW1wb3J0IHJlDQppbXBvcnQganNvbg0KaW1wb3J0IGxvZ2dpbmcNCmltcG9ydCBwYW5kYXMgYXMgcGQNCmltcG9ydCBudW1weSBhcyBucA0KDQpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcyIpDQoNCiMgTGV4aWNvbiBvZiBoZWRnaW5nIC8gdW5jZXJ0YWludHkgaW5kaWNhdG9ycw0KSEVER0VfTEVYSUNPTiA9IHsNCiAgICAibWF5YmUiLCAibWlnaHQiLCAibGlrZWx5IiwgInBvc3NpYmx5IiwgInByb2JhYmx5IiwgImNvdWxkIiwgInNlZW1zIiwgDQogICAgInVuY2VydGFpbiIsICJ1bmNsZWFyIiwgImFsbGVnZWRseSIsICJyZXBvcnRlZGx5IiwgInByZXN1bWFibHkiLCANCiAgICAic3VwcG9zZWRseSIsICJpIHRoaW5rIiwgImkgYmVsaWV2ZSIsICJhcHBlYXJzIHRvIiwgIml0IHNlZW1zIg0KfQ0KDQpkZWYgZXh0cmFjdF9sZW5ndGhfZmVhdHVyZXMocXVlc3Rpb246IHN0ciwgY29udGV4dDogc3RyLCBhbnN3ZXI6IHN0cikgLT4gZGljdDoNCiAgICAiIiJHcm91cCAxOiBMZW5ndGggYW5kIHN0eWxpc3RpYyBmZWF0dXJlcy4iIiINCiAgICB3b3JkcyA9IGFuc3dlci5zcGxpdCgpDQogICAgbl93b3JkcyA9IGxlbih3b3JkcykNCiAgICBuX2NoYXJzID0gbGVuKGFuc3dlcikNCiAgICBzZW50ZW5jZXMgPSBbcyBmb3IgcyBpbiByZS5zcGxpdChyJ1suIT9dKycsIGFuc3dlcikgaWYgcy5zdHJpcCgpXQ0KICAgIG5fc2VudGVuY2VzID0gbWF4KDEsIGxlbihzZW50ZW5jZXMpKQ0KICAgIGF2Z193b3JkX2xlbiA9IG5fY2hhcnMgLyBtYXgoMSwgbl93b3JkcykNCg0KICAgIHJldHVybiB7DQogICAgICAgICJuX2NoYXJzIjogbl9jaGFycywNCiAgICAgICAgIm5fd29yZHMiOiBuX3dvcmRzLA0KICAgICAgICAibl9zZW50ZW5jZXMiOiBuX3NlbnRlbmNlcywNCiAgICAgICAgImF2Z193b3JkX2xlbiI6IGF2Z193b3JkX2xlbiwNCiAgICB9DQoNCmRlZiBleHRyYWN0X2xleGljYWxfZmVhdHVyZXMocXVlc3Rpb246IHN0ciwgY29udGV4dDogc3RyLCBhbnN3ZXI6IHN0cikgLT4gZGljdDoNCiAgICAiIiJHcm91cCAyOiBMZXhpY2FsIG92ZXJsYXAgYW5kIGdyb3VuZGluZyBmZWF0dXJlcy4iIiINCiAgICBhbnNfdG9rZW5zID0gc2V0KHJlLmZpbmRhbGwocidcdysnLCBhbnN3ZXIubG93ZXIoKSkpDQogICAgY3R4X3Rva2VucyA9IHNldChyZS5maW5kYWxsKHInXHcrJywgY29udGV4dC5sb3dlcigpKSkNCiAgICBxX3Rva2VucyA9IHNldChyZS5maW5kYWxsKHInXHcrJywgcXVlc3Rpb24ubG93ZXIoKSkpDQoNCiAgICBpZiBub3QgYW5zX3Rva2VuczoNCiAgICAgICAgcmV0dXJuIHsNCiAgICAgICAgICAgICJvdmVybGFwX2Fuc3dlcl9jb250ZXh0IjogMC4wLA0KICAgICAgICAgICAgIm92ZXJsYXBfYW5zd2VyX3F1ZXN0aW9uIjogMC4wLA0KICAgICAgICAgICAgImphY2NhcmRfYW5zX2N0eCI6IDAuMCwNCiAgICAgICAgICAgICJqYWNjYXJkX2Fuc19xIjogMC4wDQogICAgICAgIH0NCg0KICAgIGFuc19jdHhfaW50ZXJzZWN0ID0gYW5zX3Rva2Vucy5pbnRlcnNlY3Rpb24oY3R4X3Rva2VucykNCiAgICBhbnNfcV9pbnRlcnNlY3QgPSBhbnNfdG9rZW5zLmludGVyc2VjdGlvbihxX3Rva2VucykNCg0KICAgIG92ZXJsYXBfYW5zX2N0eCA9IGxlbihhbnNfY3R4X2ludGVyc2VjdCkgLyBsZW4oYW5zX3Rva2VucykNCiAgICBvdmVybGFwX2Fuc19xID0gbGVuKGFuc19xX2ludGVyc2VjdCkgLyBsZW4oYW5zX3Rva2VucykNCg0KICAgIHVuaW9uX2Fuc19jdHggPSBhbnNfdG9rZW5zLnVuaW9uKGN0eF90b2tlbnMpDQogICAgamFjY2FyZF9hbnNfY3R4ID0gbGVuKGFuc19jdHhfaW50ZXJzZWN0KSAvIG1heCgxLCBsZW4odW5pb25fYW5zX2N0eCkpDQoNCiAgICB1bmlvbl9hbnNfcSA9IGFuc190b2tlbnMudW5pb24ocV90b2tlbnMpDQogICAgamFjY2FyZF9hbnNfcSA9IGxlbihhbnNfcV9pbnRlcnNlY3QpIC8gbWF4KDEsIGxlbih1bmlvbl9hbnNfcSkpDQoNCiAgICByZXR1cm4gew0KICAgICAgICAib3ZlcmxhcF9hbnN3ZXJfY29udGV4dCI6IG92ZXJsYXBfYW5zX2N0eCwNCiAgICAgICAgIm92ZXJsYXBfYW5zd2VyX3F1ZXN0aW9uIjogb3ZlcmxhcF9hbnNfcSwNCiAgICAgICAgImphY2NhcmRfYW5zX2N0eCI6IGphY2NhcmRfYW5zX2N0eCwNCiAgICAgICAgImphY2NhcmRfYW5zX3EiOiBqYWNjYXJkX2Fuc19xDQogICAgfQ0KDQpkZWYgZXh0cmFjdF9udW1lcmljX2ZlYXR1cmVzKHF1ZXN0aW9uOiBzdHIsIGNvbnRleHQ6IHN0ciwgYW5zd2VyOiBzdHIpIC0+IGRpY3Q6DQogICAgIiIiR3JvdXAgNTogTnVtZXJpYyBjb25zaXN0ZW5jeSBmZWF0dXJlcy4iIiINCiAgICBudW1fcGF0dGVybiA9IHInXGJcZCsoPzpcLlxkKyk/JT9cYicNCiAgICBhbnNfbnVtcyA9IHNldChyZS5maW5kYWxsKG51bV9wYXR0ZXJuLCBhbnN3ZXIpKQ0KICAgIGN0eF9udW1zID0gc2V0KHJlLmZpbmRhbGwobnVtX3BhdHRlcm4sIGNvbnRleHQpKQ0KDQogICAgbl9udW1zX2FucyA9IGxlbihhbnNfbnVtcykNCiAgICBuX251bXNfY3R4ID0gbGVuKGN0eF9udW1zKQ0KDQogICAgaWYgbl9udW1zX2FucyA9PSAwOg0KICAgICAgICByZXR1cm4gew0KICAgICAgICAgICAgIm5fbnVtYmVyc19hbnN3ZXIiOiAwLA0KICAgICAgICAgICAgIm5fbnVtYmVyc19jb250ZXh0Ijogbl9udW1zX2N0eCwNCiAgICAgICAgICAgICJudW1iZXJfb3ZlcmxhcF9yYXRpbyI6IDEuMCwgICMgTm8gbnVtYmVycyBpbiBhbnN3ZXIgLT4gbm8gbnVtZXJpYyBoYWxsdWNpbmF0aW9uDQogICAgICAgICAgICAibm92ZWxfbnVtYmVycyI6IDANCiAgICAgICAgfQ0KDQogICAgb3ZlcmxhcF9udW1zID0gYW5zX251bXMuaW50ZXJzZWN0aW9uKGN0eF9udW1zKQ0KICAgIG92ZXJsYXBfcmF0aW8gPSBsZW4ob3ZlcmxhcF9udW1zKSAvIG5fbnVtc19hbnMNCiAgICBub3ZlbF9udW1zID0gbGVuKGFuc19udW1zIC0gY3R4X251bXMpDQoNCiAgICByZXR1cm4gew0KICAgICAgICAibl9udW1iZXJzX2Fuc3dlciI6IG5fbnVtc19hbnMsDQogICAgICAgICJuX251bWJlcnNfY29udGV4dCI6IG5fbnVtc19jdHgsDQogICAgICAgICJudW1iZXJfb3ZlcmxhcF9yYXRpbyI6IG92ZXJsYXBfcmF0aW8sDQogICAgICAgICJub3ZlbF9udW1iZXJzIjogbm92ZWxfbnVtcw0KICAgIH0NCg0KZGVmIGV4dHJhY3RfaGVkZ2luZ19mZWF0dXJlcyhxdWVzdGlvbjogc3RyLCBjb250ZXh0OiBzdHIsIGFuc3dlcjogc3RyKSAtPiBkaWN0Og0KICAgICIiIkdyb3VwIDY6IEhlZGdpbmcgYW5kIHVuY2VydGFpbnR5IGZlYXR1cmVzLiIiIg0KICAgIGFuc19sb3dlciA9IGFuc3dlci5sb3dlcigpDQogICAgaGVkZ2VfY291bnQgPSAwDQogICAgZm9yIGhlZGdlIGluIEhFREdFX0xFWElDT046DQogICAgICAgIGlmIGhlZGdlIGluIGFuc19sb3dlcjoNCiAgICAgICAgICAgIGhlZGdlX2NvdW50ICs9IGFuc19sb3dlci5jb3VudChoZWRnZSkNCg0KICAgIHdvcmRzID0gYW5zX2xvd2VyLnNwbGl0KCkNCiAgICBoZWRnZV9kZW5zaXR5ID0gaGVkZ2VfY291bnQgLyBtYXgoMSwgbGVuKHdvcmRzKSkNCg0KICAgIHJldHVybiB7DQogICAgICAgICJoZWRnZV9jb3VudCI6IGhlZGdlX2NvdW50LA0KICAgICAgICAiaGVkZ2VfZGVuc2l0eSI6IGhlZGdlX2RlbnNpdHkNCiAgICB9DQoNCmRlZiBleHRyYWN0X2FsbF9jb3JlX2ZlYXR1cmVzKGRmOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToNCiAgICAiIiJDb21wdXRlcyBhbGwgY29yZSBmZWF0dXJlcyBmb3IgYSBEYXRhRnJhbWUgY29udGFpbmluZyBxdWVzdGlvbiwgY29udGV4dCwgYW5zd2VyLiIiIg0KICAgIGxvZ2dpbmcuaW5mbyhmIkV4dHJhY3RpbmcgY29yZSBmZWF0dXJlcyBmb3Ige2xlbihkZil9IHNhbXBsZXMuLi4iKQ0KICAgIGZlYXR1cmVfcm93cyA9IFtdDQoNCiAgICBmb3IgaWR4LCByb3cgaW4gZGYuaXRlcnJvd3MoKToNCiAgICAgICAgcSwgYywgYSA9IHJvd1sicXVlc3Rpb24iXSwgcm93WyJjb250ZXh0Il0sIHJvd1siYW5zd2VyIl0NCiAgICAgICAgZmVhdHMgPSB7fQ0KICAgICAgICBmZWF0cy51cGRhdGUoZXh0cmFjdF9sZW5ndGhfZmVhdHVyZXMocSwgYywgYSkpDQogICAgICAgIGZlYXRzLnVwZGF0ZShleHRyYWN0X2xleGljYWxfZmVhdHVyZXMocSwgYywgYSkpDQogICAgICAgIGZlYXRzLnVwZGF0ZShleHRyYWN0X251bWVyaWNfZmVhdHVyZXMocSwgYywgYSkpDQogICAgICAgIGZlYXRzLnVwZGF0ZShleHRyYWN0X2hlZGdpbmdfZmVhdHVyZXMocSwgYywgYSkpDQogICAgICAgIGZlYXR1cmVfcm93cy5hcHBlbmQoZmVhdHMpDQoNCiAgICBmZWF0dXJlc19kZiA9IHBkLkRhdGFGcmFtZShmZWF0dXJlX3Jvd3MsIGluZGV4PWRmLmluZGV4KQ0KICAgIHJlc3VsdF9kZiA9IHBkLmNvbmNhdChbZGZbWyJzYW1wbGVfaWQiLCAiaXRlbV9pZHgiLCAibGFiZWwiLCAic3BsaXQiXV0sIGZlYXR1cmVzX2RmXSwgYXhpcz0xKQ0KICAgIGxvZ2dpbmcuaW5mbyhmIlN1Y2Nlc3NmdWxseSBleHRyYWN0ZWQge2ZlYXR1cmVzX2RmLnNoYXBlWzFdfSBjb3JlIGZlYXR1cmVzLiIpDQogICAgcmV0dXJuIHJlc3VsdF9kZg0KDQoNCmRlZiBsb2FkX2hlYXZ5X21vZGVscyhubGlfbW9kZWxfbmFtZTogc3RyIHwgTm9uZSA9IE5vbmUsIGRldmljZTogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Q6DQogICAgIiIiTG9hZCBORVIgKyBOTEkgKyBlbWJlZGRpbmcgbW9kZWxzIG9uY2UgKHVzZWQgYnkgYmF0Y2ggZXh0cmFjdGlvbiBhbmQgQVBJKS4NCg0KICAgIGRldmljZT1Ob25lIC0+IGxpYnJhcnkgZGVmYXVsdDsgcGFzcyAiY3B1IiBmb3Igc3RhYmlsaXR5IChubyBWUkFNIE9PTSkuDQogICAgT24gQ1VEQSwgbW9kZWxzIGFyZSBsb2FkZWQgaW4gZmxvYXQxNiB0byBoYWx2ZSBWUkFNIGFuZCByZWR1Y2UgT09NIHJpc2sNCiAgICBvbiBzbWFsbCBHUFVzIChSVFggMzA2MCA2IEdCKS4NCiAgICAiIiINCiAgICBpbXBvcnQgdGltZQ0KDQogICAgaW1wb3J0IHRvcmNoDQoNCiAgICBpZiBkZXZpY2UgPT0gImN1ZGEiIGFuZCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOg0KICAgICAgICBtb2RlbF9rd2FyZ3MgPSB7InRvcmNoX2R0eXBlIjogdG9yY2guZmxvYXQxNn0NCiAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpDQogICAgZWxzZToNCiAgICAgICAgbW9kZWxfa3dhcmdzID0gTm9uZQ0KDQogICAgZnJvbSBzcmMuZmVhdHVyZXMuZW50aXR5X2ZlYXR1cmVzIGltcG9ydCBsb2FkX25lcl9tb2RlbA0KICAgIGZyb20gc3JjLmZlYXR1cmVzLm5saV9mZWF0dXJlcyBpbXBvcnQgbG9hZF9ubGlfbW9kZWwNCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5zZW1hbnRpY19mZWF0dXJlcyBpbXBvcnQgbG9hZF9lbWJlZGRpbmdfbW9kZWwNCg0KICAgIG1vZGVscyA9IHt9DQogICAgdDAgPSB0aW1lLnRpbWUoKQ0KICAgIG1vZGVsc1sibmxwIl0gPSBsb2FkX25lcl9tb2RlbCgpDQogICAgbG9nZ2luZy5pbmZvKGYiTkVSIG1vZGVsIGxvYWRlZCBpbiB7dGltZS50aW1lKCkgLSB0MDouMWZ9cyIpDQogICAgdDAgPSB0aW1lLnRpbWUoKQ0KICAgIG1vZGVsc1sibmxpIl0sIG1vZGVsc1sibmxpX25hbWUiXSA9IGxvYWRfbmxpX21vZGVsKG5saV9tb2RlbF9uYW1lLCBkZXZpY2U9ZGV2aWNlLCBtb2RlbF9rd2FyZ3M9bW9kZWxfa3dhcmdzKQ0KICAgIGxvZ2dpbmcuaW5mbyhmIk5MSSBtb2RlbCAoe21vZGVsc1snbmxpX25hbWUnXX0pIGxvYWRlZCBpbiB7dGltZS50aW1lKCkgLSB0MDouMWZ9cyIpDQogICAgdDAgPSB0aW1lLnRpbWUoKQ0KICAgIG1vZGVsc1siZW1iZWRkZXIiXSA9IGxvYWRfZW1iZWRkaW5nX21vZGVsKGRldmljZT1kZXZpY2UsIG1vZGVsX2t3YXJncz1tb2RlbF9rd2FyZ3MpDQogICAgbG9nZ2luZy5pbmZvKGYiRW1iZWRkaW5nIG1vZGVsIGxvYWRlZCBpbiB7dGltZS50aW1lKCkgLSB0MDouMWZ9cyIpDQogICAgaWYgbW9kZWxfa3dhcmdzIGlzIG5vdCBOb25lOg0KICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkNCiAgICByZXR1cm4gbW9kZWxzDQoNCg0KZGVmIGV4dHJhY3RfYWxsX2ZlYXR1cmVzX3NpbmdsZShxdWVzdGlvbjogc3RyLCBjb250ZXh0OiBzdHIsIGFuc3dlcjogc3RyLCBtb2RlbHM6IGRpY3QpIC0+IGRpY3Q6DQogICAgIiIiQWxsIDcgZmVhdHVyZSBncm91cHMgZm9yIG9uZSBzYW1wbGUuIFVzZWQgYnkgdGhlIEZhc3RBUEkgaW5mZXJlbmNlIHNlcnZlci4iIiINCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5lbnRpdHlfZmVhdHVyZXMgaW1wb3J0IGV4dHJhY3RfZW50aXR5X2ZlYXR1cmVzDQogICAgZnJvbSBzcmMuZmVhdHVyZXMubmxpX2ZlYXR1cmVzIGltcG9ydCBleHRyYWN0X25saV9mZWF0dXJlcw0KICAgIGZyb20gc3JjLmZlYXR1cmVzLnNlbWFudGljX2ZlYXR1cmVzIGltcG9ydCBleHRyYWN0X3NlbWFudGljX2ZlYXR1cmVzDQoNCiAgICBmZWF0cyA9IHt9DQogICAgZmVhdHMudXBkYXRlKGV4dHJhY3RfbGVuZ3RoX2ZlYXR1cmVzKHF1ZXN0aW9uLCBjb250ZXh0LCBhbnN3ZXIpKQ0KICAgIGZlYXRzLnVwZGF0ZShleHRyYWN0X2xleGljYWxfZmVhdHVyZXMocXVlc3Rpb24sIGNvbnRleHQsIGFuc3dlcikpDQogICAgZmVhdHMudXBkYXRlKGV4dHJhY3RfbnVtZXJpY19mZWF0dXJlcyhxdWVzdGlvbiwgY29udGV4dCwgYW5zd2VyKSkNCiAgICBmZWF0cy51cGRhdGUoZXh0cmFjdF9oZWRnaW5nX2ZlYXR1cmVzKHF1ZXN0aW9uLCBjb250ZXh0LCBhbnN3ZXIpKQ0KICAgIGZlYXRzLnVwZGF0ZShleHRyYWN0X2VudGl0eV9mZWF0dXJlcyhxdWVzdGlvbiwgY29udGV4dCwgYW5zd2VyLCBtb2RlbHNbIm5scCJdKSkNCiAgICBmZWF0cy51cGRhdGUoZXh0cmFjdF9ubGlfZmVhdHVyZXMocXVlc3Rpb24sIGNvbnRleHQsIGFuc3dlciwgbW9kZWxzWyJubGkiXSkpDQogICAgZmVhdHMudXBkYXRlKGV4dHJhY3Rfc2VtYW50aWNfZmVhdHVyZXMocXVlc3Rpb24sIGNvbnRleHQsIGFuc3dlciwgbW9kZWxzWyJlbWJlZGRlciJdKSkNCiAgICByZXR1cm4gZmVhdHMNCg0KDQpkZWYgZXh0cmFjdF9mdWxsX2ZlYXR1cmVfc2V0KGRmOiBwZC5EYXRhRnJhbWUsIG1vZGVsczogZGljdCB8IE5vbmUgPSBOb25lLCBiYXRjaF9zaXplOiBpbnQgPSAxMjgpIC0+IHBkLkRhdGFGcmFtZToNCiAgICAiIiJFeHRyYWN0cyBhbGwgNyBmZWF0dXJlIGdyb3VwcyAoY29yZSArIGVudGl0eSArIE5MSSArIHNlbWFudGljKSB3aXRoIHBlci1ncm91cCBsYXRlbmN5Lg0KDQogICAgYmF0Y2hfc2l6ZSBjb250cm9scyB0aGUgTkxJL2VtYmVkZGluZyBpbmZlcmVuY2UgYmF0Y2ggKGxhcmdlciBvbiBHUFUgPSBmYXN0ZXIpLg0KICAgICIiIg0KICAgIGltcG9ydCB0aW1lDQoNCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5lbnRpdHlfZmVhdHVyZXMgaW1wb3J0IGV4dHJhY3RfZW50aXR5X2ZlYXR1cmVzX2RmDQogICAgZnJvbSBzcmMuZmVhdHVyZXMubmxpX2ZlYXR1cmVzIGltcG9ydCBleHRyYWN0X25saV9mZWF0dXJlc19kZg0KICAgIGZyb20gc3JjLmZlYXR1cmVzLnNlbWFudGljX2ZlYXR1cmVzIGltcG9ydCBleHRyYWN0X3NlbWFudGljX2ZlYXR1cmVzX2RmDQoNCiAgICBpZiBtb2RlbHMgaXMgTm9uZToNCiAgICAgICAgbW9kZWxzID0gbG9hZF9oZWF2eV9tb2RlbHMoKQ0KDQogICAgcmVzdWx0X2RmID0gZXh0cmFjdF9hbGxfY29yZV9mZWF0dXJlcyhkZikNCg0KICAgIHQwID0gdGltZS50aW1lKCkNCiAgICBlbnRpdHlfZGYgPSBleHRyYWN0X2VudGl0eV9mZWF0dXJlc19kZihkZiwgbW9kZWxzWyJubHAiXSkNCiAgICBsb2dnaW5nLmluZm8oZiJFbnRpdHkgZmVhdHVyZXMgZG9uZSBpbiB7dGltZS50aW1lKCkgLSB0MDouMWZ9cyIpDQoNCiAgICB0MCA9IHRpbWUudGltZSgpDQogICAgbmxpX2RmID0gZXh0cmFjdF9ubGlfZmVhdHVyZXNfZGYoZGYsIG1vZGVsc1sibmxpIl0sIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSkNCiAgICBsb2dnaW5nLmluZm8oZiJOTEkgZmVhdHVyZXMgZG9uZSBpbiB7dGltZS50aW1lKCkgLSB0MDouMWZ9cyAoYmF0Y2g9e2JhdGNoX3NpemV9KSIpDQoNCiAgICB0MCA9IHRpbWUudGltZSgpDQogICAgc2VtYW50aWNfZGYgPSBleHRyYWN0X3NlbWFudGljX2ZlYXR1cmVzX2RmKGRmLCBtb2RlbHNbImVtYmVkZGVyIl0sIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSkNCiAgICBsb2dnaW5nLmluZm8oZiJTZW1hbnRpYyBmZWF0dXJlcyBkb25lIGluIHt0aW1lLnRpbWUoKSAtIHQwOi4xZn1zIChiYXRjaD17YmF0Y2hfc2l6ZX0pIikNCg0KICAgIHJlc3VsdF9kZiA9IHBkLmNvbmNhdChbcmVzdWx0X2RmLCBlbnRpdHlfZGYsIG5saV9kZiwgc2VtYW50aWNfZGZdLCBheGlzPTEpDQogICAgbG9nZ2luZy5pbmZvKGYiRnVsbCBmZWF0dXJlIG1hdHJpeDoge3Jlc3VsdF9kZi5zaGFwZVsxXX0gZmVhdHVyZXMgdG90YWwuIikNCiAgICByZXR1cm4gcmVzdWx0X2RmDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICBpbXBvcnQgYXJncGFyc2UNCiAgICBpbXBvcnQgc3lzDQogICAgZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCiAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdKSkNCg0KICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJIYWx1UklTQyBmdWxsIGZlYXR1cmUgZXh0cmFjdGlvbiIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1pbnB1dCIsIGRlZmF1bHQ9b3MucGF0aC5qb2luKCJkYXRhIiwgInByb2Nlc3NlZCIsICJxYV9jbGVhbi5wYXJxdWV0IikpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiLCBkZWZhdWx0PW9zLnBhdGguam9pbigiZGF0YSIsICJwcm9jZXNzZWQiLCAiZmVhdHVyZXNfZnVsbC5wYXJxdWV0IikpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ubGktbW9kZWwiLCBkZWZhdWx0PU5vbmUsIGhlbHA9Ik92ZXJyaWRlIE5MSSBDcm9zc0VuY29kZXIgY2hlY2twb2ludCIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kZXZpY2UiLCBkZWZhdWx0PU5vbmUsIGhlbHA9ImN1ZGF8Y3B1IChkZWZhdWx0OiBhdXRvOyBjdWRhIGxvYWRzIG1vZGVscyBmcDE2KSIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1iYXRjaC1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTI4LCBoZWxwPSJOTEkvZW1iZWRkaW5nIGluZmVyZW5jZSBiYXRjaCAodXNlIDI1Nisgb24gR1BVKSIpDQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkNCg0KICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhhcmdzLmlucHV0KToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJ7YXJncy5pbnB1dH0gbm90IGZvdW5kLiBSdW4gc3JjL2RhdGEvcHJlcGFyZS5weSBmaXJzdC4iKQ0KDQogICAgZGYgPSBwZC5yZWFkX3BhcnF1ZXQoYXJncy5pbnB1dCkNCiAgICBtb2RlbHMgPSBsb2FkX2hlYXZ5X21vZGVscyhhcmdzLm5saV9tb2RlbCwgZGV2aWNlPWFyZ3MuZGV2aWNlKQ0KICAgIGZlYXR1cmVzX2RmID0gZXh0cmFjdF9mdWxsX2ZlYXR1cmVfc2V0KGRmLCBtb2RlbHMsIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplKQ0KICAgIGZlYXR1cmVzX2RmLnRvX3BhcnF1ZXQoYXJncy5vdXRwdXQsIGluZGV4PUZhbHNlKQ0KICAgIGxvZ2dpbmcuaW5mbyhmIlNhdmVkIGZ1bGwgZmVhdHVyZSBtYXRyaXggdG8ge2FyZ3Mub3V0cHV0fSIpDQoNCiAgICAjIFJlY29yZCB3aGljaCBOTEkgY2hlY2twb2ludCB3YXMgYWN0dWFsbHkgdXNlZCAocHJvdmVuYW5jZSBmb3IgcGFyYW1zLmpzb24vbWFuaWZlc3QpDQogICAgbmxpX3VzZWQgPSB7DQogICAgICAgICJubGlfbW9kZWwiOiBtb2RlbHMuZ2V0KCJubGlfbmFtZSIpLA0KICAgICAgICAiZGV2aWNlIjogc3RyKGdldGF0dHIobW9kZWxzLmdldCgibmxpIiksICJkZXZpY2UiLCAidW5rbm93biIpKSwNCiAgICAgICAgImJhdGNoX3NpemUiOiBhcmdzLmJhdGNoX3NpemUsDQogICAgICAgICJleHRyYWN0ZWRfYXQiOiBwZC5UaW1lc3RhbXAubm93KCJVVEMiKS5pc29mb3JtYXQoKSwNCiAgICB9DQogICAgb3MubWFrZWRpcnMob3MucGF0aC5qb2luKCJkYXRhIiwgInByb2Nlc3NlZCIpLCBleGlzdF9vaz1UcnVlKQ0KICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4oImRhdGEiLCAicHJvY2Vzc2VkIiwgIm5saV9tb2RlbF91c2VkLmpzb24iKSwgInciKSBhcyBmOg0KICAgICAgICBqc29uLmR1bXAobmxpX3VzZWQsIGYsIGluZGVudD0yKQ0KICAgIGxvZ2dpbmcuaW5mbyhmIlJlY29yZGVkIE5MSSBwcm92ZW5hbmNlOiB7bmxpX3VzZWRbJ25saV9tb2RlbCddfSIpDQo=",
 "src/features/nli_features.py": "IiIiCk5MSSBjb25zaXN0ZW5jeSBmZWF0dXJlIGV4dHJhY3Rpb24gZm9yIEhhbHVSSVNDLgoKR3JvdXAgNCBmZWF0dXJlcyAocm9hZG1hcCDCpzYsIG1hbmRhdG9yeSBwZXIgYmx1ZXByaW50IEE4KToKICBubGlfY3R4X2VudGFpbHNfYW5zLCBubGlfY3R4X2NvbnRyYWRpY3RzX2FucywgbmxpX2N0eF9uZXV0cmFsX2FucwogIG5saV9hbnNfZW50YWlsc19jdHgsIG5saV9hbnNfY29udHJhZGljdHNfY3R4LCBubGlfYW5zX25ldXRyYWxfY3R4CgpQcmltYXJ5IG1vZGVsOiBjcm9zcy1lbmNvZGVyL25saS1kZWJlcnRhLXYzLWJhc2UgKGJsdWVwcmludCBBOCkuCkZhbGxiYWNrIGlmIGRvd25sb2FkIGZhaWxzOiBjcm9zcy1lbmNvZGVyL25saS1NaW5pTE0yLUw2LUg3NjguCgpDcm9zc0VuY29kZXIgb3V0cHV0IGlzIGEgMy1jbGFzcyBzb2Z0bWF4IGluIG9yZGVyCltjb250cmFkaWN0aW9uLCBlbnRhaWxtZW50LCBuZXV0cmFsXSAoU05MSS9NdWx0aU5MSSBzY2hlbWEpLgoKRW1wdHkgY29udGV4dCAtPiBhbGwgdGhyZWUgcHJvYnMgc2V0IHRvIDEvMyAobmV1dHJhbCksIHBlciBwcmVwYXJlLnB5IHJ1bGUuCiIiIgoKaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmZyb20gdHlwaW5nIGltcG9ydCBPcHRpb25hbAoKaW1wb3J0IHBhbmRhcyBhcyBwZAoKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pCgpOTElfTU9ERUxfUFJJTUFSWSA9ICJjcm9zcy1lbmNvZGVyL25saS1kZWJlcnRhLXYzLWJhc2UiCk5MSV9NT0RFTF9GQUxMQkFDSyA9ICJjcm9zcy1lbmNvZGVyL25saS1NaW5pTE0yLUw2LUg3NjgiCk5MSV9NT0RFTF9FTlYgPSAiSEFMVV9OTElfTU9ERUwiCgpORVVUUkFMID0gMS4wIC8gMy4wCgojIENyb3NzRW5jb2RlciBsYWJlbCBvcmRlciBmb3IgTkxJIGNoZWNrcG9pbnRzCkxBQkVMUyA9IFsiY29udHJhZGljdGlvbiIsICJlbnRhaWxtZW50IiwgIm5ldXRyYWwiXQoKCmRlZiBfc2FmZV9kZXZpY2UoZGV2aWNlOiBPcHRpb25hbFtzdHJdKSAtPiBPcHRpb25hbFtzdHJdOgogICAgIiIiUmV0dXJuIHRoZSBkZXZpY2UgdG8gdXNlOyBjdWRhIGlzIG9ubHkgaG9ub3JlZCB3aGVuIHRvcmNoIHN1cHBvcnRzIGl0LiIiIgogICAgaWYgZGV2aWNlID09ICJjdWRhIjoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGltcG9ydCB0b3JjaAoKICAgICAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgICAgIHJldHVybiAiY3VkYSIKICAgICAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgICAgIHBhc3MKICAgICAgICBsb2dnZXIud2FybmluZygiZGV2aWNlPWN1ZGEgcmVxdWVzdGVkIGJ1dCB0b3JjaCBoYXMgbm8gQ1VEQSBzdXBwb3J0OyB1c2luZyBDUFUiKQogICAgICAgIHJldHVybiBOb25lCiAgICByZXR1cm4gZGV2aWNlCgoKZGVmIGxvYWRfbmxpX21vZGVsKG1vZGVsX25hbWU6IE9wdGlvbmFsW3N0cl0gPSBOb25lLCBkZXZpY2U6IE9wdGlvbmFsW3N0cl0gPSBOb25lLCBtb2RlbF9rd2FyZ3M6IE9wdGlvbmFsW2RpY3RdID0gTm9uZSk6CiAgICAiIiJMb2FkIHRoZSBOTEkgQ3Jvc3NFbmNvZGVyIChmYWxscyBiYWNrIHRvIE1pbmlMTTIgb24gZmFpbHVyZSkuCgogICAgZGV2aWNlPU5vbmUgLT4gbGlicmFyeSBkZWZhdWx0IChHUFUgaWYgYXZhaWxhYmxlKTsgc2V0ICJjcHUiIGZvciBzdGFiaWxpdHkuCiAgICBtb2RlbF9rd2FyZ3MgLT4gZXh0cmEga3dhcmdzIGZvciB0aGUgbW9kZWwgbG9hZGVyIChlLmcuIHRvcmNoX2R0eXBlPWZsb2F0MTYpLgogICAgIiIiCiAgICBmcm9tIHNlbnRlbmNlX3RyYW5zZm9ybWVycyBpbXBvcnQgQ3Jvc3NFbmNvZGVyCgogICAgZGV2aWNlID0gX3NhZmVfZGV2aWNlKGRldmljZSkKICAgIGNob3NlbiA9IG1vZGVsX25hbWUgb3Igb3MuZW52aXJvbi5nZXQoTkxJX01PREVMX0VOViwgTkxJX01PREVMX1BSSU1BUlkpCiAgICB0cnk6CiAgICAgICAgbG9nZ2VyLmluZm8oZiJMb2FkaW5nIE5MSSBDcm9zc0VuY29kZXI6IHtjaG9zZW59IChkZXZpY2U9e2RldmljZSBvciAnYXV0byd9KSAuLi4iKQogICAgICAgIG1vZGVsID0gQ3Jvc3NFbmNvZGVyKGNob3NlbiwgZGV2aWNlPWRldmljZSwgbW9kZWxfa3dhcmdzPW1vZGVsX2t3YXJncykKICAgICAgICBsb2dnZXIuaW5mbygiTkxJIENyb3NzRW5jb2RlciBsb2FkZWQuIikKICAgICAgICByZXR1cm4gbW9kZWwsIGNob3NlbgogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIGlmIGNob3NlbiAhPSBOTElfTU9ERUxfRkFMTEJBQ0s6CiAgICAgICAgICAgIGxvZ2dlci53YXJuaW5nKGYiTkxJIG1vZGVsIHtjaG9zZW59IGZhaWxlZCAoe2V9KTsgZmFsbGluZyBiYWNrIHRvIHtOTElfTU9ERUxfRkFMTEJBQ0t9IikKICAgICAgICAgICAgcmV0dXJuIGxvYWRfbmxpX21vZGVsKE5MSV9NT0RFTF9GQUxMQkFDSywgZGV2aWNlPWRldmljZSwgbW9kZWxfa3dhcmdzPW1vZGVsX2t3YXJncykKICAgICAgICByYWlzZQoKCmRlZiBfbmV1dHJhbF9yb3coKSAtPiBkaWN0OgogICAgcmV0dXJuIHsKICAgICAgICAibmxpX2N0eF9lbnRhaWxzX2FucyI6IE5FVVRSQUwsCiAgICAgICAgIm5saV9jdHhfY29udHJhZGljdHNfYW5zIjogTkVVVFJBTCwKICAgICAgICAibmxpX2N0eF9uZXV0cmFsX2FucyI6IE5FVVRSQUwsCiAgICAgICAgIm5saV9hbnNfZW50YWlsc19jdHgiOiBORVVUUkFMLAogICAgICAgICJubGlfYW5zX2NvbnRyYWRpY3RzX2N0eCI6IE5FVVRSQUwsCiAgICAgICAgIm5saV9hbnNfbmV1dHJhbF9jdHgiOiBORVVUUkFMLAogICAgfQoKCmRlZiBleHRyYWN0X25saV9mZWF0dXJlcyhxdWVzdGlvbjogc3RyLCBjb250ZXh0OiBzdHIsIGFuc3dlcjogc3RyLCBtb2RlbCkgLT4gZGljdDoKICAgICIiIkdyb3VwIDQ6IGVudGFpbG1lbnQvY29udHJhZGljdGlvbiBwcm9iYWJpbGl0aWVzLCBib3RoIGRpcmVjdGlvbnMuIiIiCiAgICBpZiBub3QgY29udGV4dC5zdHJpcCgpOgogICAgICAgIHJldHVybiBfbmV1dHJhbF9yb3coKQoKICAgIGxvZ2l0cyA9IG1vZGVsLnByZWRpY3QoW1tjb250ZXh0LCBhbnN3ZXJdLCBbYW5zd2VyLCBjb250ZXh0XV0sIGFwcGx5X3NvZnRtYXg9VHJ1ZSkKICAgIHByb2JzID0gbG9naXRzIGlmIGxvZ2l0cy5zaGFwZVsxXSA9PSAzIGVsc2UgTm9uZQogICAgaWYgcHJvYnMgaXMgTm9uZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5leHBlY3RlZCBOTEkgb3V0cHV0IHNoYXBlIHtsb2dpdHMuc2hhcGV9OyBleHBlY3RlZCAobiwgMykiKQoKICAgIHBfY3R4ID0ge0xBQkVMU1tpXTogZmxvYXQocHJvYnNbMCwgaV0pIGZvciBpIGluIHJhbmdlKDMpfQogICAgcF9hbnMgPSB7TEFCRUxTW2ldOiBmbG9hdChwcm9ic1sxLCBpXSkgZm9yIGkgaW4gcmFuZ2UoMyl9CgogICAgcmV0dXJuIHsKICAgICAgICAibmxpX2N0eF9lbnRhaWxzX2FucyI6IHJvdW5kKHBfY3R4WyJlbnRhaWxtZW50Il0sIDYpLAogICAgICAgICJubGlfY3R4X2NvbnRyYWRpY3RzX2FucyI6IHJvdW5kKHBfY3R4WyJjb250cmFkaWN0aW9uIl0sIDYpLAogICAgICAgICJubGlfY3R4X25ldXRyYWxfYW5zIjogcm91bmQocF9jdHhbIm5ldXRyYWwiXSwgNiksCiAgICAgICAgIm5saV9hbnNfZW50YWlsc19jdHgiOiByb3VuZChwX2Fuc1siZW50YWlsbWVudCJdLCA2KSwKICAgICAgICAibmxpX2Fuc19jb250cmFkaWN0c19jdHgiOiByb3VuZChwX2Fuc1siY29udHJhZGljdGlvbiJdLCA2KSwKICAgICAgICAibmxpX2Fuc19uZXV0cmFsX2N0eCI6IHJvdW5kKHBfYW5zWyJuZXV0cmFsIl0sIDYpLAogICAgfQoKCmRlZiBleHRyYWN0X25saV9mZWF0dXJlc19kZihkZjogcGQuRGF0YUZyYW1lLCBtb2RlbCwgYmF0Y2hfc2l6ZTogaW50ID0gNjQpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIkJhdGNoIE5MSSBmZWF0dXJlczsgcHJvY2Vzc2VzIGJvdGggKGN0eCwgYW5zKSBhbmQgKGFucywgY3R4KSBkaXJlY3Rpb25zLiIiIgogICAgbG9nZ2VyLmluZm8oZiJFeHRyYWN0aW5nIE5MSSBmZWF0dXJlcyBmb3Ige2xlbihkZil9IHNhbXBsZXMgKDIgZGlyZWN0aW9ucyBlYWNoKS4uLiIpCiAgICByb3dzID0gW10KICAgIGJhdGNoX2N0eF9hbnMsIGJhdGNoX2Fuc19jdHgsIGJhdGNoX2lkeCA9IFtdLCBbXSwgW10KCiAgICBkZWYgZmx1c2goKToKICAgICAgICBub25sb2NhbCBiYXRjaF9jdHhfYW5zLCBiYXRjaF9hbnNfY3R4LCBiYXRjaF9pZHgKICAgICAgICBpZiBub3QgYmF0Y2hfaWR4OgogICAgICAgICAgICByZXR1cm4KICAgICAgICBhbGxfcHJvYnMgPSBtb2RlbC5wcmVkaWN0KGJhdGNoX2N0eF9hbnMgKyBiYXRjaF9hbnNfY3R4LCBiYXRjaF9zaXplPWJhdGNoX3NpemUsIGFwcGx5X3NvZnRtYXg9VHJ1ZSkKICAgICAgICBuID0gbGVuKGJhdGNoX2lkeCkKICAgICAgICBmb3IgaywgaWR4IGluIGVudW1lcmF0ZShiYXRjaF9pZHgpOgogICAgICAgICAgICBwX2N0eCA9IHtMQUJFTFNbaV06IGZsb2F0KGFsbF9wcm9ic1trLCBpXSkgZm9yIGkgaW4gcmFuZ2UoMyl9CiAgICAgICAgICAgIHBfYW5zID0ge0xBQkVMU1tpXTogZmxvYXQoYWxsX3Byb2JzW24gKyBrLCBpXSkgZm9yIGkgaW4gcmFuZ2UoMyl9CiAgICAgICAgICAgIHJvd3MuYXBwZW5kKChpZHgsIHsKICAgICAgICAgICAgICAgICJubGlfY3R4X2VudGFpbHNfYW5zIjogcm91bmQocF9jdHhbImVudGFpbG1lbnQiXSwgNiksCiAgICAgICAgICAgICAgICAibmxpX2N0eF9jb250cmFkaWN0c19hbnMiOiByb3VuZChwX2N0eFsiY29udHJhZGljdGlvbiJdLCA2KSwKICAgICAgICAgICAgICAgICJubGlfY3R4X25ldXRyYWxfYW5zIjogcm91bmQocF9jdHhbIm5ldXRyYWwiXSwgNiksCiAgICAgICAgICAgICAgICAibmxpX2Fuc19lbnRhaWxzX2N0eCI6IHJvdW5kKHBfYW5zWyJlbnRhaWxtZW50Il0sIDYpLAogICAgICAgICAgICAgICAgIm5saV9hbnNfY29udHJhZGljdHNfY3R4Ijogcm91bmQocF9hbnNbImNvbnRyYWRpY3Rpb24iXSwgNiksCiAgICAgICAgICAgICAgICAibmxpX2Fuc19uZXV0cmFsX2N0eCI6IHJvdW5kKHBfYW5zWyJuZXV0cmFsIl0sIDYpLAogICAgICAgICAgICB9KSkKICAgICAgICBiYXRjaF9jdHhfYW5zLCBiYXRjaF9hbnNfY3R4LCBiYXRjaF9pZHggPSBbXSwgW10sIFtdCgogICAgZm9yIGlkeCwgcm93IGluIGRmLml0ZXJyb3dzKCk6CiAgICAgICAgY29udGV4dCwgYW5zd2VyID0gc3RyKHJvd1siY29udGV4dCJdKSwgc3RyKHJvd1siYW5zd2VyIl0pCiAgICAgICAgaWYgbm90IGNvbnRleHQuc3RyaXAoKToKICAgICAgICAgICAgcm93cy5hcHBlbmQoKGlkeCwgX25ldXRyYWxfcm93KCkpKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGJhdGNoX2N0eF9hbnMuYXBwZW5kKChjb250ZXh0LCBhbnN3ZXIpKQogICAgICAgIGJhdGNoX2Fuc19jdHguYXBwZW5kKChhbnN3ZXIsIGNvbnRleHQpKQogICAgICAgIGJhdGNoX2lkeC5hcHBlbmQoaWR4KQogICAgICAgIGlmIGxlbihiYXRjaF9pZHgpID49IGJhdGNoX3NpemUgKiA0OgogICAgICAgICAgICBmbHVzaCgpCiAgICBmbHVzaCgpCgogICAgcm93cy5zb3J0KGtleT1sYW1iZGEgdDogdFswXSkKICAgIHJldHVybiBwZC5EYXRhRnJhbWUoW3IgZm9yIF8sIHIgaW4gcm93c10sIGluZGV4PVtpIGZvciBpLCBfIGluIHJvd3NdKQo=",
 "src/features/semantic_features.py": "IiIiClNlbWFudGljIHNpbWlsYXJpdHkgZmVhdHVyZSBleHRyYWN0aW9uIGZvciBIYWx1UklTQy4KCkdyb3VwIDcgZmVhdHVyZXMgKHJvYWRtYXAgwqc2KToKICBjb3NpbmVfY3R4X2FucywgY29zaW5lX3FfYW5zCgpVc2VzIHNlbnRlbmNlLXRyYW5zZm9ybWVycyBgYWxsLU1pbmlMTS1MNi12MmAgZW1iZWRkaW5ncy4KRW1wdHkgY29udGV4dCAtPiBjb3NpbmVfY3R4X2FucyA9IDAuMCAobm8gZXZpZGVuY2UgYXZhaWxhYmxlKS4KIiIiCgppbXBvcnQgbG9nZ2luZwpmcm9tIHR5cGluZyBpbXBvcnQgT3B0aW9uYWwKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykKCkVNQkVERElOR19NT0RFTCA9ICJzZW50ZW5jZS10cmFuc2Zvcm1lcnMvYWxsLU1pbmlMTS1MNi12MiIKCgpkZWYgX3NhZmVfZGV2aWNlKGRldmljZTogT3B0aW9uYWxbc3RyXSkgLT4gT3B0aW9uYWxbc3RyXToKICAgICIiIlJldHVybiB0aGUgZGV2aWNlIHRvIHVzZTsgY3VkYSBpcyBvbmx5IGhvbm9yZWQgd2hlbiB0b3JjaCBzdXBwb3J0cyBpdC4iIiIKICAgIGlmIGRldmljZSA9PSAiY3VkYSI6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBpbXBvcnQgdG9yY2gKCiAgICAgICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgICAgICByZXR1cm4gImN1ZGEiCiAgICAgICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgICAgICBwYXNzCiAgICAgICAgbG9nZ2VyLndhcm5pbmcoImRldmljZT1jdWRhIHJlcXVlc3RlZCBidXQgdG9yY2ggaGFzIG5vIENVREEgc3VwcG9ydDsgdXNpbmcgQ1BVIikKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIGRldmljZQoKCmRlZiBsb2FkX2VtYmVkZGluZ19tb2RlbChkZXZpY2U6IE9wdGlvbmFsW3N0cl0gPSBOb25lLCBtb2RlbF9rd2FyZ3M6IE9wdGlvbmFsW2RpY3RdID0gTm9uZSk6CiAgICAiIiJMb2FkIHRoZSBTQkVSVCBlbWJlZGRpbmcgbW9kZWwgKGxhenksIGNhY2hlZCBhdCBjYWxsIHNpdGUpLgoKICAgIGRldmljZT1Ob25lIC0+IGxpYnJhcnkgZGVmYXVsdCAoR1BVIGlmIGF2YWlsYWJsZSk7IHNldCAiY3B1IiBmb3Igc3RhYmlsaXR5LgogICAgbW9kZWxfa3dhcmdzIC0+IGV4dHJhIGt3YXJncyBmb3IgdGhlIG1vZGVsIGxvYWRlciAoZS5nLiB0b3JjaF9kdHlwZT1mbG9hdDE2KS4KICAgICIiIgogICAgZnJvbSBzZW50ZW5jZV90cmFuc2Zvcm1lcnMgaW1wb3J0IFNlbnRlbmNlVHJhbnNmb3JtZXIKCiAgICByZXR1cm4gU2VudGVuY2VUcmFuc2Zvcm1lcihFTUJFRERJTkdfTU9ERUwsIGRldmljZT1fc2FmZV9kZXZpY2UoZGV2aWNlKSwgbW9kZWxfa3dhcmdzPW1vZGVsX2t3YXJncykKCgpkZWYgX2Nvc2luZShhOiBucC5uZGFycmF5LCBiOiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgIG5hLCBuYiA9IG5wLmxpbmFsZy5ub3JtKGEpLCBucC5saW5hbGcubm9ybShiKQogICAgaWYgbmEgPT0gMC4wIG9yIG5iID09IDAuMDoKICAgICAgICByZXR1cm4gMC4wCiAgICByZXR1cm4gZmxvYXQobnAuZG90KGEsIGIpIC8gKG5hICogbmIpKQoKCmRlZiBleHRyYWN0X3NlbWFudGljX2ZlYXR1cmVzKHF1ZXN0aW9uOiBzdHIsIGNvbnRleHQ6IHN0ciwgYW5zd2VyOiBzdHIsIG1vZGVsKSAtPiBkaWN0OgogICAgIiIiR3JvdXAgNzogY29zaW5lIHNpbWlsYXJpdHkgYmV0d2VlbiBhbnN3ZXIgYW5kIGNvbnRleHQvcXVlc3Rpb24uIiIiCiAgICB0ZXh0cyA9IFthbnN3ZXJdCiAgICBpZiBjb250ZXh0LnN0cmlwKCk6CiAgICAgICAgdGV4dHMuYXBwZW5kKGNvbnRleHQpCiAgICB0ZXh0cy5hcHBlbmQocXVlc3Rpb24pCiAgICB2ZWNzID0gbW9kZWwuZW5jb2RlKHRleHRzLCBjb252ZXJ0X3RvX251bXB5PVRydWUpCgogICAgYW5zX3ZlYyA9IHZlY3NbMF0KICAgIG9mZnNldCA9IDEKICAgIGNvc2luZV9jdHhfYW5zID0gX2Nvc2luZShhbnNfdmVjLCB2ZWNzW29mZnNldF0pIGlmIGNvbnRleHQuc3RyaXAoKSBlbHNlIDAuMAogICAgaWYgY29udGV4dC5zdHJpcCgpOgogICAgICAgIG9mZnNldCArPSAxCiAgICBjb3NpbmVfcV9hbnMgPSBfY29zaW5lKGFuc192ZWMsIHZlY3Nbb2Zmc2V0XSkKCiAgICByZXR1cm4gewogICAgICAgICJjb3NpbmVfY3R4X2FucyI6IHJvdW5kKGNvc2luZV9jdHhfYW5zLCA2KSwKICAgICAgICAiY29zaW5lX3FfYW5zIjogcm91bmQoY29zaW5lX3FfYW5zLCA2KSwKICAgIH0KCgpkZWYgZXh0cmFjdF9zZW1hbnRpY19mZWF0dXJlc19kZihkZjogcGQuRGF0YUZyYW1lLCBtb2RlbCwgYmF0Y2hfc2l6ZTogaW50ID0gNjQpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIkJhdGNoIHNlbWFudGljIGZlYXR1cmVzOyBlbmNvZGVzIGVhY2ggY29sdW1uIG9uY2UgYW5kIHZlY3Rvcml6ZXMuIiIiCiAgICBsb2dnZXIuaW5mbyhmIkV4dHJhY3Rpbmcgc2VtYW50aWMgKFNCRVJUKSBmZWF0dXJlcyBmb3Ige2xlbihkZil9IHNhbXBsZXMuLi4iKQogICAgYW5zd2VycyA9IGRmWyJhbnN3ZXIiXS5hc3R5cGUoc3RyKS50b2xpc3QoKQogICAgY29udGV4dHMgPSBkZlsiY29udGV4dCJdLmFzdHlwZShzdHIpLnRvbGlzdCgpCiAgICBxdWVzdGlvbnMgPSBkZlsicXVlc3Rpb24iXS5hc3R5cGUoc3RyKS50b2xpc3QoKQoKICAgIGFuc192ZWNzID0gbW9kZWwuZW5jb2RlKGFuc3dlcnMsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwgY29udmVydF90b19udW1weT1UcnVlLCBzaG93X3Byb2dyZXNzX2Jhcj1UcnVlKQogICAgcV92ZWNzID0gbW9kZWwuZW5jb2RlKHF1ZXN0aW9ucywgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBjb252ZXJ0X3RvX251bXB5PVRydWUpCiAgICBjdHhfdmVjcyA9ICgKICAgICAgICBtb2RlbC5lbmNvZGUoW2MgZm9yIGMgaW4gY29udGV4dHMgaWYgYy5zdHJpcCgpXSwgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBjb252ZXJ0X3RvX251bXB5PVRydWUpCiAgICAgICAgaWYgYW55KGMuc3RyaXAoKSBmb3IgYyBpbiBjb250ZXh0cykKICAgICAgICBlbHNlIG5wLnplcm9zKCgwLCBhbnNfdmVjcy5zaGFwZVsxXSkpCiAgICApCgogICAgY3R4X2l0ZXIgPSBpdGVyKGN0eF92ZWNzKQogICAgcm93cyA9IFtdCiAgICBmb3IgaSwgYyBpbiBlbnVtZXJhdGUoY29udGV4dHMpOgogICAgICAgIGFuc192ZWMsIHFfdmVjID0gYW5zX3ZlY3NbaV0sIHFfdmVjc1tpXQogICAgICAgIGN0eF92ZWMgPSBuZXh0KGN0eF9pdGVyKSBpZiBjLnN0cmlwKCkgZWxzZSBOb25lCiAgICAgICAgcm93cy5hcHBlbmQoewogICAgICAgICAgICAiY29zaW5lX2N0eF9hbnMiOiByb3VuZChfY29zaW5lKGFuc192ZWMsIGN0eF92ZWMpLCA2KSBpZiBjdHhfdmVjIGlzIG5vdCBOb25lIGVsc2UgMC4wLAogICAgICAgICAgICAiY29zaW5lX3FfYW5zIjogcm91bmQoX2Nvc2luZShhbnNfdmVjLCBxX3ZlYyksIDYpLAogICAgICAgIH0pCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MsIGluZGV4PWRmLmluZGV4KQo=",
 "src/models/config.py": "IiIiCkhhbHVSSVNDIHNpbmdsZSBzb3VyY2Ugb2YgdHJ1dGggZm9yIGV4cGVyaW1lbnQgY29uZmlndXJhdGlvbgooYmx1ZXByaW50IEExODogIkFsbCByYW5kb20gc2VlZHMgZG9jdW1lbnRlZCBpbiBhIHNpbmdsZSBjb25maWcgZmlsZSIpLgoKSW1wb3J0ZWQgYnkgdHJhaW5fcGlwZWxpbmUsIHNoYXBfYW5hbHlzaXMsIGVycm9yX2FuYWx5c2lzLCBldmFsX2xsbV9qdWRnZSwKZXZhbF9lZmZpY2llbmN5IOKAlCBuZXZlciByZWRlZmluZSBzZWVkcyBlbHNld2hlcmUuCiIiIgoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KCiMgRXhwZXJpbWVudCBzZWVkcyAoYmx1ZXByaW50IEE5OiByZXBlYXQgZXZlcnkgZXhwZXJpbWVudCB3aXRoIDQyLCAxMjMsIDQ1NikKU0VFRFMgPSBbNDIsIDEyMywgNDU2XQoKIyBCb290c3RyYXAgLyBzYW1wbGluZyBzZWVkcyAoZml4ZWQsIHNlcGFyYXRlIGZyb20gZXhwZXJpbWVudCBzZWVkcykKQk9PVFNUUkFQX1NFRUQgPSA3NzcKU0FNUExFX1NFRUQgPSA0MgoKIyBDb3VudHMKTl9CT09UU1RSQVAgPSAxMDAwCgojIFBhdGhzCkFSVElGQUNUU19ESVIgPSBST09UIC8gImFydGlmYWN0cyIKTU9ERUxTX0RJUiA9IEFSVElGQUNUU19ESVIgLyAibW9kZWxzIgpSRVNVTFRTX0RJUiA9IEFSVElGQUNUU19ESVIgLyAicmVzdWx0cyIKRklHVVJFU19ESVIgPSBBUlRJRkFDVFNfRElSIC8gImZpZ3VyZXMiCkRBVEFfUFJPQ0VTU0VEID0gUk9PVCAvICJkYXRhIiAvICJwcm9jZXNzZWQiCgpGRUFUVVJFU19GVUxMID0gREFUQV9QUk9DRVNTRUQgLyAiZmVhdHVyZXNfZnVsbC5wYXJxdWV0IgpGRUFUVVJFU19GQUxMQkFDSyA9IERBVEFfUFJPQ0VTU0VEIC8gImZlYXR1cmVzX2NvcmUucGFycXVldCIKUUFfQ0xFQU4gPSBEQVRBX1BST0NFU1NFRCAvICJxYV9jbGVhbi5wYXJxdWV0Igo=",
 "src/models/error_analysis.py": "IiIiCkhhbHVSSVNDIGVycm9yIGFuYWx5c2lzIChibHVlcHJpbnQgQTEwOiBpbnNwZWN0IDIwIHdyb25nIHByZWRpY3Rpb25zLCAxMCBGUCArIDEwIEZOKS4KClN0ZXBzOgogIDEuIFByZWRpY3QgdGhlIHRlc3Qgc3BsaXQgd2l0aCB0aGUgY2FsaWJyYXRlZCBtb2RlbC4KICAyLiBTYW1wbGUgMTAgZmFsc2UgcG9zaXRpdmVzICsgMTAgZmFsc2UgbmVnYXRpdmVzIChzZWVkZWQpLgogIDMuIEF1dG8tdGFnIGVhY2ggY2FzZSB3aXRoIHRoZSBibHVlcHJpbnQgZXJyb3IgdGF4b25vbXkgKGhldXJpc3RpYyBydWxlcykuCiAgNC4gU2F2ZSBhIHJldmlld2FibGUgY2FzZSBkdW1wICsgYSBjYXRlZ29yeS1jb3VudCB0YWJsZS4KClRoZSB0YXhvbm9teSB0YWdzIGFyZSBhIHN0YXJ0aW5nIHBvaW50IGZvciBtYW51YWwgcmV2aWV3IOKAlCB2ZXJpZnkgYW5kIGFkanVzdAp0aGUgY2F0ZWdvcmllcyB3aGVuIHdyaXRpbmcgdGhlIHBhcGVyJ3MgZXJyb3IgYW5hbHlzaXMgc2VjdGlvbi4KClJ1biAocmVwbyByb290LCAudmVudik6CiAgcHl0aG9uIHNyYy9tb2RlbHMvZXJyb3JfYW5hbHlzaXMucHkKIiIiCgppbXBvcnQganNvbgppbXBvcnQgbG9nZ2luZwppbXBvcnQgb3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0pKQoKaW1wb3J0IGpvYmxpYgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8sIGZvcm1hdD0iJShhc2N0aW1lKXMgLSAlKGxldmVsbmFtZSlzIC0gJShtZXNzYWdlKXMiKQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigiZXJyb3JfYW5hbHlzaXMiKQoKZnJvbSBzcmMubW9kZWxzLmNvbmZpZyBpbXBvcnQgRkVBVFVSRVNfRkFMTEJBQ0ssIEZFQVRVUkVTX0ZVTEwsIE1PREVMU19ESVIsIFFBX0NMRUFOLCBSRVNVTFRTX0RJUiwgU0FNUExFX1NFRUQKCk5fRlAgPSAxMApOX0ZOID0gMTAKCgpkZWYgbG9hZF90ZXN0X3NldCgpOgogICAgcGF0aCA9IEZFQVRVUkVTX0ZVTEwgaWYgRkVBVFVSRVNfRlVMTC5leGlzdHMoKSBlbHNlIEZFQVRVUkVTX0ZBTExCQUNLCiAgICBkZiA9IHBkLnJlYWRfcGFycXVldChwYXRoKQogICAgY2xlYW4gPSBwZC5yZWFkX3BhcnF1ZXQoUUFfQ0xFQU4pCiAgICB0ZXh0X2NvbHMgPSBbYyBmb3IgYyBpbiBbInF1ZXN0aW9uIiwgImNvbnRleHQiLCAiYW5zd2VyIl0gaWYgYyBpbiBjbGVhbi5jb2x1bW5zXQogICAgaWYgdGV4dF9jb2xzOgogICAgICAgIGRmID0gcGQuY29uY2F0KFtkZiwgY2xlYW5bdGV4dF9jb2xzXV0sIGF4aXM9MSkKICAgIGZlYXR1cmVfY29scyA9IGpzb24ubG9hZHMoKE1PREVMU19ESVIgLyAiZmVhdHVyZV9uYW1lcy5qc29uIikucmVhZF90ZXh0KCkpCiAgICB0ZXN0X2RmID0gZGZbZGZbInNwbGl0Il0gPT0gInRlc3QiXS5jb3B5KCkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgWF90ZXN0ID0gdGVzdF9kZltmZWF0dXJlX2NvbHNdLnZhbHVlcwogICAgeV90ZXN0ID0gdGVzdF9kZlsibGFiZWwiXS52YWx1ZXMKICAgIHJldHVybiB0ZXN0X2RmLCBYX3Rlc3QsIHlfdGVzdCwgZmVhdHVyZV9jb2xzCgoKZGVmIGxvYWRfcHJlZGljdGlvbnMoWF90ZXN0LCBmZWF0dXJlX2NvbHMpOgogICAgYnVuZGxlID0gam9ibGliLmxvYWQoTU9ERUxTX0RJUiAvICJtb2RlbF94Z2Jvb3N0X2NhbGlicmF0ZWQuam9ibGliIikKICAgIGlmIGlzaW5zdGFuY2UoYnVuZGxlLCBkaWN0KSBhbmQgYnVuZGxlLmdldCgia2luZCIpID09ICJ4Z2IrcGxhdHQiOgogICAgICAgIHJhdywgcGxhdHQgPSBidW5kbGVbIm1vZGVsIl0sIGJ1bmRsZVsiY2FsaWJyYXRvciJdCgogICAgICAgIGRlZiBwcmVkaWN0X3Byb2JhKFgpOgogICAgICAgICAgICBwID0gcmF3LnByZWRpY3RfcHJvYmEoWClbOiwgMV0KICAgICAgICAgICAgcmV0dXJuIHBsYXR0LnByZWRpY3RfcHJvYmEocC5yZXNoYXBlKC0xLCAxKSlbOiwgMV0KCiAgICBlbHNlOgogICAgICAgIHByZWRpY3RfcHJvYmEgPSBidW5kbGUucHJlZGljdF9wcm9iYQogICAgeV9wcm9iID0gcHJlZGljdF9wcm9iYShYX3Rlc3QpCiAgICByZXR1cm4geV9wcm9iCgoKZGVmIHRhZ19jYXNlKHJvdzogcGQuU2VyaWVzKSAtPiBzdHI6CiAgICAiIiJIZXVyaXN0aWMgdGF4b25vbXkgdGFnZ2luZyAoYmx1ZXByaW50IEExMCBjYXRlZ29yaWVzKSAtIHJldmlldyBtYW51YWxseS4iIiIKICAgIG5fd29yZHMgPSBmbG9hdChyb3cuZ2V0KCJuX3dvcmRzIiwgMCkpCiAgICBvdmVybGFwID0gZmxvYXQocm93LmdldCgib3ZlcmxhcF9hbnN3ZXJfY29udGV4dCIsIDAuMCkpCiAgICBubGlfY29udHJhID0gZmxvYXQocm93LmdldCgibmxpX2N0eF9jb250cmFkaWN0c19hbnMiLCAwLjApKQogICAgbmxpX2VudGFpbCA9IGZsb2F0KHJvdy5nZXQoIm5saV9jdHhfZW50YWlsc19hbnMiLCAwLjApKQogICAgbl9lbnRzID0gZmxvYXQocm93LmdldCgibl9lbnRpdGllc19hbnN3ZXIiLCAwKSkKICAgIGN0eF9sZW4gPSBsZW4oc3RyKHJvdy5nZXQoImNvbnRleHQiLCAiIikpKQogICAgYW5zX2xlbiA9IGxlbihzdHIocm93LmdldCgiYW5zd2VyIiwgIiIpKSkKCiAgICBpZiBuX3dvcmRzIDw9IDQ6CiAgICAgICAgcmV0dXJuICJzaG9ydF9hbnN3ZXJfYW1iaWd1aXR5IgogICAgaWYgbmxpX2VudGFpbCA+IDAuOCBhbmQgb3ZlcmxhcCA+IDAuNjoKICAgICAgICByZXR1cm4gImxhYmVsX2FtYmlndWl0eSIKICAgIGlmIG5saV9jb250cmEgPiAwLjUgYW5kIG92ZXJsYXAgPCAwLjM6CiAgICAgICAgcmV0dXJuICJsYWJlbF9hbWJpZ3VpdHkiCiAgICBpZiBuX2VudHMgPT0gMDoKICAgICAgICByZXR1cm4gImVudGl0eV9leHRyYWN0aW9uX2ZhaWx1cmUiCiAgICBpZiBjdHhfbGVuIDwgODA6CiAgICAgICAgcmV0dXJuICJ3ZWFrX2NvbnRleHQiCiAgICBpZiBhbnNfbGVuIDwgNDAgYW5kIG5fd29yZHMgPD0gODoKICAgICAgICByZXR1cm4gInNob3J0X2Fuc3dlcl9hbWJpZ3VpdHkiCiAgICBpZiBvdmVybGFwID4gMC42OgogICAgICAgIHJldHVybiAidW5zdXBwb3J0ZWRfYnV0X3NlbWFudGljYWxseV9zaW1pbGFyIgogICAgcmV0dXJuICJvdGhlciIKCgpkZWYgbWFpbigpOgogICAgdGVzdF9kZiwgWF90ZXN0LCB5X3Rlc3QsIGZlYXR1cmVfY29scyA9IGxvYWRfdGVzdF9zZXQoKQogICAgeV9wcm9iID0gbG9hZF9wcmVkaWN0aW9ucyhYX3Rlc3QsIGZlYXR1cmVfY29scykKICAgIHlfcHJlZCA9ICh5X3Byb2IgPj0gMC41KS5hc3R5cGUoaW50KQoKICAgIGZwX2lkeCA9IG5wLndoZXJlKCh5X3ByZWQgPT0gMSkgJiAoeV90ZXN0ID09IDApKVswXQogICAgZm5faWR4ID0gbnAud2hlcmUoKHlfcHJlZCA9PSAwKSAmICh5X3Rlc3QgPT0gMSkpWzBdCiAgICBsb2dnZXIuaW5mbyhmIk1pc2NsYXNzaWZpZWQ6IEZQPXtsZW4oZnBfaWR4KX0sIEZOPXtsZW4oZm5faWR4KX0iKQoKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhTQU1QTEVfU0VFRCkKICAgIGZwX3NhbXBsZSA9IHJuZy5jaG9pY2UoZnBfaWR4LCBzaXplPW1pbihOX0ZQLCBsZW4oZnBfaWR4KSksIHJlcGxhY2U9RmFsc2UpCiAgICBmbl9zYW1wbGUgPSBybmcuY2hvaWNlKGZuX2lkeCwgc2l6ZT1taW4oTl9GTiwgbGVuKGZuX2lkeCkpLCByZXBsYWNlPUZhbHNlKQoKICAgIHJvd3MgPSBbXQogICAgZm9yIGlkeCBpbiBucC5jb25jYXRlbmF0ZShbZnBfc2FtcGxlLCBmbl9zYW1wbGVdKToKICAgICAgICByb3cgPSB0ZXN0X2RmLmlsb2NbaWR4XQogICAgICAgIGNhc2UgPSB7CiAgICAgICAgICAgICJzYW1wbGVfaWQiOiBzdHIocm93WyJzYW1wbGVfaWQiXSksCiAgICAgICAgICAgICJlcnJvcl90eXBlIjogImZhbHNlX3Bvc2l0aXZlIiBpZiB5X3Rlc3RbaWR4XSA9PSAwIGVsc2UgImZhbHNlX25lZ2F0aXZlIiwKICAgICAgICAgICAgInRydWVfbGFiZWwiOiBpbnQoeV90ZXN0W2lkeF0pLAogICAgICAgICAgICAicHJlZGljdGVkX2xhYmVsIjogaW50KHlfcHJlZFtpZHhdKSwKICAgICAgICAgICAgInByb2JhYmlsaXR5Ijogcm91bmQoZmxvYXQoeV9wcm9iW2lkeF0pLCA0KSwKICAgICAgICAgICAgImNhdGVnb3J5IjogdGFnX2Nhc2Uocm93KSwKICAgICAgICAgICAgInF1ZXN0aW9uIjogc3RyKHJvdy5nZXQoInF1ZXN0aW9uIiwgIiIpKVs6MzAwXSwKICAgICAgICAgICAgImNvbnRleHQiOiBzdHIocm93LmdldCgiY29udGV4dCIsICIiKSlbOjQwMF0sCiAgICAgICAgICAgICJhbnN3ZXIiOiBzdHIocm93LmdldCgiYW5zd2VyIiwgIiIpKVs6MzAwXSwKICAgICAgICAgICAgImZlYXR1cmVzIjogewogICAgICAgICAgICAgICAgIm5fd29yZHMiOiBmbG9hdChyb3cuZ2V0KCJuX3dvcmRzIiwgMCkpLAogICAgICAgICAgICAgICAgIm92ZXJsYXBfYW5zd2VyX2NvbnRleHQiOiByb3VuZChmbG9hdChyb3cuZ2V0KCJvdmVybGFwX2Fuc3dlcl9jb250ZXh0IiwgMC4wKSksIDQpLAogICAgICAgICAgICAgICAgIm5saV9jdHhfY29udHJhZGljdHNfYW5zIjogcm91bmQoZmxvYXQocm93LmdldCgibmxpX2N0eF9jb250cmFkaWN0c19hbnMiLCAwLjApKSwgNCksCiAgICAgICAgICAgICAgICAibmxpX2N0eF9lbnRhaWxzX2FucyI6IHJvdW5kKGZsb2F0KHJvdy5nZXQoIm5saV9jdHhfZW50YWlsc19hbnMiLCAwLjApKSwgNCksCiAgICAgICAgICAgICAgICAibl9lbnRpdGllc19hbnN3ZXIiOiBmbG9hdChyb3cuZ2V0KCJuX2VudGl0aWVzX2Fuc3dlciIsIDApKSwKICAgICAgICAgICAgICAgICJlbnRpdHlfb3ZlcmxhcF9yYXRpbyI6IHJvdW5kKGZsb2F0KHJvdy5nZXQoImVudGl0eV9vdmVybGFwX3JhdGlvIiwgMC4wKSksIDQpLAogICAgICAgICAgICAgICAgImNvc2luZV9jdHhfYW5zIjogcm91bmQoZmxvYXQocm93LmdldCgiY29zaW5lX2N0eF9hbnMiLCAwLjApKSwgNCksCiAgICAgICAgICAgIH0sCiAgICAgICAgfQogICAgICAgIHJvd3MuYXBwZW5kKGNhc2UpCgogICAgY2FzZXNfZGYgPSBwZC5EYXRhRnJhbWUocm93cykKICAgIGNvdW50cyA9IGNhc2VzX2RmLmdyb3VwYnkoWyJlcnJvcl90eXBlIiwgImNhdGVnb3J5Il0pLnNpemUoKS51bnN0YWNrKGZpbGxfdmFsdWU9MCkKCiAgICBzdW1tYXJ5ID0gewogICAgICAgICJuX3Rlc3QiOiBpbnQobGVuKHlfdGVzdCkpLAogICAgICAgICJuX2ZhbHNlX3Bvc2l0aXZlcyI6IGludChsZW4oZnBfaWR4KSksCiAgICAgICAgIm5fZmFsc2VfbmVnYXRpdmVzIjogaW50KGxlbihmbl9pZHgpKSwKICAgICAgICAic2FtcGxlZCI6IHsiZnAiOiBpbnQobGVuKGZwX3NhbXBsZSkpLCAiZm4iOiBpbnQobGVuKGZuX3NhbXBsZSkpfSwKICAgICAgICAiY2F0ZWdvcnlfY291bnRzIjogewogICAgICAgICAgICAiZmFsc2VfcG9zaXRpdmUiOiBjYXNlc19kZltjYXNlc19kZlsiZXJyb3JfdHlwZSJdID09ICJmYWxzZV9wb3NpdGl2ZSJdWyJjYXRlZ29yeSJdLnZhbHVlX2NvdW50cygpLnRvX2RpY3QoKSwKICAgICAgICAgICAgImZhbHNlX25lZ2F0aXZlIjogY2FzZXNfZGZbY2FzZXNfZGZbImVycm9yX3R5cGUiXSA9PSAiZmFsc2VfbmVnYXRpdmUiXVsiY2F0ZWdvcnkiXS52YWx1ZV9jb3VudHMoKS50b19kaWN0KCksCiAgICAgICAgfSwKICAgICAgICAibm90ZSI6ICJBdXRvLXRhZ2dlZCB3aXRoIGhldXJpc3RpYyBydWxlcyAtIHZlcmlmeSBjYXRlZ29yaWVzIG1hbnVhbGx5IGJlZm9yZSBwYXBlciB1c2UuIiwKICAgIH0KCiAgICB3aXRoIG9wZW4oUkVTVUxUU19ESVIgLyAiZXJyb3JfYW5hbHlzaXNfY2FzZXMuanNvbiIsICJ3IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAocm93cywgZiwgaW5kZW50PTIpCiAgICB3aXRoIG9wZW4oUkVTVUxUU19ESVIgLyAiZXJyb3JfYW5hbHlzaXMuanNvbiIsICJ3IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAoc3VtbWFyeSwgZiwgaW5kZW50PTIpCgogICAgcHJpbnQoIlxuIiArICI9IiAqIDgwKQogICAgcHJpbnQoIiBIYWx1UklTQyBFcnJvciBBbmFseXNpcyAoMTAgRlAgKyAxMCBGTiBvbiB0ZXN0IHNldCkiKQogICAgcHJpbnQoIj0iICogODApCiAgICBwcmludChjb3VudHMuZmlsbG5hKDApLmFzdHlwZShpbnQpLnRvX3N0cmluZygpKQogICAgcHJpbnQoIj0iICogODApCiAgICBsb2dnZXIuaW5mbyhmIlNhdmVkIGVycm9yX2FuYWx5c2lzLmpzb24gKyBlcnJvcl9hbmFseXNpc19jYXNlcy5qc29uIHRvIHtSRVNVTFRTX0RJUn0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK",
 "src/models/eval_efficiency.py": "IiIiCkhhbHVSSVNDIGVmZmljaWVuY3kgJiBjb3N0IGFuYWx5c2lzIChibHVlcHJpbnQgQTEwIGVmZmljaWVuY3kgYmxvY2spLgoKT24gYSBzYW1wbGUgb2YgdGhlIHRlc3Qgc2V0LCB0aW1lcyBlYWNoIGZlYXR1cmUtZXh0cmFjdGlvbiBncm91cCwgbW9kZWwKcHJlZGljdGlvbiwgYW5kIFNIQVAgZXhwbGFuYXRpb247IHJlcG9ydHMgcDUwL3A5NSwgbW9kZWwgYXJ0aWZhY3Qgc2l6ZSwgYW5kCmFuIGVzdGltYXRlZCBjb3N0IHBlciAxLDAwMCBwcmVkaWN0aW9ucyB2cyBhbiBMTE0ganVkZ2UuCgpSdW4gKHJlcG8gcm9vdCwgLnZlbnYpOgogIHB5dGhvbiBzcmMvbW9kZWxzL2V2YWxfZWZmaWNpZW5jeS5weQoiIiIKCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1syXSkpCgppbXBvcnQgam9ibGliCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcyIpCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJldmFsX2VmZmljaWVuY3kiKQoKZnJvbSBzcmMubW9kZWxzLmNvbmZpZyBpbXBvcnQgRkVBVFVSRVNfRkFMTEJBQ0ssIEZFQVRVUkVTX0ZVTEwsIE1PREVMU19ESVIsIFFBX0NMRUFOLCBSRVNVTFRTX0RJUiwgU0FNUExFX1NFRUQKCk5fU0FNUExFUyA9IDIwMAoKCmRlZiBwZXJjZW50aWxlKHZhbHMsIHApOgogICAgcmV0dXJuIGZsb2F0KG5wLnBlcmNlbnRpbGUodmFscywgcCkpCgoKZGVmIG1haW4oKToKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0pKQogICAgZnJvbSBzcmMuZmVhdHVyZXMuZXh0cmFjdF9mZWF0dXJlcyBpbXBvcnQgKAogICAgICAgIGV4dHJhY3RfaGVkZ2luZ19mZWF0dXJlcywKICAgICAgICBleHRyYWN0X2xlbmd0aF9mZWF0dXJlcywKICAgICAgICBleHRyYWN0X2xleGljYWxfZmVhdHVyZXMsCiAgICAgICAgZXh0cmFjdF9udW1lcmljX2ZlYXR1cmVzLAogICAgICAgIGxvYWRfaGVhdnlfbW9kZWxzLAogICAgKQogICAgZnJvbSBzcmMuZmVhdHVyZXMuZW50aXR5X2ZlYXR1cmVzIGltcG9ydCBleHRyYWN0X2VudGl0eV9mZWF0dXJlcwogICAgZnJvbSBzcmMuZmVhdHVyZXMubmxpX2ZlYXR1cmVzIGltcG9ydCBleHRyYWN0X25saV9mZWF0dXJlcwogICAgZnJvbSBzcmMuZmVhdHVyZXMuc2VtYW50aWNfZmVhdHVyZXMgaW1wb3J0IGV4dHJhY3Rfc2VtYW50aWNfZmVhdHVyZXMKCiAgICBwYXRoID0gRkVBVFVSRVNfRlVMTCBpZiBGRUFUVVJFU19GVUxMLmV4aXN0cygpIGVsc2UgRkVBVFVSRVNfRkFMTEJBQ0sKICAgIGRmID0gcGQucmVhZF9wYXJxdWV0KHBhdGgpCiAgICBjbGVhbiA9IHBkLnJlYWRfcGFycXVldChRQV9DTEVBTikKICAgIHRleHRfY29scyA9IFtjIGZvciBjIGluIFsicXVlc3Rpb24iLCAiY29udGV4dCIsICJhbnN3ZXIiXSBpZiBjIGluIGNsZWFuLmNvbHVtbnNdCiAgICBpZiB0ZXh0X2NvbHM6CiAgICAgICAgZGYgPSBwZC5jb25jYXQoW2RmLCBjbGVhblt0ZXh0X2NvbHNdXSwgYXhpcz0xKQogICAgZmVhdHVyZV9jb2xzID0ganNvbi5sb2FkcygoTU9ERUxTX0RJUiAvICJmZWF0dXJlX25hbWVzLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIHRlc3QgPSBkZltkZlsic3BsaXQiXSA9PSAidGVzdCJdLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoU0FNUExFX1NFRUQpCiAgICBpZHggPSBybmcuY2hvaWNlKGxlbih0ZXN0KSwgc2l6ZT1taW4oTl9TQU1QTEVTLCBsZW4odGVzdCkpLCByZXBsYWNlPUZhbHNlKQogICAgc2FtcGxlID0gdGVzdC5pbG9jW2lkeF0KCiAgICBidW5kbGUgPSBqb2JsaWIubG9hZChNT0RFTFNfRElSIC8gIm1vZGVsX3hnYm9vc3RfY2FsaWJyYXRlZC5qb2JsaWIiKQogICAgaWYgaXNpbnN0YW5jZShidW5kbGUsIGRpY3QpIGFuZCBidW5kbGUuZ2V0KCJraW5kIikgPT0gInhnYitwbGF0dCI6CiAgICAgICAgcmF3LCBwbGF0dCA9IGJ1bmRsZVsibW9kZWwiXSwgYnVuZGxlWyJjYWxpYnJhdG9yIl0KCiAgICAgICAgZGVmIHByZWRpY3RfcHJvYmEoWCk6CiAgICAgICAgICAgIHAgPSByYXcucHJlZGljdF9wcm9iYShYKVs6LCAxXQogICAgICAgICAgICByZXR1cm4gcGxhdHQucHJlZGljdF9wcm9iYShwLnJlc2hhcGUoLTEsIDEpKVs6LCAxXQoKICAgIGVsc2U6CiAgICAgICAgcmF3LCBwbGF0dCA9IE5vbmUsIE5vbmUKICAgICAgICBwcmVkaWN0X3Byb2JhID0gYnVuZGxlLnByZWRpY3RfcHJvYmEKCiAgICBpbXBvcnQgc2hhcAoKICAgIGV4cGxhaW5lciA9IGpvYmxpYi5sb2FkKE1PREVMU19ESVIgLyAic2hhcF9leHBsYWluZXIuam9ibGliIikKCiAgICBtb2RlbHMgPSBsb2FkX2hlYXZ5X21vZGVscygpCiAgICBubHAsIG5saSwgZW1iZWRkZXIgPSBtb2RlbHNbIm5scCJdLCBtb2RlbHNbIm5saSJdLCBtb2RlbHNbImVtYmVkZGVyIl0KCiAgICB0aW1pbmdzID0ge2c6IFtdIGZvciBnIGluIFsiY29yZV9sZXhpY2FsIiwgImVudGl0eSIsICJubGkiLCAic2VtYW50aWMiLCAibW9kZWwiLCAic2hhcCJdfQogICAgZm9yIF8sIHJvdyBpbiBzYW1wbGUuaXRlcnJvd3MoKToKICAgICAgICBxLCBjLCBhID0gc3RyKHJvd1sicXVlc3Rpb24iXSksIHN0cihyb3dbImNvbnRleHQiXSksIHN0cihyb3dbImFuc3dlciJdKQoKICAgICAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBleHRyYWN0X2xlbmd0aF9mZWF0dXJlcyhxLCBjLCBhKQogICAgICAgIGV4dHJhY3RfbGV4aWNhbF9mZWF0dXJlcyhxLCBjLCBhKQogICAgICAgIGV4dHJhY3RfbnVtZXJpY19mZWF0dXJlcyhxLCBjLCBhKQogICAgICAgIGV4dHJhY3RfaGVkZ2luZ19mZWF0dXJlcyhxLCBjLCBhKQogICAgICAgIHRpbWluZ3NbImNvcmVfbGV4aWNhbCJdLmFwcGVuZCgodGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwKSAqIDEwMDApCgogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGV4dHJhY3RfZW50aXR5X2ZlYXR1cmVzKHEsIGMsIGEsIG5scCkKICAgICAgICB0aW1pbmdzWyJlbnRpdHkiXS5hcHBlbmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCkgKiAxMDAwKQoKICAgICAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBleHRyYWN0X25saV9mZWF0dXJlcyhxLCBjLCBhLCBubGkpCiAgICAgICAgdGltaW5nc1sibmxpIl0uYXBwZW5kKCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDApICogMTAwMCkKCiAgICAgICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgZXh0cmFjdF9zZW1hbnRpY19mZWF0dXJlcyhxLCBjLCBhLCBlbWJlZGRlcikKICAgICAgICB0aW1pbmdzWyJzZW1hbnRpYyJdLmFwcGVuZCgodGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwKSAqIDEwMDApCgogICAgICAgIGZlYXRzID0gewogICAgICAgICAgICAibl9jaGFycyI6IDAsICJuX3dvcmRzIjogMCwgIm5fc2VudGVuY2VzIjogMSwgImF2Z193b3JkX2xlbiI6IDAsCiAgICAgICAgICAgICJvdmVybGFwX2Fuc3dlcl9jb250ZXh0IjogMC4wLCAib3ZlcmxhcF9hbnN3ZXJfcXVlc3Rpb24iOiAwLjAsCiAgICAgICAgICAgICJqYWNjYXJkX2Fuc19jdHgiOiAwLjAsICJqYWNjYXJkX2Fuc19xIjogMC4wLAogICAgICAgICAgICAibl9lbnRpdGllc19hbnN3ZXIiOiAwLCAibl9lbnRpdGllc19jb250ZXh0IjogMCwKICAgICAgICAgICAgImVudGl0eV9vdmVybGFwX3JhdGlvIjogMS4wLCAibm92ZWxfZW50aXR5X3JhdGlvIjogMC4wLAogICAgICAgICAgICAibmxpX2N0eF9lbnRhaWxzX2FucyI6IDEgLyAzLCAibmxpX2N0eF9jb250cmFkaWN0c19hbnMiOiAxIC8gMywgIm5saV9jdHhfbmV1dHJhbF9hbnMiOiAxIC8gMywKICAgICAgICAgICAgIm5saV9hbnNfZW50YWlsc19jdHgiOiAxIC8gMywgIm5saV9hbnNfY29udHJhZGljdHNfY3R4IjogMSAvIDMsICJubGlfYW5zX25ldXRyYWxfY3R4IjogMSAvIDMsCiAgICAgICAgICAgICJuX251bWJlcnNfYW5zd2VyIjogMCwgIm5fbnVtYmVyc19jb250ZXh0IjogMCwgIm51bWJlcl9vdmVybGFwX3JhdGlvIjogMS4wLCAibm92ZWxfbnVtYmVycyI6IDAsCiAgICAgICAgICAgICJoZWRnZV9jb3VudCI6IDAsICJoZWRnZV9kZW5zaXR5IjogMC4wLAogICAgICAgICAgICAiY29zaW5lX2N0eF9hbnMiOiAwLjAsICJjb3NpbmVfcV9hbnMiOiAwLjAsCiAgICAgICAgfQogICAgICAgIGZlYXRzLnVwZGF0ZSh7azogZmxvYXQocm93W2tdKSBmb3IgayBpbiBmZWF0dXJlX2NvbHMgaWYgayBpbiByb3d9KQoKICAgICAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBwcmVkaWN0X3Byb2JhKG5wLmFycmF5KFtbZmVhdHNbY10gZm9yIGMgaW4gZmVhdHVyZV9jb2xzXV0sIGR0eXBlPW5wLmZsb2F0NjQpKQogICAgICAgIHRpbWluZ3NbIm1vZGVsIl0uYXBwZW5kKCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDApICogMTAwMCkKCiAgICAgICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgZXhwbGFpbmVyLnNoYXBfdmFsdWVzKG5wLmFycmF5KFtbZmVhdHNbY10gZm9yIGMgaW4gZmVhdHVyZV9jb2xzXV0sIGR0eXBlPW5wLmZsb2F0NjQpKQogICAgICAgIHRpbWluZ3NbInNoYXAiXS5hcHBlbmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCkgKiAxMDAwKQoKICAgIHN1bW1hcnkgPSB7CiAgICAgICAgIm5fc2FtcGxlcyI6IGludChsZW4oc2FtcGxlKSksCiAgICAgICAgImxhdGVuY3lfbXMiOiB7CiAgICAgICAgICAgIGdyb3VwOiB7InA1MCI6IHJvdW5kKHBlcmNlbnRpbGUodiwgNTApLCAyKSwgInA5NSI6IHJvdW5kKHBlcmNlbnRpbGUodiwgOTUpLCAyKSwgIm1lYW4iOiByb3VuZChmbG9hdChucC5tZWFuKHYpKSwgMil9CiAgICAgICAgICAgIGZvciBncm91cCwgdiBpbiB0aW1pbmdzLml0ZW1zKCkKICAgICAgICB9LAogICAgICAgICJ0b3RhbF9wZXJfc2FtcGxlX21zIjogewogICAgICAgICAgICAicDUwIjogcm91bmQoZmxvYXQobnAubWVkaWFuKFtzdW0odGltaW5nc1tnXVtpXSBmb3IgZyBpbiB0aW1pbmdzKSBmb3IgaSBpbiByYW5nZShsZW4oc2FtcGxlKSldKSksIDIpCiAgICAgICAgfSwKICAgICAgICAibW9kZWxfYXJ0aWZhY3RfbWIiOiB7CiAgICAgICAgICAgICJtb2RlbF94Z2Jvb3N0X2NhbGlicmF0ZWQuam9ibGliIjogcm91bmQob3MucGF0aC5nZXRzaXplKE1PREVMU19ESVIgLyAibW9kZWxfeGdib29zdF9jYWxpYnJhdGVkLmpvYmxpYiIpIC8gMWU2LCAyKSwKICAgICAgICAgICAgIm1vZGVsX3hnYm9vc3RfcmF3LmpvYmxpYiI6IHJvdW5kKG9zLnBhdGguZ2V0c2l6ZShNT0RFTFNfRElSIC8gIm1vZGVsX3hnYm9vc3RfcmF3LmpvYmxpYiIpIC8gMWU2LCAyKSwKICAgICAgICB9LAogICAgICAgICJjb3N0X3Blcl8xMDAwX3ByZWRpY3Rpb25zX3VzZCI6IHsKICAgICAgICAgICAgImhhbHVyaXNjX2xvY2FsIjogMC4wMDEsICAjIGVsZWN0cmljaXR5IG9ubHk7IG5vIEFQSSBjb3N0CiAgICAgICAgICAgICJsbG1fanVkZ2VfZXN0aW1hdGUiOiAwLjExLCAgIyAxMDAwIHggfjExMDAgdG9rZW5zIGF0ICQwLjIwL00gaW4gKyAkMS4yMC9NIG91dAogICAgICAgIH0sCiAgICB9CgogICAgd2l0aCBvcGVuKFJFU1VMVFNfRElSIC8gImxhdGVuY3lfYW5hbHlzaXMuanNvbiIsICJ3IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAoc3VtbWFyeSwgZiwgaW5kZW50PTIpCgogICAgcHJpbnQoIlxuIiArICI9IiAqIDgwKQogICAgcHJpbnQoZiIgSGFsdVJJU0MgTGF0ZW5jeSBBbmFseXNpcyAobj17bGVuKHNhbXBsZSl9IHRlc3Qgc2FtcGxlcykiKQogICAgcHJpbnQoIj0iICogODApCiAgICBwcmludChmInsnY29tcG9uZW50Jzo8MTZ9eydwNTAgbXMnOj4xMH17J3A5NSBtcyc6PjEwfXsnbWVhbiBtcyc6PjEwfSIpCiAgICBmb3IgZ3JvdXAsIHYgaW4gdGltaW5ncy5pdGVtcygpOgogICAgICAgIHByaW50KGYie2dyb3VwOjwxNn17cGVyY2VudGlsZSh2LDUwKTo+MTAuMmZ9e3BlcmNlbnRpbGUodiw5NSk6PjEwLjJmfXtucC5tZWFuKHYpOj4xMC4yZn0iKQogICAgcHJpbnQoIj0iICogODApCiAgICBsb2dnZXIuaW5mbyhmIlNhdmVkIGxhdGVuY3lfYW5hbHlzaXMuanNvbiB0byB7UkVTVUxUU19ESVJ9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
 "src/models/eval_llm_judge.py": "IiIiCkhhbHVSSVNDIExMTS1hcy1qdWRnZSBjb21wYXJpc29uIChibHVlcHJpbnQgQTEwIGV4dGVybmFsIGJhc2VsaW5lICsgY29zdCB0YWJsZSkuCgpSdW5zIEdQVCA1LjYgTHVuYSBhcyBhIGhhbGx1Y2luYXRpb24ganVkZ2Ugb24gYSBiYWxhbmNlZCBzYW1wbGUgb2YgdGhlIHRlc3Qgc2V0CmFuZCBjb21wYXJlcyBhZ2FpbnN0IHRoZSBYR0Jvb3N0IG1vZGVsOiBhY2N1cmFjeS9wcmVjaXNpb24vcmVjYWxsL0YxLCBhZ3JlZW1lbnQsCmxhdGVuY3ksIGFuZCBhIHJlYWwgdG9rZW4tY29zdCBlc3RpbWF0ZS4KCkNvc3Q6IH4yMDAgc2FtcGxlcyB4IH4xLjFLIHRva2VucyDiiYggJDAuMDUtMC4xNSBkZXBlbmRpbmcgb24gbW9kZWwgcHJpY2luZy4KT3ZlcnJpZGUgc2FtcGxlIHNpemUgd2l0aCBIQUxVX0pVREdFX04uCgpSdW4gKHJlcG8gcm9vdCwgLnZlbnYpOgogIHB5dGhvbiBzcmMvbW9kZWxzL2V2YWxfbGxtX2p1ZGdlLnB5CiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdKSkKCmltcG9ydCBqb2JsaWIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBkb3RlbnYgaW1wb3J0IGxvYWRfZG90ZW52CmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBhY2N1cmFjeV9zY29yZSwgZjFfc2NvcmUsIHByZWNpc2lvbl9zY29yZSwgcmVjYWxsX3Njb3JlCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcyIpCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJldmFsX2xsbV9qdWRnZSIpCgpsb2FkX2RvdGVudigpICAjIE9QRU5BSV9BUElfS0VZLCBPUEVOQUlfTU9ERUwKCmZyb20gc3JjLm1vZGVscy5jb25maWcgaW1wb3J0IEZFQVRVUkVTX0ZBTExCQUNLLCBGRUFUVVJFU19GVUxMLCBNT0RFTFNfRElSLCBRQV9DTEVBTiwgUkVTVUxUU19ESVIsIFNBTVBMRV9TRUVECgpOX1NBTVBMRVMgPSBpbnQob3MuZW52aXJvbi5nZXQoIkhBTFVfSlVER0VfTiIsICIyMDAiKSkKClBSSUNJTkcgPSB7ImlucHV0X3Blcl9tdG9rIjogMC4yMCwgIm91dHB1dF9wZXJfbXRvayI6IDEuMjB9ICAjIEdQVCA1LjYgTHVuYSAocm9hZG1hcCBQaGFzZSA1KQoKCmRlZiBsb2FkX3Rlc3Rfc2V0KCk6CiAgICBwYXRoID0gRkVBVFVSRVNfRlVMTCBpZiBGRUFUVVJFU19GVUxMLmV4aXN0cygpIGVsc2UgRkVBVFVSRVNfRkFMTEJBQ0sKICAgIGRmID0gcGQucmVhZF9wYXJxdWV0KHBhdGgpCiAgICBjbGVhbiA9IHBkLnJlYWRfcGFycXVldChRQV9DTEVBTikKICAgIHRleHRfY29scyA9IFtjIGZvciBjIGluIFsicXVlc3Rpb24iLCAiY29udGV4dCIsICJhbnN3ZXIiXSBpZiBjIGluIGNsZWFuLmNvbHVtbnNdCiAgICBpZiB0ZXh0X2NvbHM6CiAgICAgICAgZGYgPSBwZC5jb25jYXQoW2RmLCBjbGVhblt0ZXh0X2NvbHNdXSwgYXhpcz0xKQogICAgZmVhdHVyZV9jb2xzID0ganNvbi5sb2FkcygoTU9ERUxTX0RJUiAvICJmZWF0dXJlX25hbWVzLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIHRlc3QgPSBkZltkZlsic3BsaXQiXSA9PSAidGVzdCJdLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIHJldHVybiB0ZXN0LCBmZWF0dXJlX2NvbHMKCgpkZWYgeGdiX3Byb2JhYmlsaXRpZXModGVzdCwgZmVhdHVyZV9jb2xzKToKICAgIGJ1bmRsZSA9IGpvYmxpYi5sb2FkKE1PREVMU19ESVIgLyAibW9kZWxfeGdib29zdF9jYWxpYnJhdGVkLmpvYmxpYiIpCiAgICBpZiBpc2luc3RhbmNlKGJ1bmRsZSwgZGljdCkgYW5kIGJ1bmRsZS5nZXQoImtpbmQiKSA9PSAieGdiK3BsYXR0IjoKICAgICAgICByYXcsIHBsYXR0ID0gYnVuZGxlWyJtb2RlbCJdLCBidW5kbGVbImNhbGlicmF0b3IiXQoKICAgICAgICBkZWYgcHJlZGljdF9wcm9iYShYKToKICAgICAgICAgICAgcCA9IHJhdy5wcmVkaWN0X3Byb2JhKFgpWzosIDFdCiAgICAgICAgICAgIHJldHVybiBwbGF0dC5wcmVkaWN0X3Byb2JhKHAucmVzaGFwZSgtMSwgMSkpWzosIDFdCgogICAgZWxzZToKICAgICAgICBwcmVkaWN0X3Byb2JhID0gYnVuZGxlLnByZWRpY3RfcHJvYmEKICAgIFggPSB0ZXN0W2ZlYXR1cmVfY29sc10udmFsdWVzCiAgICByZXR1cm4gcHJlZGljdF9wcm9iYShYKQoKCkpVREdFX1NZU1RFTSA9ICgKICAgICJZb3UgYXJlIGFuIGV4cGVydCBoYWxsdWNpbmF0aW9uLWp1ZGdlLiBHaXZlbiBhIHF1ZXN0aW9uLCBhIHJlZmVyZW5jZSBjb250ZXh0LCBhbmQgYW4gYW5zd2VyLCAiCiAgICAiZGVjaWRlIHdoZXRoZXIgdGhlIGFuc3dlciBjb250YWlucyBoYWxsdWNpbmF0ZWQgY29udGVudCAodW5zdXBwb3J0ZWQsIGNvbnRyYWRpY3RvcnksIG9yIGZhYnJpY2F0ZWQgIgogICAgImluZm9ybWF0aW9uIHJlbGF0aXZlIHRvIHRoZSBjb250ZXh0KS4gUmVzcG9uZCB3aXRoIEpTT04gb25seTogIgogICAgJ3sianVkZ21lbnQiOiAiaGFsbHVjaW5hdGVkInwiZ3JvdW5kZWQiLCAiY29uZmlkZW5jZSI6IDAuMC0xLjB9JwopCgoKZGVmIGp1ZGdlX29uZShjbGllbnQsIG1vZGVsX25hbWUsIHEsIGMsIGEpOgogICAgaW1wb3J0IGpzb24gYXMgX2pzb24KICAgIGltcG9ydCByZQoKICAgIHVzZXIgPSBmIlF1ZXN0aW9uOiB7cX1cbkNvbnRleHQ6IHtjIG9yICcobm9uZSknfVxuQW5zd2VyOiB7YX0iCiAgICByZXNwID0gY2xpZW50LmNoYXQuY29tcGxldGlvbnMuY3JlYXRlKAogICAgICAgIG1vZGVsPW1vZGVsX25hbWUsCiAgICAgICAgbWVzc2FnZXM9WwogICAgICAgICAgICB7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBKVURHRV9TWVNURU19LAogICAgICAgICAgICB7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogdXNlcn0sCiAgICAgICAgXSwKICAgICAgICBtYXhfY29tcGxldGlvbl90b2tlbnM9MTAwMCwKICAgICAgICByZXNwb25zZV9mb3JtYXQ9eyJ0eXBlIjogImpzb25fb2JqZWN0In0sCiAgICApCiAgICBjb250ZW50ID0gcmVzcC5jaG9pY2VzWzBdLm1lc3NhZ2UuY29udGVudC5zdHJpcCgpCiAgICBjb250ZW50ID0gcmUuc3ViKHIiXmBgYCg/Ompzb24pP3xgYGAkIiwgIiIsIGNvbnRlbnQsIGZsYWdzPXJlLk1VTFRJTElORSkuc3RyaXAoKQogICAgdHJ5OgogICAgICAgIGRhdGEgPSBfanNvbi5sb2Fkcyhjb250ZW50W2NvbnRlbnQuZmluZCgieyIpIDogY29udGVudC5yZmluZCgifSIpICsgMV0pCiAgICBleGNlcHQgX2pzb24uSlNPTkRlY29kZUVycm9yOgogICAgICAgIGp1ZGdtZW50ID0gImhhbGx1Y2luYXRlZCIgaWYgImhhbGx1Y2luYXRlZCIgaW4gY29udGVudC5sb3dlcigpIGVsc2UgImdyb3VuZGVkIgogICAgICAgIHJldHVybiBqdWRnbWVudCwgMC41LCByZXNwLnVzYWdlCiAgICBqdWRnbWVudCA9IGRhdGEuZ2V0KCJqdWRnbWVudCIsICJncm91bmRlZCIpCiAgICBjb25maWRlbmNlID0gZmxvYXQoZGF0YS5nZXQoImNvbmZpZGVuY2UiLCAwLjUpKQogICAgcmV0dXJuIGp1ZGdtZW50LCBjb25maWRlbmNlLCByZXNwLnVzYWdlCgoKZGVmIG1haW4oKToKICAgIGZyb20gb3BlbmFpIGltcG9ydCBPcGVuQUkKCiAgICBhcGlfa2V5ID0gb3MuZW52aXJvbi5nZXQoIk9QRU5BSV9BUElfS0VZIikKICAgIGlmIG5vdCBhcGlfa2V5OgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIk9QRU5BSV9BUElfS0VZIG5vdCBzZXQgaW4gLmVudiDigJQgY2Fubm90IHJ1biBMTE0ganVkZ2UuIikKCiAgICBtb2RlbF9uYW1lID0gb3MuZW52aXJvbi5nZXQoIk9QRU5BSV9NT0RFTCIsICJncHQtNS42LWx1bmEiKQogICAgY2xpZW50ID0gT3BlbkFJKGFwaV9rZXk9YXBpX2tleSkKCiAgICB0ZXN0LCBmZWF0dXJlX2NvbHMgPSBsb2FkX3Rlc3Rfc2V0KCkKICAgIHlfcHJvYl94Z2IgPSB4Z2JfcHJvYmFiaWxpdGllcyh0ZXN0LCBmZWF0dXJlX2NvbHMpCiAgICB5X3ByZWRfeGdiID0gKHlfcHJvYl94Z2IgPj0gMC41KS5hc3R5cGUoaW50KQogICAgeV90cnVlID0gdGVzdFsibGFiZWwiXS52YWx1ZXMKCiAgICAjIEJhbGFuY2VkIHNhbXBsZSAoaGFsZiBoYWxsdWNpbmF0ZWQsIGhhbGYgZ3JvdW5kZWQpCiAgICBwb3MgPSBucC53aGVyZSh5X3RydWUgPT0gMSlbMF0KICAgIG5lZyA9IG5wLndoZXJlKHlfdHJ1ZSA9PSAwKVswXQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKFNBTVBMRV9TRUVEKQogICAgbl9oYWxmID0gTl9TQU1QTEVTIC8vIDIKICAgIHNhbXBsZV9pZHggPSBucC5jb25jYXRlbmF0ZShbcm5nLmNob2ljZShwb3MsIG5faGFsZiwgcmVwbGFjZT1GYWxzZSksIHJuZy5jaG9pY2UobmVnLCBuX2hhbGYsIHJlcGxhY2U9RmFsc2UpXSkKCiAgICBqdWRnbWVudHMsIGNvbmZzLCBsYXRlbmNpZXMgPSBbXSwgW10sIFtdCiAgICBpbl90b2tlbnMgPSBvdXRfdG9rZW5zID0gMAogICAgZm9yIGksIGlkeCBpbiBlbnVtZXJhdGUoc2FtcGxlX2lkeCk6CiAgICAgICAgcm93ID0gdGVzdC5pbG9jW2lkeF0KICAgICAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBqdWRnbWVudCwgY29uZiwgdXNhZ2UgPSBqdWRnZV9vbmUoY2xpZW50LCBtb2RlbF9uYW1lLCByb3dbInF1ZXN0aW9uIl0sIHJvd1siY29udGV4dCJdLCByb3dbImFuc3dlciJdKQogICAgICAgIGxhdGVuY2llcy5hcHBlbmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCkgKiAxMDAwKQogICAgICAgIGp1ZGdtZW50cy5hcHBlbmQoMSBpZiBqdWRnbWVudCA9PSAiaGFsbHVjaW5hdGVkIiBlbHNlIDApCiAgICAgICAgY29uZnMuYXBwZW5kKGNvbmYpCiAgICAgICAgaW5fdG9rZW5zICs9IHVzYWdlLnByb21wdF90b2tlbnMKICAgICAgICBvdXRfdG9rZW5zICs9IHVzYWdlLmNvbXBsZXRpb25fdG9rZW5zCiAgICAgICAgaWYgKGkgKyAxKSAlIDUwID09IDA6CiAgICAgICAgICAgIGxvZ2dlci5pbmZvKGYianVkZ2VkIHtpICsgMX0ve2xlbihzYW1wbGVfaWR4KX0iKQoKICAgIHlfcHJlZF9qdWRnZSA9IG5wLmFycmF5KGp1ZGdtZW50cykKICAgIHlfdHJ1ZV9zdWIgPSB5X3RydWVbc2FtcGxlX2lkeF0KICAgIHlfeGdiX3N1YiA9IHlfcHJlZF94Z2Jbc2FtcGxlX2lkeF0KCiAgICAjIE1jTmVtYXI6IGp1ZGdlIHZzIFhHQm9vc3Qgb24gdGhlIHNhbWUgMjAwIHNhbXBsZXMgKG9mZi1kaWFnb25hbCA9IGRpc2NvcmRhbnQpCiAgICBqdWRnZV93cm9uZyA9IHlfcHJlZF9qdWRnZSAhPSB5X3RydWVfc3ViCiAgICB4Z2Jfd3JvbmcgPSB5X3hnYl9zdWIgIT0geV90cnVlX3N1YgogICAgYm90aF93cm9uZyA9IGludCgoanVkZ2Vfd3JvbmcgJiB4Z2Jfd3JvbmcpLnN1bSgpKQogICAganVkZ2Vfd3JvbmdfeGdiX3JpZ2h0ID0gaW50KChqdWRnZV93cm9uZyAmIH54Z2Jfd3JvbmcpLnN1bSgpKQogICAganVkZ2VfcmlnaHRfeGdiX3dyb25nID0gaW50KCh+anVkZ2Vfd3JvbmcgJiB4Z2Jfd3JvbmcpLnN1bSgpKQogICAgYm90aF9yaWdodCA9IGludCgofmp1ZGdlX3dyb25nICYgfnhnYl93cm9uZykuc3VtKCkpCiAgICBmcm9tIHN0YXRzbW9kZWxzLnN0YXRzLmNvbnRpbmdlbmN5X3RhYmxlcyBpbXBvcnQgbWNuZW1hcgoKICAgIG1jbiA9IG1jbmVtYXIoCiAgICAgICAgW1tib3RoX3dyb25nLCBqdWRnZV93cm9uZ194Z2JfcmlnaHRdLCBbanVkZ2VfcmlnaHRfeGdiX3dyb25nLCBib3RoX3JpZ2h0XV0sCiAgICAgICAgZXhhY3Q9RmFsc2UsCiAgICAgICAgY29ycmVjdGlvbj1UcnVlLAogICAgKQogICAgbWNuZW1hcl9wID0gZmxvYXQobWNuLnB2YWx1ZSkKCiAgICBjb3N0ID0gKGluX3Rva2VucyAvIDFlNikgKiBQUklDSU5HWyJpbnB1dF9wZXJfbXRvayJdICsgKG91dF90b2tlbnMgLyAxZTYpICogUFJJQ0lOR1sib3V0cHV0X3Blcl9tdG9rIl0KCiAgICByZXN1bHRzID0gewogICAgICAgICJuX3NhbXBsZXMiOiBpbnQobGVuKHNhbXBsZV9pZHgpKSwKICAgICAgICAibW9kZWwiOiBtb2RlbF9uYW1lLAogICAgICAgICJqdWRnZSI6IHsKICAgICAgICAgICAgImFjY3VyYWN5Ijogcm91bmQoZmxvYXQoYWNjdXJhY3lfc2NvcmUoeV90cnVlX3N1YiwgeV9wcmVkX2p1ZGdlKSksIDQpLAogICAgICAgICAgICAicHJlY2lzaW9uIjogcm91bmQoZmxvYXQocHJlY2lzaW9uX3Njb3JlKHlfdHJ1ZV9zdWIsIHlfcHJlZF9qdWRnZSwgemVyb19kaXZpc2lvbj0wKSksIDQpLAogICAgICAgICAgICAicmVjYWxsIjogcm91bmQoZmxvYXQocmVjYWxsX3Njb3JlKHlfdHJ1ZV9zdWIsIHlfcHJlZF9qdWRnZSwgemVyb19kaXZpc2lvbj0wKSksIDQpLAogICAgICAgICAgICAiZjEiOiByb3VuZChmbG9hdChmMV9zY29yZSh5X3RydWVfc3ViLCB5X3ByZWRfanVkZ2UsIHplcm9fZGl2aXNpb249MCkpLCA0KSwKICAgICAgICAgICAgImxhdGVuY3lfbXNfcDUwIjogcm91bmQoZmxvYXQobnAubWVkaWFuKGxhdGVuY2llcykpLCAxKSwKICAgICAgICAgICAgImxhdGVuY3lfbXNfcDk1Ijogcm91bmQoZmxvYXQobnAucGVyY2VudGlsZShsYXRlbmNpZXMsIDk1KSksIDEpLAogICAgICAgIH0sCiAgICAgICAgInhnYm9vc3Rfb25fc2FtZV9zdWJzZXQiOiB7CiAgICAgICAgICAgICJhY2N1cmFjeSI6IHJvdW5kKGZsb2F0KGFjY3VyYWN5X3Njb3JlKHlfdHJ1ZV9zdWIsIHlfeGdiX3N1YikpLCA0KSwKICAgICAgICAgICAgInByZWNpc2lvbiI6IHJvdW5kKGZsb2F0KHByZWNpc2lvbl9zY29yZSh5X3RydWVfc3ViLCB5X3hnYl9zdWIsIHplcm9fZGl2aXNpb249MCkpLCA0KSwKICAgICAgICAgICAgInJlY2FsbCI6IHJvdW5kKGZsb2F0KHJlY2FsbF9zY29yZSh5X3RydWVfc3ViLCB5X3hnYl9zdWIsIHplcm9fZGl2aXNpb249MCkpLCA0KSwKICAgICAgICAgICAgImYxIjogcm91bmQoZmxvYXQoZjFfc2NvcmUoeV90cnVlX3N1YiwgeV94Z2Jfc3ViLCB6ZXJvX2RpdmlzaW9uPTApKSwgNCksCiAgICAgICAgfSwKICAgICAgICAiYWdyZWVtZW50X3dpdGhfeGdib29zdCI6IHJvdW5kKGZsb2F0KCh5X3ByZWRfanVkZ2UgPT0geV94Z2Jfc3ViKS5tZWFuKCkpLCA0KSwKICAgICAgICAibWNuZW1hcl9qdWRnZV92c194Z2Jvb3N0X3AiOiBtY25lbWFyX3AsCiAgICAgICAgImRpc2NvcmRhbnRfcGFpcnMiOiB7Imp1ZGdlX3dyb25nX3hnYl9yaWdodCI6IGp1ZGdlX3dyb25nX3hnYl9yaWdodCwgImp1ZGdlX3JpZ2h0X3hnYl93cm9uZyI6IGp1ZGdlX3JpZ2h0X3hnYl93cm9uZ30sCiAgICAgICAgImNvc3RfdXNkIjogcm91bmQoY29zdCwgNCksCiAgICAgICAgImNvc3RfcGVyXzEwMDBfdXNkIjogcm91bmQoY29zdCAvIGxlbihzYW1wbGVfaWR4KSAqIDEwMDAsIDMpLAogICAgICAgICJ0b2tlbnMiOiB7ImlucHV0IjogaW50KGluX3Rva2VucyksICJvdXRwdXQiOiBpbnQob3V0X3Rva2Vucyl9LAogICAgICAgICJwcmljaW5nIjogUFJJQ0lORywKICAgIH0KCiAgICB3aXRoIG9wZW4oUkVTVUxUU19ESVIgLyAibGxtX2p1ZGdlX3Jlc3VsdHMuanNvbiIsICJ3IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAocmVzdWx0cywgZiwgaW5kZW50PTIpCgogICAgcHJpbnQoIlxuIiArICI9IiAqIDgwKQogICAgcHJpbnQoZiIgTExNLWFzLUp1ZGdlIChHUFQgNS42IEx1bmEpIHZzIFhHQm9vc3Qgb24ge2xlbihzYW1wbGVfaWR4KX0gdGVzdCBzYW1wbGVzIikKICAgIHByaW50KCI9IiAqIDgwKQogICAgZm9yIG5hbWUsIG0gaW4gWygiSnVkZ2UiLCByZXN1bHRzWyJqdWRnZSJdKSwgKCJYR0Jvb3N0IiwgcmVzdWx0c1sieGdib29zdF9vbl9zYW1lX3N1YnNldCJdKV06CiAgICAgICAgcHJpbnQoZiJ7bmFtZTo8MTB9IGFjYz17bVsnYWNjdXJhY3knXTouNGZ9IFA9e21bJ3ByZWNpc2lvbiddOi40Zn0gUj17bVsncmVjYWxsJ106LjRmfSBGMT17bVsnZjEnXTouNGZ9IikKICAgIHByaW50KGYiQWdyZWVtZW50OiB7cmVzdWx0c1snYWdyZWVtZW50X3dpdGhfeGdib29zdCddOi40Zn0gfCBNY05lbWFyIHA9e21jbmVtYXJfcDouMmV9IHwgIgogICAgICAgICAgZiJKdWRnZSBjb3N0OiAke3Jlc3VsdHNbJ2Nvc3RfdXNkJ106LjRmfSAoe3Jlc3VsdHNbJ2Nvc3RfcGVyXzEwMDBfdXNkJ106LjNmfS8xSykgfCBwNTAge3Jlc3VsdHNbJ2p1ZGdlJ11bJ2xhdGVuY3lfbXNfcDUwJ119bXMiKQogICAgcHJpbnQoIj0iICogODApCiAgICBsb2dnZXIuaW5mbyhmIlNhdmVkIGxsbV9qdWRnZV9yZXN1bHRzLmpzb24gdG8ge1JFU1VMVFNfRElSfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
 "src/models/eval_ragtruth.py": "IiIiClplcm8tc2hvdCBleHRlcm5hbCB2YWxpZGF0aW9uIG9mIHRoZSBIYWx1UklTQyBtb2RlbCBvbiBSQUdUcnV0aCBRQSAoYmx1ZXByaW50IEExMCwKcm9hZG1hcCBQaGFzZSA1ICJFeHRlcm5hbCBjb21wYXJpc29uIikuCgpSdW5zIHRoZSBmaW5hbCBjYWxpYnJhdGVkIFhHQm9vc3QgbW9kZWwgd2l0aCBOTyB0cmFpbmluZyBvbiBSQUdUcnV0aCBkYXRhOgogIGZlYXR1cmVzIGFyZSBleHRyYWN0ZWQgd2l0aCB0aGUgc2FtZSBwaXBlbGluZSwgdGhlbiBwcmVkaWN0LgoKUnVuIChyZXBvIHJvb3QsIC52ZW52KToKICBweXRob24gc3JjL21vZGVscy9ldmFsX3JhZ3RydXRoLnB5CiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgam9ibGliCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBhdmVyYWdlX3ByZWNpc2lvbl9zY29yZSwgYnJpZXJfc2NvcmVfbG9zcywgZjFfc2NvcmUsIG1hdHRoZXdzX2NvcnJjb2VmLCBwcmVjaXNpb25fc2NvcmUsIHJlY2FsbF9zY29yZSwgcm9jX2F1Y19zY29yZQoKbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8sIGZvcm1hdD0iJShhc2N0aW1lKXMgLSAlKGxldmVsbmFtZSlzIC0gJShtZXNzYWdlKXMiKQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigiZXZhbF9yYWd0cnV0aCIpCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KUkFHVFJVVEhfUEFUSCA9IFJPT1QgLyAiZGF0YSIgLyAicmF3IiAvICJyYWd0cnV0aCIgLyAicmFndHJ1dGhfcWEucGFycXVldCIKVU5JRklFRF9QQVRIID0gUk9PVCAvICJkYXRhIiAvICJwcm9jZXNzZWQiIC8gInVuaWZpZWRfcmVjb3Jkcy5wYXJxdWV0IgpNT0RFTFNfRElSID0gUk9PVCAvICJhcnRpZmFjdHMiIC8gIm1vZGVscyIKUkVTVUxUU19ESVIgPSBST09UIC8gImFydGlmYWN0cyIgLyAicmVzdWx0cyIKCk5fU0FNUExFUyA9IDIwMDAKCgpkZWYgbG9hZF9yYWd0cnV0aF9mcmFtZShuOiBpbnQgPSBOX1NBTVBMRVMpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIlJBR1RydXRoIFFBIHJvd3MgZm9yIHplcm8tc2hvdCB2YWxpZGF0aW9uLgoKICAgIFByaW9yaXR5OiAoMSkgbGVnYWN5IFZlcnNpb24gQSBwYXJxdWV0IGlmIHByZXNlbnQsICgyKSB0aGUgQjEgdW5pZmllZAogICAgZGF0YXNldCBidWlsdCBieSBjZWxsIDdkIChkYXRhL3Byb2Nlc3NlZC91bmlmaWVkX3JlY29yZHMucGFycXVldCksIHdoaWNoCiAgICBpcyB3cml0dGVuIHRvIHRoZSBsZWdhY3kgcGF0aCBvbiBmaXJzdCB1c2Ugc28gcmVwZWF0IHJ1bnMgc2tpcCB0aGUgbWVyZ2UuCiAgICAiIiIKICAgIGlmIFJBR1RSVVRIX1BBVEguZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIHBkLnJlYWRfcGFycXVldChSQUdUUlVUSF9QQVRIKS5oZWFkKG4pCiAgICBpZiBub3QgVU5JRklFRF9QQVRILmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICBmIlJBR1RydXRoIGRhdGEgbWlzc2luZzogcnVuIGNlbGwgN2QgKEIxIHVuaWZpZWQgYnVpbGQpIG9yIHBsYWNlICIKICAgICAgICAgICAgZiJ7UkFHVFJVVEhfUEFUSH0iCiAgICAgICAgKQogICAgdW5pZmllZCA9IHBkLnJlYWRfcGFycXVldCgKICAgICAgICBVTklGSUVEX1BBVEgsIGNvbHVtbnM9WyJzb3VyY2VfZGF0YXNldCIsICJ0YXNrIiwgInF1ZXN0aW9uIiwgImNvbnRleHQiLCAiYW5zd2VyIiwgImxhYmVsIl0KICAgICkKICAgIHFhID0gdW5pZmllZFsodW5pZmllZFsic291cmNlX2RhdGFzZXQiXSA9PSAicmFndHJ1dGgiKSAmICh1bmlmaWVkWyJ0YXNrIl0gPT0gInFhIildCiAgICBxYSA9IHFhW1sicXVlc3Rpb24iLCAiY29udGV4dCIsICJhbnN3ZXIiLCAibGFiZWwiXV0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgUkFHVFJVVEhfUEFUSC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcWEudG9fcGFycXVldChSQUdUUlVUSF9QQVRILCBpbmRleD1GYWxzZSkKICAgIGxvZ2dlci5pbmZvKGYiQnVpbHQge1JBR1RSVVRIX1BBVEh9IGZyb20gdGhlIHVuaWZpZWQgZGF0YXNldCAoe2xlbihxYSl9IFFBIHJvd3MpIikKICAgIHJldHVybiBxYS5oZWFkKG4pCgoKZGVmIGVjZSh5X3RydWUsIHlfcHJvYiwgbl9iaW5zOiBpbnQgPSAxMCkgLT4gZmxvYXQ6CiAgICBmcm9tIHNyYy5tb2RlbHMudHJhaW5fcGlwZWxpbmUgaW1wb3J0IGVjZSBhcyBlY2VfZm4KCiAgICByZXR1cm4gZWNlX2ZuKHlfdHJ1ZSwgeV9wcm9iLCBuX2JpbnMpCgoKZGVmIG1haW4oKToKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoUk9PVCkpCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5leHRyYWN0X2ZlYXR1cmVzIGltcG9ydCBleHRyYWN0X2FsbF9mZWF0dXJlc19zaW5nbGUsIGxvYWRfaGVhdnlfbW9kZWxzCgogICAgZGYgPSBsb2FkX3JhZ3RydXRoX2ZyYW1lKE5fU0FNUExFUykKICAgIGxvZ2dlci5pbmZvKGYiUkFHVHJ1dGggUUEgaG9sZG91dDoge2xlbihkZil9IHNhbXBsZXMgKGxhYmVsIGJhbGFuY2U6IHtkZlsnbGFiZWwnXS52YWx1ZV9jb3VudHMoKS50b19kaWN0KCl9KSIpCgogICAgYnVuZGxlID0gam9ibGliLmxvYWQoTU9ERUxTX0RJUiAvICJtb2RlbF94Z2Jvb3N0X2NhbGlicmF0ZWQuam9ibGliIikKICAgIGlmIGlzaW5zdGFuY2UoYnVuZGxlLCBkaWN0KSBhbmQgYnVuZGxlLmdldCgia2luZCIpID09ICJ4Z2IrcGxhdHQiOgogICAgICAgIHJhdywgcGxhdHQgPSBidW5kbGVbIm1vZGVsIl0sIGJ1bmRsZVsiY2FsaWJyYXRvciJdCgogICAgICAgIGRlZiBwcmVkaWN0X3Byb2JhKFgpOgogICAgICAgICAgICBwID0gcmF3LnByZWRpY3RfcHJvYmEoWClbOiwgMV0KICAgICAgICAgICAgcmV0dXJuIHBsYXR0LnByZWRpY3RfcHJvYmEocC5yZXNoYXBlKC0xLCAxKSkKCiAgICBlbHNlOgogICAgICAgIHByZWRpY3RfcHJvYmEgPSBidW5kbGUucHJlZGljdF9wcm9iYQogICAgZmVhdHVyZV9jb2xzID0ganNvbi5sb2FkcygoTU9ERUxTX0RJUiAvICJmZWF0dXJlX25hbWVzLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIG1vZGVscyA9IGxvYWRfaGVhdnlfbW9kZWxzKCkKCiAgICBsb2dnZXIuaW5mbygiRXh0cmFjdGluZyBmZWF0dXJlcyBvbiBSQUdUcnV0aCAoemVyby1zaG90KS4uLiIpCiAgICByb3dzID0gW10KICAgIGZvciBfLCByIGluIGRmLml0ZXJyb3dzKCk6CiAgICAgICAgcm93cy5hcHBlbmQoZXh0cmFjdF9hbGxfZmVhdHVyZXNfc2luZ2xlKHJbInF1ZXN0aW9uIl0sIHJbImNvbnRleHQiXSwgclsiYW5zd2VyIl0sIG1vZGVscykpCiAgICBYID0gcGQuRGF0YUZyYW1lKHJvd3MpW2ZlYXR1cmVfY29sc10udmFsdWVzCiAgICB5ID0gZGZbImxhYmVsIl0udmFsdWVzCgogICAgeV9wcm9iID0gcHJlZGljdF9wcm9iYShYKVs6LCAxXQogICAgeV9wcmVkID0gKHlfcHJvYiA+PSAwLjUpLmFzdHlwZShpbnQpCgogICAgcmVzdWx0cyA9IHsKICAgICAgICAibl9zYW1wbGVzIjogaW50KGxlbih5KSksCiAgICAgICAgInByZWNpc2lvbiI6IGZsb2F0KHByZWNpc2lvbl9zY29yZSh5LCB5X3ByZWQsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJyZWNhbGwiOiBmbG9hdChyZWNhbGxfc2NvcmUoeSwgeV9wcmVkLCB6ZXJvX2RpdmlzaW9uPTApKSwKICAgICAgICAiZjEiOiBmbG9hdChmMV9zY29yZSh5LCB5X3ByZWQsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJhdXJvYyI6IGZsb2F0KHJvY19hdWNfc2NvcmUoeSwgeV9wcm9iKSksCiAgICAgICAgInByX2F1YyI6IGZsb2F0KGF2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlKHksIHlfcHJvYikpLAogICAgICAgICJtY2MiOiBmbG9hdChtYXR0aGV3c19jb3JyY29lZih5LCB5X3ByZWQpKSwKICAgICAgICAiZWNlIjogZmxvYXQoZWNlKHksIHlfcHJvYikpLAogICAgICAgICJicmllciI6IGZsb2F0KGJyaWVyX3Njb3JlX2xvc3MoeSwgeV9wcm9iKSksCiAgICAgICAgImxhYmVsX2Rpc3RyaWJ1dGlvbiI6IGRmWyJsYWJlbCJdLnZhbHVlX2NvdW50cygpLnRvX2RpY3QoKSwKICAgIH0KICAgIGxvZ2dlci5pbmZvKGYiUkFHVHJ1dGggemVyby1zaG90OiB7anNvbi5kdW1wcyh7azogKHJvdW5kKHYsIDQpIGlmIGlzaW5zdGFuY2UodiwgZmxvYXQpIGVsc2UgdikgZm9yIGssIHYgaW4gcmVzdWx0cy5pdGVtcygpfSwgaW5kZW50PTIpfSIpCgogICAgb3MubWFrZWRpcnMoUkVTVUxUU19ESVIsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIG9wZW4oUkVTVUxUU19ESVIgLyAicmFndHJ1dGhfcmVzdWx0cy5qc29uIiwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChyZXN1bHRzLCBmLCBpbmRlbnQ9MikKICAgIGxvZ2dlci5pbmZvKGYiU2F2ZWQge1JFU1VMVFNfRElSIC8gJ3JhZ3RydXRoX3Jlc3VsdHMuanNvbid9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
 "src/models/make_manifest.py": "IiIiDQpIYWx1UklTQyBhcnRpZmFjdCBtYW5pZmVzdCBnZW5lcmF0b3IgKGJsdWVwcmludCBBMTggLyByb2FkbWFwIEI2KS4NCg0KV3JpdGVzIGFydGlmYWN0cy9yZXN1bHRzL21hbmlmZXN0Lmpzb24gd2l0aCBkYXRhc2V0IGhhc2hlcywgc3BsaXQgcmVwb3J0LA0KcGFja2FnZSB2ZXJzaW9ucywgaGFyZHdhcmUvc29mdHdhcmUgaW5mbywgbW9kZWwvZmVhdHVyZSB2ZXJzaW9ucywgYW5kIHRoZQ0KbGlzdCBvZiBwcm9kdWNlZCBhcnRpZmFjdHMuIENvbGFiLXNhZmU6IHJlcG8tcm9vdC1yZWxhdGl2ZSBwYXRocyBvbmx5Lg0KDQpSdW4gKHJlcG8gcm9vdCwgLnZlbnYgb3IgQ29sYWIpOg0KICBweXRob24gc3JjL21vZGVscy9tYWtlX21hbmlmZXN0LnB5DQoiIiINCg0KaW1wb3J0IGhhc2hsaWINCmltcG9ydCBqc29uDQppbXBvcnQgbG9nZ2luZw0KaW1wb3J0IG9zDQppbXBvcnQgcGxhdGZvcm0NCmltcG9ydCBzdWJwcm9jZXNzDQppbXBvcnQgc3lzDQppbXBvcnQgdGltZQ0KZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0Kc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1syXSkpDQoNCmZyb20gc3JjLm1vZGVscy5jb25maWcgaW1wb3J0IEFSVElGQUNUU19ESVIsIERBVEFfUFJPQ0VTU0VELCBNT0RFTFNfRElSLCBSRVNVTFRTX0RJUiAgIyBub3FhOiBFNDAyDQoNCmxvZ2dpbmcuYmFzaWNDb25maWcobGV2ZWw9bG9nZ2luZy5JTkZPLCBmb3JtYXQ9IiUoYXNjdGltZSlzIC0gJShsZXZlbG5hbWUpcyAtICUobWVzc2FnZSlzIikNCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJtYWtlX21hbmlmZXN0IikNCg0KU0hBX0ZJTEVTID0gWw0KICAgIERBVEFfUFJPQ0VTU0VEIC8gInFhX2NsZWFuLnBhcnF1ZXQiLA0KICAgIERBVEFfUFJPQ0VTU0VEIC8gImZlYXR1cmVzX2Z1bGwucGFycXVldCIsDQogICAgTU9ERUxTX0RJUiAvICJtb2RlbF94Z2Jvb3N0X3Jhdy5qb2JsaWIiLA0KICAgIE1PREVMU19ESVIgLyAibW9kZWxfeGdib29zdF9jYWxpYnJhdGVkLmpvYmxpYiIsDQogICAgQVJUSUZBQ1RTX0RJUiAvICJzcGxpdF9pbmRpY2VzLmpzb24iLA0KICAgIEFSVElGQUNUU19ESVIgLyAic3BsaXRfaW50ZWdyaXR5X3JlcG9ydC5qc29uIiwNCl0NCg0KDQpkZWYgc2hhMjU2KHBhdGg6IFBhdGgpIC0+IHN0ciB8IE5vbmU6DQogICAgaWYgbm90IHBhdGguZXhpc3RzKCk6DQogICAgICAgIHJldHVybiBOb25lDQogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkNCiAgICB3aXRoIG9wZW4ocGF0aCwgInJiIikgYXMgZjoNCiAgICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBmLnJlYWQoMSA8PCAyMCksIGIiIik6DQogICAgICAgICAgICBoLnVwZGF0ZShjaHVuaykNCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQ0KDQoNCmRlZiBnaXRfY29tbWl0KCkgLT4gc3RyIHwgTm9uZToNCiAgICB0cnk6DQogICAgICAgIG91dCA9IHN1YnByb2Nlc3MucnVuKA0KICAgICAgICAgICAgWyJnaXQiLCAicmV2LXBhcnNlIiwgIkhFQUQiXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1lb3V0PTEwDQogICAgICAgICkNCiAgICAgICAgcmV0dXJuIG91dC5zdGRvdXQuc3RyaXAoKSBvciBOb25lDQogICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgcmV0dXJuIE5vbmUNCg0KDQpkZWYgdmVyc2lvbnMoKSAtPiBkaWN0Og0KICAgIGRlZiB2ZXIobmFtZTogc3RyKToNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgbW9kID0gX19pbXBvcnRfXyhuYW1lKQ0KICAgICAgICAgICAgcmV0dXJuIGdldGF0dHIobW9kLCAiX192ZXJzaW9uX18iLCAidW5rbm93biIpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICByZXR1cm4gTm9uZQ0KDQogICAgcmV0dXJuIHsNCiAgICAgICAgInB5dGhvbiI6IHBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCksDQogICAgICAgICJwbGF0Zm9ybSI6IHBsYXRmb3JtLnBsYXRmb3JtKCksDQogICAgICAgICJ0b3JjaCI6IHZlcigidG9yY2giKSwNCiAgICAgICAgIm51bXB5IjogdmVyKCJudW1weSIpLA0KICAgICAgICAicGFuZGFzIjogdmVyKCJwYW5kYXMiKSwNCiAgICAgICAgInNjaWtpdF9sZWFybiI6IHZlcigic2tsZWFybiIpLA0KICAgICAgICAieGdib29zdCI6IHZlcigieGdib29zdCIpLA0KICAgICAgICAic2hhcCI6IHZlcigic2hhcCIpLA0KICAgICAgICAic3BhY3kiOiB2ZXIoInNwYWN5IiksDQogICAgICAgICJzZW50ZW5jZV90cmFuc2Zvcm1lcnMiOiB2ZXIoInNlbnRlbmNlX3RyYW5zZm9ybWVycyIpLA0KICAgICAgICAiZmFzdGFwaSI6IHZlcigiZmFzdGFwaSIpLA0KICAgIH0NCg0KDQpkZWYgaGFyZHdhcmUoKSAtPiBkaWN0Og0KICAgIGh3ID0geyJjcHUiOiBwbGF0Zm9ybS5wcm9jZXNzb3IoKSwgImN1ZGEiOiBGYWxzZSwgImdwdV9uYW1lIjogTm9uZSwgImdwdV90b3RhbF9tZW1vcnlfZ2IiOiBOb25lfQ0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IHRvcmNoDQoNCiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToNCiAgICAgICAgICAgIGh3WyJjdWRhIl0gPSBUcnVlDQogICAgICAgICAgICBod1siZ3B1X25hbWUiXSA9IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApDQogICAgICAgICAgICBod1siZ3B1X3RvdGFsX21lbW9yeV9nYiJdID0gcm91bmQodG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkudG90YWxfbWVtb3J5IC8gMioqMzAsIDIpDQogICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgcGFzcw0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IHBzdXRpbA0KDQogICAgICAgIGh3WyJyYW1fdG90YWxfZ2IiXSA9IHJvdW5kKHBzdXRpbC52aXJ0dWFsX21lbW9yeSgpLnRvdGFsIC8gMioqMzAsIDIpDQogICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgaHdbInJhbV90b3RhbF9nYiJdID0gTm9uZQ0KICAgIHJldHVybiBodw0KDQoNCmRlZiByZWFkX2pzb24ocGF0aDogUGF0aCk6DQogICAgdHJ5Og0KICAgICAgICByZXR1cm4ganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dCgpKQ0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHJldHVybiBOb25lDQoNCg0KZGVmIG1haW4oKToNCiAgICBwYXJhbXMgPSByZWFkX2pzb24oTU9ERUxTX0RJUiAvICJwYXJhbXMuanNvbiIpIG9yIHt9DQogICAgc3BsaXRfcmVwb3J0ID0gcmVhZF9qc29uKEFSVElGQUNUU19ESVIgLyAic3BsaXRfaW50ZWdyaXR5X3JlcG9ydC5qc29uIikNCiAgICBubGlfdXNlZCA9IHJlYWRfanNvbihEQVRBX1BST0NFU1NFRCAvICJubGlfbW9kZWxfdXNlZC5qc29uIikNCg0KICAgIGRlZiBzaGFfb3Jfbm9uZShyZWxfcGF0aDogUGF0aCk6DQogICAgICAgIHJldHVybiBzaGEyNTYocmVsX3BhdGgpIGlmIHJlbF9wYXRoLmV4aXN0cygpIGVsc2UgTm9uZQ0KDQogICAgYl9wYXRocyA9IHsNCiAgICAgICAgImIyX3J1bl9jb25maWcuanNvbiI6IFJFU1VMVFNfRElSIC8gImIyIiAvICJiMl9ydW5fY29uZmlnLmpzb24iLA0KICAgICAgICAiYjJfbW9kZWxfY29tcGFyaXNvbi5qc29uIjogUkVTVUxUU19ESVIgLyAiYjIiIC8gImIyX21vZGVsX2NvbXBhcmlzb24uanNvbiIsDQogICAgICAgICJiMl9wcmVkaWN0aW9ucy5wYXJxdWV0IjogUkVTVUxUU19ESVIgLyAiYjIiIC8gImIyX3ByZWRpY3Rpb25zLnBhcnF1ZXQiLA0KICAgICAgICAiYjJfeGdib29zdF9zZWVkXzQyLmpvYmxpYiI6IE1PREVMU19ESVIgLyAiYjIiIC8gInhnYm9vc3Rfc2VlZF80Mi5qb2JsaWIiLA0KICAgICAgICAiYjNfcHJlZGljdGlvbnMucGFycXVldCI6IFJFU1VMVFNfRElSIC8gImIzIiAvICJiM19wcmVkaWN0aW9ucy5wYXJxdWV0IiwNCiAgICAgICAgImIzX2RhdGFzZXRfbWV0cmljcy5qc29uIjogUkVTVUxUU19ESVIgLyAiYjMiIC8gImIzX2RhdGFzZXRfbWV0cmljcy5qc29uIiwNCiAgICAgICAgImIzX2Jvb3RzdHJhcF9jaXMuanNvbiI6IFJFU1VMVFNfRElSIC8gImIzIiAvICJiM19ib290c3RyYXBfY2lzLmpzb24iLA0KICAgICAgICAiYjNfcnVuX2NvbmZpZy5qc29uIjogUkVTVUxUU19ESVIgLyAiYjMiIC8gImIzX3J1bl9jb25maWcuanNvbiIsDQogICAgICAgICJiNF9wcmVkaWN0aW9ucy5wYXJxdWV0IjogUkVTVUxUU19ESVIgLyAiYjQiIC8gImI0X3ByZWRpY3Rpb25zLnBhcnF1ZXQiLA0KICAgICAgICAiYjRfY2FsaWJyYXRpb25fbWV0cmljcy5qc29uIjogUkVTVUxUU19ESVIgLyAiYjQiIC8gImI0X2NhbGlicmF0aW9uX21ldHJpY3MuanNvbiIsDQogICAgICAgICJiNF90YXJnZXRfY2FsaWJyYXRpb24uanNvbiI6IFJFU1VMVFNfRElSIC8gImI0IiAvICJiNF90YXJnZXRfY2FsaWJyYXRpb24uanNvbiIsDQogICAgICAgICJiNF9jYWxpYnJhdG9yX3BsYXR0X3NvdXJjZV9zZWVkXzQyLmpvYmxpYiI6IE1PREVMU19ESVIgLyAiYjQiIC8gImNhbGlicmF0b3JfcGxhdHRfc291cmNlX3NlZWRfNDIuam9ibGliIiwNCiAgICAgICAgImI0X2NhbGlicmF0b3JfaXNvdG9uaWNfc291cmNlX3NlZWRfNDIuam9ibGliIjogTU9ERUxTX0RJUiAvICJiNCIgLyAiY2FsaWJyYXRvcl9pc290b25pY19zb3VyY2Vfc2VlZF80Mi5qb2JsaWIiLA0KICAgICAgICAidW5pZmllZF9yZWNvcmRzLnBhcnF1ZXQiOiBEQVRBX1BST0NFU1NFRCAvICJ1bmlmaWVkX3JlY29yZHMucGFycXVldCIsDQogICAgICAgICJiM19leHRlcm5hbF9mZWF0dXJlcy5wYXJxdWV0IjogREFUQV9QUk9DRVNTRUQgLyAiYjNfZXh0ZXJuYWxfZmVhdHVyZXMucGFycXVldCIsDQogICAgfQ0KDQogICAgbWFuaWZlc3QgPSB7DQogICAgICAgICJnZW5lcmF0ZWRfYXQiOiBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQoKSwNCiAgICAgICAgImdpdF9jb21taXQiOiBnaXRfY29tbWl0KCksDQogICAgICAgICJtb2RlbF92ZXJzaW9uIjogcGFyYW1zLmdldCgibW9kZWxfdmVyc2lvbiIpLA0KICAgICAgICAiZmVhdHVyZV92ZXJzaW9uIjogImNvdXJzZS12MS4wIiwNCiAgICAgICAgIm5fZmVhdHVyZXMiOiBwYXJhbXMuZ2V0KCJuX2ZlYXR1cmVzIiksDQogICAgICAgICJubGlfbW9kZWwiOiBwYXJhbXMuZ2V0KCJubGlfbW9kZWwiKSwNCiAgICAgICAgIm5saV9wcm92ZW5hbmNlIjogbmxpX3VzZWQsDQogICAgICAgICJzcGxpdF9yZXBvcnQiOiBzcGxpdF9yZXBvcnQsDQogICAgICAgICJkYXRhc2V0X3NoYTI1NiI6IHsNCiAgICAgICAgICAgICJxYV9jbGVhbi5wYXJxdWV0Ijogc2hhMjU2KFNIQV9GSUxFU1swXSksDQogICAgICAgICAgICAiZmVhdHVyZXNfZnVsbC5wYXJxdWV0Ijogc2hhMjU2KFNIQV9GSUxFU1sxXSksDQogICAgICAgICAgICAibW9kZWxfeGdib29zdF9yYXcuam9ibGliIjogc2hhMjU2KFNIQV9GSUxFU1syXSksDQogICAgICAgICAgICAibW9kZWxfeGdib29zdF9jYWxpYnJhdGVkLmpvYmxpYiI6IHNoYTI1NihTSEFfRklMRVNbM10pLA0KICAgICAgICAgICAgInNwbGl0X2luZGljZXMuanNvbiI6IHNoYTI1NihTSEFfRklMRVNbNF0pLA0KICAgICAgICAgICAgInNwbGl0X2ludGVncml0eV9yZXBvcnQuanNvbiI6IHNoYTI1NihTSEFfRklMRVNbNV0pLA0KICAgICAgICB9LA0KICAgICAgICAiYl9hcnRpZmFjdHNfc2hhMjU2Ijoge25hbWU6IHNoYV9vcl9ub25lKHApIGZvciBuYW1lLCBwIGluIGJfcGF0aHMuaXRlbXMoKX0sDQogICAgICAgICJ2ZXJzaW9ucyI6IHZlcnNpb25zKCksDQogICAgICAgICJoYXJkd2FyZSI6IGhhcmR3YXJlKCksDQogICAgICAgICJhcnRpZmFjdHMiOiBzb3J0ZWQoDQogICAgICAgICAgICBzdHIocC5yZWxhdGl2ZV90byhBUlRJRkFDVFNfRElSLnBhcmVudCkpDQogICAgICAgICAgICBmb3IgcCBpbiBbDQogICAgICAgICAgICAgICAgKkFSVElGQUNUU19ESVIucmdsb2IoIioiKSwNCiAgICAgICAgICAgICAgICAqREFUQV9QUk9DRVNTRUQuZ2xvYigiKi5wYXJxdWV0IiksDQogICAgICAgICAgICAgICAgREFUQV9QUk9DRVNTRUQgLyAibmxpX21vZGVsX3VzZWQuanNvbiIsDQogICAgICAgICAgICBdDQogICAgICAgICAgICBpZiBwLmlzX2ZpbGUoKQ0KICAgICAgICApLA0KICAgIH0NCg0KICAgIFJFU1VMVFNfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBvdXQgPSBSRVNVTFRTX0RJUiAvICJtYW5pZmVzdC5qc29uIg0KICAgIG91dC53cml0ZV90ZXh0KGpzb24uZHVtcHMobWFuaWZlc3QsIGluZGVudD0yKSkNCiAgICBsb2dnZXIuaW5mbyhmIlNhdmVkIG1hbmlmZXN0IHRvIHtvdXR9IikNCiAgICByZXR1cm4gbWFuaWZlc3QNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIG1haW4oKQ0K",
 "src/models/run_b2_baselines.py": "...",
 "src/models/run_b3_cross_domain.py": "...",
 "src/models/run_b4_calibration_shift.py": "...",
 "src/models/run_b5_explanation_reliability.py": "...",
 "src/models/train_baselines.py": "IiIiCkJhc2VsaW5lIG1vZGVsaW5nIHNjcmlwdCBmb3IgSGFsdVJJU0MuClRyYWlucyBhbmQgZXZhbHVhdGVzIEhldXJpc3RpYyBSdWxlLCBMb2dpc3RpYyBSZWdyZXNzaW9uLCBSYW5kb20gRm9yZXN0LCBhbmQgWEdCb29zdCBtb2RlbHMKb24gZXh0cmFjdGVkIGZlYXR1cmVzLCBwcmludGluZyBhIHBlcmZvcm1hbmNlIGNvbXBhcmlzb24gdGFibGUuCiIiIgoKaW1wb3J0IG9zCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2tsZWFybi5wcmVwcm9jZXNzaW5nIGltcG9ydCBTdGFuZGFyZFNjYWxlcgpmcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBMb2dpc3RpY1JlZ3Jlc3Npb24KZnJvbSBza2xlYXJuLmVuc2VtYmxlIGltcG9ydCBSYW5kb21Gb3Jlc3RDbGFzc2lmaWVyCmZyb20geGdib29zdCBpbXBvcnQgWEdCQ2xhc3NpZmllcgpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgKAogICAgcHJlY2lzaW9uX3Njb3JlLCByZWNhbGxfc2NvcmUsIGYxX3Njb3JlLCAKICAgIHJvY19hdWNfc2NvcmUsIGF2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlLCBtYXR0aGV3c19jb3JyY29lZgopCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcyIpCgpGRUFUVVJFU19QQVRIID0gb3MucGF0aC5qb2luKCJkYXRhIiwgInByb2Nlc3NlZCIsICJmZWF0dXJlc19jb3JlLnBhcnF1ZXQiKQpSRVNVTFRTX0RJUiA9IG9zLnBhdGguam9pbigiYXJ0aWZhY3RzIiwgInJlc3VsdHMiKQpNT0RFTFNfRElSID0gb3MucGF0aC5qb2luKCJhcnRpZmFjdHMiLCAibW9kZWxzIikKCmRlZiBldmFsdWF0ZV9wcmVkaWN0aW9ucyh5X3RydWUsIHlfcHJlZCwgeV9wcm9iKSAtPiBkaWN0OgogICAgIiIiQ2FsY3VsYXRlcyBldmFsdWF0aW9uIG1ldHJpY3MgZm9yIGJpbmFyeSBjbGFzc2lmaWNhdGlvbi4iIiIKICAgIHJldHVybiB7CiAgICAgICAgInByZWNpc2lvbiI6IGZsb2F0KHByZWNpc2lvbl9zY29yZSh5X3RydWUsIHlfcHJlZCwgemVyb19kaXZpc2lvbj0wKSksCiAgICAgICAgInJlY2FsbCI6IGZsb2F0KHJlY2FsbF9zY29yZSh5X3RydWUsIHlfcHJlZCwgemVyb19kaXZpc2lvbj0wKSksCiAgICAgICAgImYxIjogZmxvYXQoZjFfc2NvcmUoeV90cnVlLCB5X3ByZWQsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJhdXJvYyI6IGZsb2F0KHJvY19hdWNfc2NvcmUoeV90cnVlLCB5X3Byb2IpKSwKICAgICAgICAicHJfYXVjIjogZmxvYXQoYXZlcmFnZV9wcmVjaXNpb25fc2NvcmUoeV90cnVlLCB5X3Byb2IpKSwKICAgICAgICAibWNjIjogZmxvYXQobWF0dGhld3NfY29ycmNvZWYoeV90cnVlLCB5X3ByZWQpKQogICAgfQoKZGVmIHJ1bl9oZXVyaXN0aWNfYmFzZWxpbmUodmFsX2RmOiBwZC5EYXRhRnJhbWUsIHRlc3RfZGY6IHBkLkRhdGFGcmFtZSk6CiAgICAiIiJSdWxlLWJhc2VkIGhldXJpc3RpYzogaGlnaCBvdmVybGFwX2Fuc3dlcl9jb250ZXh0IC0+IGxvdyByaXNrICgwKSwgbG93IG92ZXJsYXAgLT4gaGlnaCByaXNrICgxKS4iIiIKICAgICMgVGhyZXNob2xkIHR1bmVkIG9uIHZhbCBzZXQKICAgIHRocmVzaG9sZHMgPSBucC5saW5zcGFjZSgwLCAxLCAxMDEpCiAgICBiZXN0X3RocmVzaCA9IDAuNQogICAgYmVzdF92YWxfZjEgPSAwLjAKCiAgICBmb3IgdCBpbiB0aHJlc2hvbGRzOgogICAgICAgIHZhbF9wcmVkcyA9ICh2YWxfZGZbIm92ZXJsYXBfYW5zd2VyX2NvbnRleHQiXSA8IHQpLmFzdHlwZShpbnQpCiAgICAgICAgZjEgPSBmMV9zY29yZSh2YWxfZGZbImxhYmVsIl0sIHZhbF9wcmVkcywgemVyb19kaXZpc2lvbj0wKQogICAgICAgIGlmIGYxID4gYmVzdF92YWxfZjE6CiAgICAgICAgICAgIGJlc3RfdmFsX2YxID0gZjEKICAgICAgICAgICAgYmVzdF90aHJlc2ggPSB0CgogICAgbG9nZ2luZy5pbmZvKGYiSGV1cmlzdGljIGJlc3Qgb3ZlcmxhcCB0aHJlc2hvbGQgb24gVmFsOiB7YmVzdF90aHJlc2g6LjJmfSAoRjE6IHtiZXN0X3ZhbF9mMTouNGZ9KSIpCgogICAgIyBQcmVkaWN0IG9uIHRlc3QKICAgIHRlc3RfcHJvYnMgPSAxLjAgLSB0ZXN0X2RmWyJvdmVybGFwX2Fuc3dlcl9jb250ZXh0Il0KICAgIHRlc3RfcHJlZHMgPSAodGVzdF9kZlsib3ZlcmxhcF9hbnN3ZXJfY29udGV4dCJdIDwgYmVzdF90aHJlc2gpLmFzdHlwZShpbnQpCgogICAgbWV0cmljcyA9IGV2YWx1YXRlX3ByZWRpY3Rpb25zKHRlc3RfZGZbImxhYmVsIl0sIHRlc3RfcHJlZHMsIHRlc3RfcHJvYnMpCiAgICByZXR1cm4gbWV0cmljcywgYmVzdF90aHJlc2gKCmRlZiB0cmFpbl9hbmRfZXZhbF9hbGwoKToKICAgIG9zLm1ha2VkaXJzKFJFU1VMVFNfRElSLCBleGlzdF9vaz1UcnVlKQogICAgb3MubWFrZWRpcnMoTU9ERUxTX0RJUiwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoRkVBVFVSRVNfUEFUSCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJ7RkVBVFVSRVNfUEFUSH0gbm90IGZvdW5kLiBSdW4gc3JjL2ZlYXR1cmVzL2V4dHJhY3RfZmVhdHVyZXMucHkgZmlyc3QuIikKCiAgICBkZiA9IHBkLnJlYWRfcGFycXVldChGRUFUVVJFU19QQVRIKQogICAgZmVhdHVyZV9jb2xzID0gW2MgZm9yIGMgaW4gZGYuY29sdW1ucyBpZiBjIG5vdCBpbiBbInNhbXBsZV9pZCIsICJpdGVtX2lkeCIsICJsYWJlbCIsICJzcGxpdCJdXQoKICAgIHRyYWluX2RmID0gZGZbZGZbInNwbGl0Il0gPT0gInRyYWluIl0KICAgIHZhbF9kZiA9IGRmW2RmWyJzcGxpdCJdID09ICJ2YWwiXQogICAgdGVzdF9kZiA9IGRmW2RmWyJzcGxpdCJdID09ICJ0ZXN0Il0KCiAgICBsb2dnaW5nLmluZm8oZiJUcmFpbiBzYW1wbGVzOiB7bGVuKHRyYWluX2RmKX0sIFZhbDoge2xlbih2YWxfZGYpfSwgVGVzdDoge2xlbih0ZXN0X2RmKX0iKQogICAgbG9nZ2luZy5pbmZvKGYiRmVhdHVyZSBsaXN0ICh7bGVuKGZlYXR1cmVfY29scyl9KToge2ZlYXR1cmVfY29sc30iKQoKICAgIFhfdHJhaW4sIHlfdHJhaW4gPSB0cmFpbl9kZltmZWF0dXJlX2NvbHNdLCB0cmFpbl9kZlsibGFiZWwiXQogICAgWF92YWwsIHlfdmFsID0gdmFsX2RmW2ZlYXR1cmVfY29sc10sIHZhbF9kZlsibGFiZWwiXQogICAgWF90ZXN0LCB5X3Rlc3QgPSB0ZXN0X2RmW2ZlYXR1cmVfY29sc10sIHRlc3RfZGZbImxhYmVsIl0KCiAgICByZXN1bHRzID0ge30KCiAgICAjIDEuIEhldXJpc3RpYyBCYXNlbGluZQogICAgaGV1cl9tZXRyaWNzLCBoZXVyX3RocmVzaCA9IHJ1bl9oZXVyaXN0aWNfYmFzZWxpbmUodmFsX2RmLCB0ZXN0X2RmKQogICAgcmVzdWx0c1siSGV1cmlzdGljIChPdmVybGFwKSJdID0gaGV1cl9tZXRyaWNzCgogICAgIyAyLiBMb2dpc3RpYyBSZWdyZXNzaW9uIChTY2FsZWQpCiAgICBzY2FsZXIgPSBTdGFuZGFyZFNjYWxlcigpCiAgICBYX3RyYWluX3NjYWxlZCA9IHNjYWxlci5maXRfdHJhbnNmb3JtKFhfdHJhaW4pCiAgICBYX3Rlc3Rfc2NhbGVkID0gc2NhbGVyLnRyYW5zZm9ybShYX3Rlc3QpCgogICAgbHIgPSBMb2dpc3RpY1JlZ3Jlc3Npb24obWF4X2l0ZXI9MjAwMCwgcmFuZG9tX3N0YXRlPTQyKQogICAgbHIuZml0KFhfdHJhaW5fc2NhbGVkLCB5X3RyYWluKQogICAgbHJfcHJvYnMgPSBsci5wcmVkaWN0X3Byb2JhKFhfdGVzdF9zY2FsZWQpWzosIDFdCiAgICBscl9wcmVkcyA9IChscl9wcm9icyA+PSAwLjUpLmFzdHlwZShpbnQpCiAgICByZXN1bHRzWyJMb2dpc3RpYyBSZWdyZXNzaW9uIl0gPSBldmFsdWF0ZV9wcmVkaWN0aW9ucyh5X3Rlc3QsIGxyX3ByZWRzLCBscl9wcm9icykKCiAgICAjIDMuIFJhbmRvbSBGb3Jlc3QKICAgIHJmID0gUmFuZG9tRm9yZXN0Q2xhc3NpZmllcihuX2VzdGltYXRvcnM9MzAwLCBtaW5fc2FtcGxlc19sZWFmPTUsIHJhbmRvbV9zdGF0ZT00Miwgbl9qb2JzPS0xKQogICAgcmYuZml0KFhfdHJhaW4sIHlfdHJhaW4pCiAgICByZl9wcm9icyA9IHJmLnByZWRpY3RfcHJvYmEoWF90ZXN0KVs6LCAxXQogICAgcmZfcHJlZHMgPSAocmZfcHJvYnMgPj0gMC41KS5hc3R5cGUoaW50KQogICAgcmVzdWx0c1siUmFuZG9tIEZvcmVzdCJdID0gZXZhbHVhdGVfcHJlZGljdGlvbnMoeV90ZXN0LCByZl9wcmVkcywgcmZfcHJvYnMpCgogICAgIyA0LiBYR0Jvb3N0CiAgICB4Z2IgPSBYR0JDbGFzc2lmaWVyKAogICAgICAgIG5fZXN0aW1hdG9ycz0zMDAsIG1heF9kZXB0aD01LCBsZWFybmluZ19yYXRlPTAuMDUsIAogICAgICAgIGV2YWxfbWV0cmljPSJsb2dsb3NzIiwgcmFuZG9tX3N0YXRlPTQyLCBuX2pvYnM9LTEKICAgICkKICAgIHhnYi5maXQoWF90cmFpbiwgeV90cmFpbikKICAgIHhnYl9wcm9icyA9IHhnYi5wcmVkaWN0X3Byb2JhKFhfdGVzdClbOiwgMV0KICAgIHhnYl9wcmVkcyA9ICh4Z2JfcHJvYnMgPj0gMC41KS5hc3R5cGUoaW50KQogICAgcmVzdWx0c1siWEdCb29zdCAoRGVmYXVsdCkiXSA9IGV2YWx1YXRlX3ByZWRpY3Rpb25zKHlfdGVzdCwgeGdiX3ByZWRzLCB4Z2JfcHJvYnMpCgogICAgIyBDb252ZXJ0IHJlc3VsdHMgdG8gRGF0YUZyYW1lIGFuZCBkaXNwbGF5CiAgICByZXN1bHRzX2RmID0gcGQuRGF0YUZyYW1lKHJlc3VsdHMpLlQKICAgIHJlc3VsdHNfZGYgPSByZXN1bHRzX2RmW1sicHJlY2lzaW9uIiwgInJlY2FsbCIsICJmMSIsICJhdXJvYyIsICJwcl9hdWMiLCAibWNjIl1dCiAgICAKICAgIHByaW50KCJcbiIgKyAiPSIqODApCiAgICBwcmludCgiIEhhbHVSSVNDIEJhc2VsaW5lIE1vZGVsIENvbXBhcmlzb24gb24gVGVzdCBTZXQgKENvcmUgRmVhdHVyZXMpIikKICAgIHByaW50KCI9Iio4MCkKICAgIHByaW50KHJlc3VsdHNfZGYudG9fc3RyaW5nKCkpCiAgICBwcmludCgiPSIqODAgKyAiXG4iKQoKICAgICMgU2F2ZSByZXN1bHRzIHRvIEpTT04gYW5kIENTVgogICAgcmVzdWx0c19kZi50b19jc3Yob3MucGF0aC5qb2luKFJFU1VMVFNfRElSLCAiYmFzZWxpbmVfcmVzdWx0cy5jc3YiKSkKICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4oUkVTVUxUU19ESVIsICJiYXNlbGluZV9yZXN1bHRzLmpzb24iKSwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChyZXN1bHRzLCBmLCBpbmRlbnQ9MikKCiAgICBsb2dnaW5nLmluZm8oZiJTYXZlZCBiYXNlbGluZSBldmFsdWF0aW9uIHJlc3VsdHMgdG8ge1JFU1VMVFNfRElSfSIpCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdHJhaW5fYW5kX2V2YWxfYWxsKCkK",
 "src/models/train_pipeline.py": "...",
 "src/models/verify_artifacts.py": "IiIiClBvc3QtcnVuIGFydGlmYWN0IHZlcmlmaWNhdGlvbiAoQ29sYWIgY2VsbCA3aSArIGxvY2FsIHBvc3QtZG93bmxvYWQgY2hlY2spLgoKTG9hZHMgZXZlcnkgYXJ0aWZhY3QgdGhlIHBpcGVsaW5lIHByb2R1Y2VzIGFuZCBwcm92ZXMgaXQgY2FuIGJlIHVzZWQgb24gVEhJUwptYWNoaW5lLCBwcmV2ZW50aW5nIHRoZSBoaXN0b3JpY2FsICJkb3dubG9hZGVkIG1vZGVsIGZhaWxzIHRvIGxvYWQiIGNsYXNzIG9mCmVycm9ycyAoQ1VEQS10cmFpbmVkIGJvb3N0ZXJzIG5vdCBwb3J0aW5nIGFjcm9zcyBwbGF0Zm9ybXMpOgoKICAtIEIyIFhHQm9vc3QgYm9vc3RlcnMgKDMgc2VlZHMpIGxvYWQgYW5kIHByZWRpY3Qgd2l0aGluIFswLCAxXQogIC0gQjQgc291cmNlICsgdGFyZ2V0IGNhbGlicmF0b3JzIChwdXJlIHNrbGVhcm4pIGxvYWQgYW5kIGFwcGx5CiAgLSBCMi9CMy9CNCBwcmVkaWN0aW9uIHBhcnF1ZXQgZmlsZXMgZXhpc3Qgd2l0aCB0aGUgZXhwZWN0ZWQgc2NoZW1hCiAgLSBGZWF0dXJlLWNvbHVtbiBvcmRlciBtYXRjaGVzIHRoZSBCMiBjb25maWcKCkV4aXQgY29kZSAwID0gZXZlcnl0aGluZyB1c2FibGU7IDEgPSBmaXJzdCBmYWlsaW5nIGNoZWNrIChjbGVhciBtZXNzYWdlKS4KClJ1biAocmVwbyByb290LCAudmVudiBvciBDb2xhYiBhZnRlciBCMy9CNCk6CiAgcHl0aG9uIHNyYy9tb2RlbHMvdmVyaWZ5X2FydGlmYWN0cy5weQoiIiIKCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdKSkKCmltcG9ydCBqb2JsaWIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmxvZ2dpbmcuYmFzaWNDb25maWcobGV2ZWw9bG9nZ2luZy5JTkZPLCBmb3JtYXQ9IiUoYXNjdGltZSlzIC0gJShsZXZlbG5hbWUpcyAtICUobWVzc2FnZSlzIikKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoInZlcmlmeV9hcnRpZmFjdHMiKQoKZnJvbSBzcmMubW9kZWxzLmNvbmZpZyBpbXBvcnQgREFUQV9QUk9DRVNTRUQsIE1PREVMU19ESVIsIFJFU1VMVFNfRElSICAjIG5vcWE6IEU0MDIKCkIyX01PREVMU19ESVIgPSBNT0RFTFNfRElSIC8gImIyIgpCMl9SRVNVTFRTID0gUkVTVUxUU19ESVIgLyAiYjIiCkIzX1JFU1VMVFMgPSBSRVNVTFRTX0RJUiAvICJiMyIKQjRfUkVTVUxUUyA9IFJFU1VMVFNfRElSIC8gImI0IgpCNF9NT0RFTFMgPSBNT0RFTFNfRElSIC8gImI0IgpGRUFUVVJFU19GVUxMID0gREFUQV9QUk9DRVNTRUQgLyAiZmVhdHVyZXNfZnVsbC5wYXJxdWV0IgpTRUVEUyA9IFs0MiwgMTIzLCA0NTZdCgpGQUlMVVJFUyA9IFtdCgoKZGVmIGNoZWNrKG5hbWUsIGZuKToKICAgIHRyeToKICAgICAgICBmbigpCiAgICAgICAgbG9nZ2VyLmluZm8oZiJPSyAge25hbWV9IikKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBGQUlMVVJFUy5hcHBlbmQobmFtZSkKICAgICAgICBsb2dnZXIuZXJyb3IoZiJGQUlMIHtuYW1lfToge2V9IikKCgpkZWYgbWFpbigpIC0+IGludDoKICAgIGIyX2NmZyA9IGpzb24ubG9hZHMoKEIyX1JFU1VMVFMgLyAiYjJfcnVuX2NvbmZpZy5qc29uIikucmVhZF90ZXh0KCkpCiAgICBmZWF0dXJlX2NvbHMgPSBsaXN0KGIyX2NmZ1siZmVhdHVyZV9jb2xzIl0pCiAgICBsb2dnZXIuaW5mbyhmIkIyIGNvbmZpZyBsb2FkZWQ6IHtsZW4oZmVhdHVyZV9jb2xzKX0gZmVhdHVyZXMiKQoKICAgIGlmIG5vdCBGRUFUVVJFU19GVUxMLmV4aXN0cygpOgogICAgICAgIGxvZ2dlci5lcnJvcigiZmVhdHVyZXNfZnVsbC5wYXJxdWV0IG1pc3NpbmcgKHJ1biBjZWxsIDYpIikKICAgICAgICByZXR1cm4gMQogICAgZmVhdHVyZXMgPSBwZC5yZWFkX3BhcnF1ZXQoRkVBVFVSRVNfRlVMTCkKICAgIHNhbXBsZSA9IGZlYXR1cmVzW2ZlYXR1cmVzWyJzcGxpdCJdID09ICJ2YWwiXS5oZWFkKDMyKQoKICAgIGZvciBzZWVkIGluIFNFRURTOgogICAgICAgIGRlZiBfbG9hZF9wcmVkaWN0KHNlZWQ9c2VlZCk6CiAgICAgICAgICAgIG1vZGVsID0gam9ibGliLmxvYWQoQjJfTU9ERUxTX0RJUiAvIGYieGdib29zdF9zZWVkX3tzZWVkfS5qb2JsaWIiKQogICAgICAgICAgICBwID0gbW9kZWwucHJlZGljdF9wcm9iYShzYW1wbGVbZmVhdHVyZV9jb2xzXS52YWx1ZXMpWzosIDFdCiAgICAgICAgICAgIGFzc2VydCAocCA+PSAwLjApLmFsbCgpIGFuZCAocCA8PSAxLjApLmFsbCgpLCAicHJvYmFiaWxpdGllcyBvdXQgb2YgWzAsMV0iCiAgICAgICAgY2hlY2soZiJCMiB4Z2Jvb3N0X3NlZWRfe3NlZWR9IGxvYWRzIGFuZCBwcmVkaWN0cyIsIF9sb2FkX3ByZWRpY3QpCgogICAgZm9yIG5hbWUgaW4gKCJtb2RlbF94Z2Jvb3N0X3Jhdy5qb2JsaWIiLCAibW9kZWxfeGdib29zdF9jYWxpYnJhdGVkLmpvYmxpYiIpOgogICAgICAgIHBhdGggPSBNT0RFTFNfRElSIC8gbmFtZQogICAgICAgIGlmIHBhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIGRlZiBfbG9hZF92ZXJzaW9uX2EocGF0aD1wYXRoKToKICAgICAgICAgICAgICAgIGJ1bmRsZSA9IGpvYmxpYi5sb2FkKHBhdGgpCiAgICAgICAgICAgICAgICBtb2RlbCA9IGJ1bmRsZVsibW9kZWwiXSBpZiBpc2luc3RhbmNlKGJ1bmRsZSwgZGljdCkgYW5kICJtb2RlbCIgaW4gYnVuZGxlIGVsc2UgYnVuZGxlCiAgICAgICAgICAgICAgICBwID0gbW9kZWwucHJlZGljdF9wcm9iYShzYW1wbGVbZmVhdHVyZV9jb2xzXS52YWx1ZXMpWzosIDFdCiAgICAgICAgICAgICAgICBhc3NlcnQgKHAgPj0gMC4wKS5hbGwoKSBhbmQgKHAgPD0gMS4wKS5hbGwoKSwgIlZlcnNpb24gQSBwcm9iYWJpbGl0aWVzIG91dCBvZiBbMCwxXSIKICAgICAgICAgICAgY2hlY2soZiJWZXJzaW9uIEEge25hbWV9IGxvYWRzIGFuZCBwcmVkaWN0cyIsIF9sb2FkX3ZlcnNpb25fYSkKCiAgICBkZWYgX2NoZWNrX2IyX3BhcnF1ZXQoKToKICAgICAgICBwcmVkcyA9IHBkLnJlYWRfcGFycXVldChCMl9SRVNVTFRTIC8gImIyX3ByZWRpY3Rpb25zLnBhcnF1ZXQiKQogICAgICAgIGFzc2VydCB7InNhbXBsZV9pZCIsICJtb2RlbCIsICJzY29yZSIsICJwcmVkIiwgImxhYmVsIn0gPD0gc2V0KHByZWRzLmNvbHVtbnMpCiAgICAgICAgYXNzZXJ0IHByZWRzWyJtb2RlbCJdLm51bmlxdWUoKSA+PSAxCiAgICBjaGVjaygiQjIgYjJfcHJlZGljdGlvbnMucGFycXVldCBzY2hlbWEiLCBfY2hlY2tfYjJfcGFycXVldCkKCiAgICBkZWYgX2NoZWNrX2IzKCk6CiAgICAgICAgcHJlZHMgPSBwZC5yZWFkX3BhcnF1ZXQoQjNfUkVTVUxUUyAvICJiM19wcmVkaWN0aW9ucy5wYXJxdWV0IikKICAgICAgICBhc3NlcnQgeyJzYW1wbGVfaWQiLCAic291cmNlX2RhdGFzZXQiLCAic291cmNlX2dyb3VwX2lkIiwgInRhc2siLCAibGFiZWwiLCAibW9kZWwiLCAic2NvcmUiLCAicHJlZCJ9IDw9IHNldChwcmVkcy5jb2x1bW5zKQogICAgICAgIGFzc2VydCB7InhnYm9vc3Rfc2VlZF80MiIsICJ4Z2Jvb3N0X3NlZWRfMTIzIiwgInhnYm9vc3Rfc2VlZF80NTYifSA8PSBzZXQocHJlZHNbIm1vZGVsIl0pCiAgICAgICAganNvbi5sb2FkcygoQjNfUkVTVUxUUyAvICJiM19kYXRhc2V0X21ldHJpY3MuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgICAgIGpzb24ubG9hZHMoKEIzX1JFU1VMVFMgLyAiYjNfYm9vdHN0cmFwX2Npcy5qc29uIikucmVhZF90ZXh0KCkpCiAgICAgICAgZXJyb3JfcGF0aCA9IEIzX1JFU1VMVFMgLyAiYjNfZXJyb3JfY2FzZXMuanNvbiIKICAgICAgICBpZiBlcnJvcl9wYXRoLmV4aXN0cygpOgogICAgICAgICAgICBmb3IgY2FzZSBpbiBqc29uLmxvYWRzKGVycm9yX3BhdGgucmVhZF90ZXh0KCkpOgogICAgICAgICAgICAgICAgaWYgY2FzZS5nZXQoInNvdXJjZV9kYXRhc2V0IikgPT0gImZhaXRoYmVuY2giOgogICAgICAgICAgICAgICAgICAgIGFzc2VydCBub3QgYW55KGNhc2UuZ2V0KGssICIiKSBmb3IgayBpbiAoInF1ZXN0aW9uIiwgImNvbnRleHQiLCAiYW5zd2VyIiwgInNwYW5fYW5ub3RhdGlvbnMiKSksIFwKICAgICAgICAgICAgICAgICAgICAgICAgInVucmVkYWN0ZWQgRmFpdGhCZW5jaCB0ZXh0IGluIEIzIGVycm9yIGNhc2VzIgogICAgY2hlY2soIkIzIHByZWRpY3Rpb25zICsgcmVwb3J0cyIsIF9jaGVja19iMykKCiAgICBmb3IgbmFtZSBpbiAoImNhbGlicmF0b3JfcGxhdHRfc291cmNlX3NlZWRfNDIuam9ibGliIiwKICAgICAgICAgICAgICAgICAiY2FsaWJyYXRvcl9pc290b25pY19zb3VyY2Vfc2VlZF80Mi5qb2JsaWIiLAogICAgICAgICAgICAgICAgICJjYWxpYnJhdG9yX3BsYXR0X3RhcmdldF9yYWd0cnV0aF9xYV9zZWVkXzQyLmpvYmxpYiIsCiAgICAgICAgICAgICAgICAgImNhbGlicmF0b3JfaXNvdG9uaWNfdGFyZ2V0X3JhZ3RydXRoX3FhX3NlZWRfNDIuam9ibGliIik6CiAgICAgICAgZGVmIF9sb2FkX2NhbChuYW1lPW5hbWUpOgogICAgICAgICAgICBjYWwgPSBqb2JsaWIubG9hZChCNF9NT0RFTFMgLyBuYW1lKQogICAgICAgICAgICBwID0gY2FsLnByZWRpY3RfcHJvYmEobnAuYXJyYXkoW1swLjFdLCBbMC41XSwgWzAuOV1dKSlbOiwgMV0gaWYgInBsYXR0IiBpbiBuYW1lIFwKICAgICAgICAgICAgICAgIGVsc2UgY2FsLnByZWRpY3QobnAuYXJyYXkoWzAuMSwgMC41LCAwLjldKSkKICAgICAgICAgICAgYXNzZXJ0IChwID49IDAuMCkuYWxsKCkgYW5kIChwIDw9IDEuMCkuYWxsKCksICJjYWxpYnJhdGVkIHNjb3JlcyBvdXQgb2YgWzAsMV0iCiAgICAgICAgY2hlY2soZiJCNCB7bmFtZX0gbG9hZHMgYW5kIGFwcGxpZXMiLCBfbG9hZF9jYWwpCgogICAgZGVmIF9jaGVja19iNCgpOgogICAgICAgIHByZWRzID0gcGQucmVhZF9wYXJxdWV0KEI0X1JFU1VMVFMgLyAiYjRfcHJlZGljdGlvbnMucGFycXVldCIpCiAgICAgICAgYXNzZXJ0IHsic2FtcGxlX2lkIiwgIm1ldGhvZCIsICJzY29yZSIsICJwcmVkIiwgImxhYmVsIn0gPD0gc2V0KHByZWRzLmNvbHVtbnMpCiAgICAgICAgYXNzZXJ0IHsicmF3IiwgInBsYXR0IiwgImlzb3RvbmljIn0gPD0gc2V0KHByZWRzWyJtZXRob2QiXSkKICAgICAgICBqc29uLmxvYWRzKChCNF9SRVNVTFRTIC8gImI0X2NhbGlicmF0aW9uX21ldHJpY3MuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgICAgIGpzb24ubG9hZHMoKEI0X1JFU1VMVFMgLyAiYjRfdGFyZ2V0X2NhbGlicmF0aW9uLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIGNoZWNrKCJCNCBwcmVkaWN0aW9ucyArIHJlcG9ydHMiLCBfY2hlY2tfYjQpCgogICAgaWYgRkFJTFVSRVM6CiAgICAgICAgbG9nZ2VyLmVycm9yKGYiVkVSSUZJQ0FUSU9OIEZBSUxFRCAoe2xlbihGQUlMVVJFUyl9KToge0ZBSUxVUkVTfSIpCiAgICAgICAgcmV0dXJuIDEKICAgIGxvZ2dlci5pbmZvKCJBTEwgQVJUSUZBQ1RTIFZFUklGSUVEOiBwb3J0YWJsZSBtb2RlbHMvY2FsaWJyYXRvcnMgbG9hZCBhbmQgcHJlZGljdCBvbiB0aGlzIG1hY2hpbmUuIikKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHN5cy5leGl0KG1haW4oKSkK",
 "colab/drive_cache.py": "IiIiCkRyaXZlLWNhY2hlIGhlbHBlcnMgZm9yIHRoZSBDb2xhYiBub3RlYm9vayAoY2VsbHMgNWIgLyA2YiAvIDdkLjUgLyA3ZSkuCgpEZXRlcm1pbmlzdGljIGhlYXZ5IGFydGlmYWN0cyAoSGFsdUV2YWwgZmVhdHVyZXMsIEIzIGV4dGVybmFsIGZlYXR1cmVzKSBhcmUKY2FjaGVkIG9uIEdvb2dsZSBEcml2ZSBiZXR3ZWVuIHNlc3Npb25zIHNvIGV4dHJhY3Rpb24gZG9lcyBub3QgcmVwZWF0IG9uIGV2ZXJ5CnJ1bi4gRXZlcnkgcmVzdG9yZSBpcyBWRVJJRklFRCBhZ2FpbnN0IGZyZXNobHkgcHJlcGFyZWQgZGF0YTsgYSBtaXNtYXRjaCBmYWxscwpiYWNrIHRvIHJlLWV4dHJhY3Rpb24gYXV0b21hdGljYWxseS4KCk5vIGdvb2dsZS5jb2xhYiBpbXBvcnRzIGhlcmUsIHNvIHRoZXNlIGZ1bmN0aW9ucyBhcmUgdW5pdC10ZXN0YWJsZSBsb2NhbGx5LgoiIiIKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKRVhQRUNURURfUk9XUyA9IDIwMDAwCkVYUEVDVEVEX0xBQkVMUyA9IHswOiAxMDAwMCwgMTogMTAwMDB9CkVYUEVDVEVEX1NQTElUUyA9IHsidHJhaW4iOiAxNDAwMCwgInZhbCI6IDMwMDAsICJ0ZXN0IjogMzAwMH0KIyBTYW5pdHkgZmVhdHVyZSBuYW1lcyB0aGF0IG11c3QgYmUgcHJlc2VudCAoZnVsbCAyNiBhcmUgY2hlY2tlZCBieSB0aGUgcnVubmVycykKU0FOSVRZX0ZFQVRVUkVTID0gWyJuX2NoYXJzIiwgIm5fd29yZHMiLCAib3ZlcmxhcF9hbnN3ZXJfY29udGV4dCIsCiAgICAgICAgICAgICAgICAgICAibmxpX2N0eF9lbnRhaWxzX2FucyIsICJjb3NpbmVfY3R4X2FucyJdCgoKZGVmIHNoYTI1Nl9maWxlKHBhdGgpIC0+IHN0cjoKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIG9wZW4ocGF0aCwgInJiIikgYXMgZjoKICAgICAgICBmb3IgY2h1bmsgaW4gaXRlcihsYW1iZGE6IGYucmVhZCgxIDw8IDIwKSwgYiIiKToKICAgICAgICAgICAgaC51cGRhdGUoY2h1bmspCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiB2ZXJpZnlfaGFsdWV2YWxfZmVhdHVyZXMoZmVhdHVyZXNfcGF0aCwgcWFfcGF0aCkgLT4gZGljdDoKICAgICIiIlZlcmlmeSBhIGNhY2hlZCBmZWF0dXJlc19mdWxsLnBhcnF1ZXQgYWdhaW5zdCB0aGUgZnJlc2hseSBidWlsdCBxYV9jbGVhbi5wYXJxdWV0LgoKICAgIFJldHVybnMgeyJvayI6IGJvb2wsICJjaGVja3MiOiBbcmVhc29uc119LgogICAgIiIiCiAgICBpbXBvcnQgcGFuZGFzIGFzIHBkCgogICAgdHJ5OgogICAgICAgIGZlYXRzID0gcGQucmVhZF9wYXJxdWV0KGZlYXR1cmVzX3BhdGgpCiAgICAgICAgcWEgPSBwZC5yZWFkX3BhcnF1ZXQocWFfcGF0aCkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICMgY29ycnVwdCBvciB1bnJlYWRhYmxlIGNhY2hlCiAgICAgICAgcmV0dXJuIHsib2siOiBGYWxzZSwgImNoZWNrcyI6IFtmInJlYWQgZmFpbGVkOiB7ZX0iXX0KCiAgICBjaGVja3MgPSBbXQogICAgaWYgbGVuKGZlYXRzKSAhPSBFWFBFQ1RFRF9ST1dTOgogICAgICAgIGNoZWNrcy5hcHBlbmQoZiJyb3dzIHtsZW4oZmVhdHMpfSAhPSB7RVhQRUNURURfUk9XU30iKQogICAgaWYgc2V0KGZlYXRzWyJzYW1wbGVfaWQiXSkgIT0gc2V0KHFhWyJzYW1wbGVfaWQiXSk6CiAgICAgICAgY2hlY2tzLmFwcGVuZCgic2FtcGxlX2lkIHNldHMgZGlmZmVyIGZyb20gcWFfY2xlYW4iKQogICAgaWYgZmVhdHNbImxhYmVsIl0udmFsdWVfY291bnRzKCkudG9fZGljdCgpICE9IEVYUEVDVEVEX0xBQkVMUzoKICAgICAgICBjaGVja3MuYXBwZW5kKGYibGFiZWwgYmFsYW5jZSB7ZmVhdHNbJ2xhYmVsJ10udmFsdWVfY291bnRzKCkudG9fZGljdCgpfSAhPSB7RVhQRUNURURfTEFCRUxTfSIpCiAgICBjb3VudHMgPSBmZWF0cy5ncm91cGJ5KCJzcGxpdCIpLnNpemUoKS50b19kaWN0KCkKICAgIGlmIGNvdW50cyAhPSBFWFBFQ1RFRF9TUExJVFM6CiAgICAgICAgY2hlY2tzLmFwcGVuZChmInNwbGl0IGNvdW50cyB7Y291bnRzfSAhPSB7RVhQRUNURURfU1BMSVRTfSIpCiAgICBpZiAiaXRlbV9pZHgiIGluIGZlYXRzLmNvbHVtbnMgYW5kIGZlYXRzLmdyb3VwYnkoIml0ZW1faWR4IilbInNwbGl0Il0ubnVuaXF1ZSgpLm1heCgpICE9IDE6CiAgICAgICAgY2hlY2tzLmFwcGVuZCgiaXRlbV9pZHggZ3JvdXBzIHNwYW4gbXVsdGlwbGUgc3BsaXRzIChsZWFrYWdlKSIpCiAgICBtaXNzaW5nX2ZlYXRzID0gW2MgZm9yIGMgaW4gU0FOSVRZX0ZFQVRVUkVTIGlmIGMgbm90IGluIGZlYXRzLmNvbHVtbnNdCiAgICBpZiBtaXNzaW5nX2ZlYXRzOgogICAgICAgIGNoZWNrcy5hcHBlbmQoZiJtaXNzaW5nIGZlYXR1cmUgY29sdW1ucyB7bWlzc2luZ19mZWF0c30iKQogICAgcmV0dXJuIHsib2siOiBub3QgY2hlY2tzLCAiY2hlY2tzIjogY2hlY2tzfQoKCmRlZiByZWFkX2NhY2hlX21ldGEobWV0YV9wYXRoKSAtPiBkaWN0OgogICAgdHJ5OgogICAgICAgIHJldHVybiBqc29uLmxvYWRzKFBhdGgobWV0YV9wYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgIHJldHVybiB7fQoKCmRlZiBjYWNoZV9rZXlfbWF0Y2hlcyhtZXRhOiBkaWN0LCB1bmlmaWVkX3BhcnF1ZXRfcGF0aCkgLT4gYm9vbDoKICAgICIiIkIzIGV4dGVybmFsLWZlYXR1cmUgY2FjaGUgaXMgcmV1c2FibGUgb25seSB3aGVuIHRoZSB1bmlmaWVkIHBhcnF1ZXQgaGFzaCBtYXRjaGVzLiIiIgogICAgaWYgbm90IG1ldGEuZ2V0KCJpbnB1dF9zaGEyNTYiKToKICAgICAgICByZXR1cm4gRmFsc2UKICAgIHRyeToKICAgICAgICByZXR1cm4gbWV0YVsiaW5wdXRfc2hhMjU2Il0gPT0gc2hhMjU2X2ZpbGUodW5pZmllZF9wYXJxdWV0X3BhdGgpCiAgICBleGNlcHQgT1NFcnJvcjoKICAgICAgICByZXR1cm4gRmFsc2UKCgpkZWYgcmVzdG9yZV9jb25maWdfaGFzaChjb25maWdfanNvbl9wYXRoLCBrZXk6IHN0ciwgZmlsZV9wYXRoKSAtPiBib29sOgogICAgIiIiR2VuZXJpYzogY2FjaGVkIHJ1bi1jb25maWcgcmVjb3JkcyBpbnB1dCBoYXNoZXM7IHJlc3RvcmUgb25seSB3aGVuIHRoZXkgbWF0Y2guCgogICAga2V5IGlzIHRoZSBkb3R0ZWQgcGF0aCBpbnRvIHRoZSBjb25maWcsIGUuZy4gImlucHV0cy5mZWF0dXJlc19mdWxsLnBhcnF1ZXQiCiAgICBvciAidW5pZmllZF9wYXJxdWV0X3NoYTI1NiIuIENvbmZpZyBrZXlzIHRoZW1zZWx2ZXMgbWF5IGNvbnRhaW4gZG90cwogICAgKCJmZWF0dXJlc19mdWxsLnBhcnF1ZXQiKSwgc28gYXQgZXZlcnkgbGV2ZWwgdGhlIGxvbmdlc3QgbGl0ZXJhbCByZW1haW5kZXIKICAgIGlzIHRyaWVkIGZpcnN0LiBSZXR1cm5zIEZhbHNlIG9uIGFueSBtaXNtYXRjaC9taXNzaW5nIGZpbGUuCiAgICAiIiIKICAgIGltcG9ydCBqc29uCgogICAgdHJ5OgogICAgICAgIGNmZyA9IGpzb24ubG9hZHMoUGF0aChjb25maWdfanNvbl9wYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgcGFydHMgPSBrZXkuc3BsaXQoIi4iKQogICAgICAgIG5vZGUgPSBjZmcKICAgICAgICBmb3IgaSwgcGFydCBpbiBlbnVtZXJhdGUocGFydHMpOgogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShub2RlLCBkaWN0KToKICAgICAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgICAgICByZXN0ID0gIi4iLmpvaW4ocGFydHNbaTpdKQogICAgICAgICAgICBpZiByZXN0IGluIG5vZGU6ICAjIGxpdGVyYWwga2V5IGNvbnRhaW5zIGRvdHM6IHRha2UgdGhlIHdob2xlIHJlbWFpbmRlcgogICAgICAgICAgICAgICAgbm9kZSA9IG5vZGVbcmVzdF0KICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGlmIHBhcnQgbm90IGluIG5vZGU6CiAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICAgICAgbm9kZSA9IG5vZGVbcGFydF0KICAgICAgICByZXR1cm4gbm9kZSA9PSBzaGEyNTZfZmlsZShmaWxlX3BhdGgpCiAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IsIFR5cGVFcnJvcik6CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKZGVmIGIyX3Jlc3RvcmVfdmFsaWQoYjJfY29uZmlnX3BhdGgsIGZlYXR1cmVzX3BhdGgpIC0+IGJvb2w6CiAgICAiIiJCMiBhcnRpZmFjdHMgYXJlIHJldXNhYmxlIHdoZW4gdGhlIGNhY2hlZCBydW4gY29uc3VtZWQgVEhJUyBmZWF0dXJlc19mdWxsLnBhcnF1ZXQuIiIiCiAgICByZXR1cm4gcmVzdG9yZV9jb25maWdfaGFzaChiMl9jb25maWdfcGF0aCwgImlucHV0cy5mZWF0dXJlc19mdWxsLnBhcnF1ZXQiLCBmZWF0dXJlc19wYXRoKQoKCmRlZiBiM19yZXN0b3JlX3ZhbGlkKGIzX2NvbmZpZ19wYXRoLCB1bmlmaWVkX3BhdGgsIGIyX21vZGVsc19kaXIpIC0+IGJvb2w6CiAgICAiIiJCMyByZXN1bHRzIGFyZSByZXVzYWJsZSB3aGVuIHVuaWZpZWQgcGFycXVldCBBTkQgQjIgbW9kZWxzIG1hdGNoIHRoZSBjYWNoZWQgcnVuLiIiIgogICAgaW1wb3J0IGpzb24KCiAgICB0cnk6CiAgICAgICAgY2ZnID0ganNvbi5sb2FkcyhQYXRoKGIzX2NvbmZpZ19wYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgIHJldHVybiBGYWxzZQogICAgdHJ5OgogICAgICAgIGlmIGNmZy5nZXQoInVuaWZpZWRfcGFycXVldF9zaGEyNTYiKSAhPSBzaGEyNTZfZmlsZSh1bmlmaWVkX3BhdGgpOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICBoYXNoZXMgPSBjZmcuZ2V0KCJiMl9tb2RlbF9oYXNoZXMiKSBvciB7fQogICAgICAgIHJldHVybiBhbGwoCiAgICAgICAgICAgIGhhc2hlcy5nZXQoZiJzZWVkX3tzfSIpID09IHNoYTI1Nl9maWxlKFBhdGgoYjJfbW9kZWxzX2RpcikgLyBmInhnYm9vc3Rfc2VlZF97c30uam9ibGliIikKICAgICAgICAgICAgZm9yIHMgaW4gKDQyLCAxMjMsIDQ1NikKICAgICAgICApCiAgICBleGNlcHQgT1NFcnJvcjoKICAgICAgICByZXR1cm4gRmFsc2UKCgpkZWYgYjRfcmVzdG9yZV92YWxpZChiNF9jb25maWdfcGF0aCwgYjNfcHJlZGljdGlvbnNfcGF0aCkgLT4gYm9vbDoKICAgICIiIkI0IGFydGlmYWN0cyBhcmUgcmV1c2FibGUgd2hlbiB0aGV5IHdlcmUgcHJvZHVjZWQgZnJvbSBUSElTIGIzIHByZWRpY3Rpb25zIGZpbGUuIiIiCiAgICByZXR1cm4gcmVzdG9yZV9jb25maWdfaGFzaChiNF9jb25maWdfcGF0aCwgImlucHV0cy5iM19wcmVkaWN0aW9ucy5wYXJxdWV0IiwgYjNfcHJlZGljdGlvbnNfcGF0aCkKCgpkZWYgdmVyc2lvbl9hX3Jlc3RvcmVfdmFsaWQobWFya2VyX3BhdGgsIGZlYXR1cmVzX3BhdGgsIHFhX3BhdGgsIHNwbGl0X3JlcG9ydF9wYXRoLCBjYWNoZV9kaXIpIC0+IGJvb2w6CiAgICAiIiJWYWxpZGF0ZSB0aGUgY2FjaGVkIHJvb3QgVmVyc2lvbiBBIGFydGlmYWN0cyBhZ2FpbnN0IGN1cnJlbnQgaW5wdXRzLiIiIgogICAgdHJ5OgogICAgICAgIG1hcmtlciA9IGpzb24ubG9hZHMoUGF0aChtYXJrZXJfcGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIGV4cGVjdGVkID0gewogICAgICAgICAgICAiZmVhdHVyZXNfZnVsbC5wYXJxdWV0Ijogc2hhMjU2X2ZpbGUoZmVhdHVyZXNfcGF0aCksCiAgICAgICAgICAgICJxYV9jbGVhbi5wYXJxdWV0Ijogc2hhMjU2X2ZpbGUocWFfcGF0aCksCiAgICAgICAgICAgICJzcGxpdF9pbnRlZ3JpdHlfcmVwb3J0Lmpzb24iOiBzaGEyNTZfZmlsZShzcGxpdF9yZXBvcnRfcGF0aCksCiAgICAgICAgfQogICAgICAgIGlmIG1hcmtlci5nZXQoImlucHV0cyIpICE9IGV4cGVjdGVkOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICByZXR1cm4gYWxsKChQYXRoKGNhY2hlX2RpcikgLyByZWwpLmV4aXN0cygpIGZvciByZWwgaW4gbWFya2VyLmdldCgiYXJ0aWZhY3RzIiwgW10pKQogICAgZXhjZXB0IChPU0Vycm9yLCBWYWx1ZUVycm9yLCBUeXBlRXJyb3IpOgogICAgICAgIHJldHVybiBGYWxzZQoKCmRlZiBiM19mZWF0dXJlX2NhY2hlX3NhZmUoY2FjaGVfcGF0aCkgLT4gYm9vbDoKICAgICIiIlJlamVjdCBvbGQgQjMgY2FjaGVzIHRoYXQgYWNjaWRlbnRhbGx5IGNvbnRhaW4gcmF3IGV4dGVybmFsIHRleHQuIiIiCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHBhbmRhcyBhcyBwZAoKICAgICAgICBjb2x1bW5zID0gc2V0KHBkLnJlYWRfcGFycXVldChjYWNoZV9wYXRoLCBjb2x1bW5zPU5vbmUpLmNvbHVtbnMpCiAgICAgICAgZm9yYmlkZGVuID0geyJxdWVzdGlvbiIsICJjb250ZXh0IiwgImFuc3dlciIsICJzcGFuX2Fubm90YXRpb25zIn0KICAgICAgICByZXR1cm4gbm90IChjb2x1bW5zICYgZm9yYmlkZGVuKSBhbmQgInNhbXBsZV9pZCIgaW4gY29sdW1ucwogICAgZXhjZXB0IChPU0Vycm9yLCBWYWx1ZUVycm9yLCBJbXBvcnRFcnJvcik6CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKZGVmIGIzX3Jlc3VsdHNfc2FmZShyZXN1bHRzX2RpcikgLT4gYm9vbDoKICAgICIiIlJlamVjdCBjYWNoZWQgQjMgZXJyb3IgY2FzZXMgY29udGFpbmluZyB1bnJlZGFjdGVkIEZhaXRoQmVuY2ggdGV4dC4iIiIKICAgIGltcG9ydCBqc29uCgogICAgZXJyb3JfcGF0aCA9IFBhdGgocmVzdWx0c19kaXIpIC8gImIzX2Vycm9yX2Nhc2VzLmpzb24iCiAgICBpZiBub3QgZXJyb3JfcGF0aC5leGlzdHMoKToKICAgICAgICByZXR1cm4gVHJ1ZQogICAgdHJ5OgogICAgICAgIGNhc2VzID0ganNvbi5sb2FkcyhlcnJvcl9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICBmb3IgY2FzZSBpbiBjYXNlczoKICAgICAgICAgICAgaWYgY2FzZS5nZXQoInNvdXJjZV9kYXRhc2V0IikgPT0gImZhaXRoYmVuY2giOgogICAgICAgICAgICAgICAgaWYgYW55KGNhc2UuZ2V0KGZpZWxkLCAiIikgZm9yIGZpZWxkIGluICgicXVlc3Rpb24iLCAiY29udGV4dCIsICJhbnN3ZXIiLCAic3Bhbl9hbm5vdGF0aW9ucyIpKToKICAgICAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICByZXR1cm4gVHJ1ZQogICAgZXhjZXB0IChPU0Vycm9yLCBWYWx1ZUVycm9yLCBUeXBlRXJyb3IpOgogICAgICAgIHJldHVybiBGYWxzZQoKCmRlZiBiM19wcmVkaWN0aW9uc19zYWZlKHByZWRpY3Rpb25zX3BhdGgpIC0+IGJvb2w6CiAgICAiIiJSZWplY3QgQjMgcHJlZGljdGlvbiBjYWNoZXMgcG9sbHV0ZWQgYnkgb3ZlcmxhcHBpbmcgc3Vic2V0IGFwcGVuZHMuIiIiCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHBhbmRhcyBhcyBwZAoKICAgICAgICBwcmVkcyA9IHBkLnJlYWRfcGFycXVldChwcmVkaWN0aW9uc19wYXRoLCBjb2x1bW5zPVsic2FtcGxlX2lkIiwgIm1vZGVsIl0pCiAgICAgICAgZXhwZWN0ZWQgPSB7InhnYm9vc3Rfc2VlZF80MiIsICJ4Z2Jvb3N0X3NlZWRfMTIzIiwgInhnYm9vc3Rfc2VlZF80NTYifQogICAgICAgIHJldHVybiAoCiAgICAgICAgICAgIGxlbihwcmVkcykgPiAwCiAgICAgICAgICAgIGFuZCBzZXQocHJlZHNbIm1vZGVsIl0uZHJvcG5hKCkudW5pcXVlKCkpID09IGV4cGVjdGVkCiAgICAgICAgICAgIGFuZCBub3QgcHJlZHMuZHVwbGljYXRlZChbInNhbXBsZV9pZCIsICJtb2RlbCJdKS5hbnkoKQogICAgICAgICkKICAgIGV4Y2VwdCAoT1NFcnJvciwgVmFsdWVFcnJvciwgSW1wb3J0RXJyb3IsIEtleUVycm9yKToKICAgICAgICByZXR1cm4gRmFsc2UKCgpkZWYgYjRfcHJlZGljdGlvbnNfc2FmZShwcmVkaWN0aW9uc19wYXRoKSAtPiBib29sOgogICAgIiIiUmVqZWN0IEI0IHByZWRpY3Rpb24gY2FjaGVzIHdpdGggZHVwbGljYXRlIHNhbXBsZS9tZXRob2Qgcm93cy4iIiIKICAgIHRyeToKICAgICAgICBpbXBvcnQgcGFuZGFzIGFzIHBkCgogICAgICAgIHByZWRzID0gcGQucmVhZF9wYXJxdWV0KAogICAgICAgICAgICBwcmVkaWN0aW9uc19wYXRoLAogICAgICAgICAgICBjb2x1bW5zPVsic2FtcGxlX2lkIiwgInNvdXJjZV9kYXRhc2V0IiwgInN1YnNldCIsICJtZXRob2QiXSwKICAgICAgICApCiAgICAgICAga2V5ID0gWyJzYW1wbGVfaWQiLCAic291cmNlX2RhdGFzZXQiLCAic3Vic2V0IiwgIm1ldGhvZCJdCiAgICAgICAgcmV0dXJuIGxlbihwcmVkcykgPiAwIGFuZCBub3QgcHJlZHMuZHVwbGljYXRlZChrZXkpLmFueSgpCiAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IsIEltcG9ydEVycm9yLCBLZXlFcnJvcik6CiAgICAgICAgcmV0dXJuIEZhbHNlCg==",
 "colab/requirements-colab.txt": "IyBIYWx1UklTQyDigJQgQ29sYWIgZGVwZW5kZW5jaWVzIChubyB0b3JjaDogQ29sYWIgc2hpcHMgaXRzIG93biBHUFUgdG9yY2gpDQojIFBpbiBldmVyeXRoaW5nIGVsc2UgaWRlbnRpY2FsbHkgdG8gcmVxdWlyZW1lbnRzLnR4dCAoMjAyNi0wOC0wNCkNCg0KbnVtcHk9PTIuNC42DQpwYW5kYXM9PTMuMC41DQpweWFycm93PT0yNS4wLjANCmpvYmxpYj09MS41LjMNCnNjaWtpdC1sZWFybj09MS45LjANCnhnYm9vc3Q9PTMuMy4wDQpzY2lweT09MS4xNy4xDQpzdGF0c21vZGVscz09MC4xNC42DQpzaGFwPT0wLjUyLjANCnRyYW5zZm9ybWVycz09NS4xNC4xDQpzZW50ZW5jZS10cmFuc2Zvcm1lcnM9PTUuNi4xDQpzcGFjeT09My44LjE0DQpkYXRhc2V0cz09NS4wLjENCmh1Z2dpbmdmYWNlLWh1Yj09MS4yNi4wDQptYXRwbG90bGliPT0zLjExLjENCm9wZW5haT09Mi41My4wDQpweXRob24tZG90ZW52PT0xLjIuMg0K"
}
HASHES = {
 "src/__init__.py": "2e015d29f635fda3f7a3758a283191cc0203f3c913bdebe133d4b04f8b9604b9",
 "src/api/main.py": "32cb1935c02d423bf83f84f47438ed4fde62617e88f6ca3f023783b17892e1ba",
 "src/data/download.py": "cf630f31ae4ea17eb19b3338eb65f1dec04e07222936071b4954d29a6ea7515b",
 "src/data/download_faithbench.py": "a71b43e84e0785639046bfb4224e429d448bfaeb713d3919a1945b4ecf0a4118",
 "src/data/download_ragtruth.py": "fde317a9a3d339162f772ad01c1fcc0da91f286d83c4afacf970331aa7177806",
 "src/data/mappings.py": "49b4813f288e74f792e853baf635bdf2b02b48f2018a6a722b178bee10a7d00c",
 "src/data/prepare.py": "53fe3cf7052b363f3711570936a2bd7552340931f03814ea680e6abeb27a059e",
 "src/data/prepare_unified.py": "a4b2478a20e5e05c31ccb7fcd745e07d1b4dd14312e3e14598649b2aaa6ce02a",
 "src/data/registry.py": "ead69514382cdc4d90d499689dab5f9b54250eb7a4f87271a4bda6a7ba44a25a",
 "src/data/schema.py": "931a2d0d8ed6108b3c5af1db0d7e81ce6fe9bfa5b39dbc842aa71d691c12aac3",
 "src/explain/shap_analysis.py": "cd77e6842d70a4edb45d5d793be6e17616d5657d8a2ecef26d1e05c2f6e93868",
 "src/features/entity_features.py": "42fba0888bea392b81fc3cf2c422a0540d5317ce7210c52f4283cf4c10b9f997",
 "src/features/extract_features.py": "18297e7d802986ab12879e16453b18e1dbe1cfe91279b65f65dbc177986a700d",
 "src/features/nli_features.py": "19c0c2c36289b79456d21f001fddbd008f0c6c7ebff9212ac388e09ba8074a1a",
 "src/features/semantic_features.py": "f899c9f77894d568a492e996c9e79412d44a8200854017cb6853e3ae43ca4394",
 "src/models/config.py": "0ba14d3221f0d4173310927f1037eb680754751f30b001c7b0f5c58d13fb66f5",
 "src/models/error_analysis.py": "8420df6b58dba27b7264bf86e75dd6805399afb021e18c1622a381f0cd180636",
 "src/models/eval_efficiency.py": "97c27458daaa297dfd2a6f2e0a1258577b9ed31be737d5e4957291729fc3baf0",
 "src/models/eval_llm_judge.py": "c62397a7b9667416027d50ba18fae73cb32416bb70f9ac7b75fda47726f5e714",
 "src/models/eval_ragtruth.py": "88191412e6f3c6e60a0310ab2e0e759cf9383a3b5872fd1bd62e109c56795a96",
 "src/models/make_manifest.py": "e23075313e541a8c8e0dc03f49fbe813cb3616ecd8e059e685850439f8f14763",
 "src/models/run_b2_baselines.py": "d2f0958740564f97221a6ca09aa60d7ff4aefb53a9b147d61e86e40b34922f7c",
 "src/models/run_b3_cross_domain.py": "5a65e4f4497f921cb3970b600293c13545e89b45b2a96eb18e4c03b08bf239d6",
 "src/models/run_b4_calibration_shift.py": "d1c2845a7a16a65fc5271c32d2124aaa4b2f858d64fa217d28e98156ecb87478",
 "src/models/run_b5_explanation_reliability.py": "bb17dbf6a37d8a1029dc5114f4fd6248f28ce4e8aa5ce39172b199bac8428025",
 "src/models/train_baselines.py": "81099aee9f3d676e576572047a940ee6531ff696a4117e80081c3a57cae3ec89",
 "src/models/train_pipeline.py": "bb5db2235040426f82641acf85ffd9ace3703c92b2d7ddf6e5e456f5d810c8fe",
 "src/models/verify_artifacts.py": "dd552460e5b9dbd8d55cee4e12b223dd4a04836825e0d42b33a23d22a98cb231",
 "colab/drive_cache.py": "d0b743c0e437ac0f3633342422713d5bb2acb507b778dfba938c5a42a97578be",
 "colab/requirements-colab.txt": "490e686ba8c51e4f734d2e8b053dd6aeafeeb334587cd0cc3d4310c8d056fae9"
}

for rel, b64 in EMBEDDED.items():
    dest = os.path.join(ROOT, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with open(dest, 'wb') as f:
        f.write(base64.b64decode(b64))

bad = [rel for rel in EMBEDDED
       if hashlib.sha256(open(os.path.join(ROOT, rel), 'rb').read()).hexdigest() != HASHES[rel]]
assert not bad, f'embedded source mismatch: {bad}'
os.chdir(ROOT)
print(f'Self-contained source ready: {len(EMBEDDED)} files in {ROOT}')
print('src present:', os.path.exists(os.path.join(ROOT, 'src')))


In [ ]:
# 4) Install pinned dependencies (Colab keeps its own torch)
!pip install -q -r colab/requirements-colab.txt
# Colab preinstalls a newer numpy that removed numpy._core.umath._center,
# which the xgboost 3.3.0 / scipy 1.17.1 wheels need at import; force the
# pinned numpy if the preinstalled one lacks it (jax etc. are never used).
import numpy
try:
    from numpy._core.umath import _center
except ImportError:
    !pip install -q --force-reinstall --no-deps numpy==2.4.6
    from numpy._core.umath import _center  # needs a fresh kernel if this fails
!python -m spacy download en_core_web_sm -q
print('deps OK')


In [ ]:
# 5) HaluEval download + prepare with GROUP-AWARE split (item_idx) + integrity check
!python src/data/download.py
!python src/data/prepare.py
import json
rep = json.load(open('artifacts/split_integrity_report.json'))
print(json.dumps(rep, indent=2))
assert rep['leakage_free'] and rep['groups_spanning_multiple_splits'] == 0, 'Split leakage detected!'


In [ ]:
# 5b) Restore cached HaluEval features from Drive when valid (cell 6 shortcut)
# Saves 5-10 min on repeat runs. The cache is VERIFIED against the freshly built
# qa_clean.parquet (rows / sample_ids / labels / splits / leakage) - a mismatch
# falls back to full extraction in cell 6. Set CACHE_OK = True on success.
import os, sys, json, shutil
sys.path.insert(0, '.')
from colab.drive_cache import verify_halueval_features

DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
cache_feat = os.path.join(DRIVE_CACHE, 'features_full.parquet')
CACHE_OK = False
if os.path.exists(cache_feat):
    tmp = '/content/_cache_features.parquet'
    shutil.copy(cache_feat, tmp)
    v = verify_halueval_features(tmp, 'data/processed/qa_clean.parquet')
    if v['ok']:
        shutil.move(tmp, 'data/processed/features_full.parquet')
        cache_nli = os.path.join(DRIVE_CACHE, 'nli_model_used.json')
        if os.path.exists(cache_nli):
            shutil.copy(cache_nli, 'data/processed/nli_model_used.json')
        CACHE_OK = True
        print('FEATURE CACHE RESTORED from Drive and verified against qa_clean.')
    else:
        print('Cached features INVALID - cell 6 will re-extract:', v['checks'])
        os.remove(tmp)
else:
    print('No HaluEval feature cache on Drive yet - cell 6 will extract.')


In [ ]:
# 6) Full feature extraction (7 groups, ~40K NLI pairs on GPU; ~5-10 min)
# Skipped automatically when cell 5b restored a verified Drive cache.
# --device cuda -> models load in fp16 + CUDA; batch size from cell 2.
# Default NLI model: cross-encoder/nli-deberta-v3-base. Fallback: --nli-model cross-encoder/nli-MiniLM2-L6-H768
import json, os
if CACHE_OK:
    print('Using Drive-cached features (verified against qa_clean.parquet).')
    print('NLI provenance:', json.load(open('data/processed/nli_model_used.json')))
else:
    os.system(f'python src/features/extract_features.py --device cuda --batch-size {BATCH_SIZE}')
    print('NLI provenance:', json.load(open('data/processed/nli_model_used.json')))


In [ ]:
# 6b) Upload the HaluEval feature cache to Drive (next session skips extraction)
# Safe: features_full.parquet contains no FaithBench text (HaluEval is MIT).
import os, shutil
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
shutil.copy('data/processed/features_full.parquet', os.path.join(DRIVE_CACHE, 'features_full.parquet'))
shutil.copy('data/processed/nli_model_used.json', os.path.join(DRIVE_CACHE, 'nli_model_used.json'))
print('Uploaded HaluEval feature cache to Drive (halurisc_cache/).')


In [ ]:
# 7.0) Restore Version A cell-7 artifacts (LOCAL first, then Drive)
import os, json, shutil
from colab.drive_cache import version_a_restore_valid, sha256_file

DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
VA_DIR = os.path.join(DRIVE_CACHE, 'version_a')
VA_MARKER = os.path.join(VA_DIR, 'marker.json')
VA_FILES = [
    ('models/model_xgboost_raw.joblib', 'artifacts/models/model_xgboost_raw.joblib'),
    ('models/model_xgboost_calibrated.joblib', 'artifacts/models/model_xgboost_calibrated.joblib'),
    ('models/calibrator_platt.joblib', 'artifacts/models/calibrator_platt.joblib'),
    ('models/scaler.joblib', 'artifacts/models/scaler.joblib'),
    ('models/params.json', 'artifacts/models/params.json'),
    ('models/feature_names.json', 'artifacts/models/feature_names.json'),
    ('results/final_results.json', 'artifacts/results/final_results.json'),
    ('results/seed_metrics.json', 'artifacts/results/seed_metrics.json'),
    ('results/model_comparison.csv', 'artifacts/results/model_comparison.csv'),
    ('results/ablation_results.csv', 'artifacts/results/ablation_results.csv'),
]
VA_OK = False
if os.path.exists('artifacts/results/final_results.json'):
    VA_OK = True
    print('Version A cell-7 artifacts already present locally - skipping.')
elif os.path.exists(VA_MARKER) and version_a_restore_valid(
    VA_MARKER, 'data/processed/features_full.parquet', 'data/processed/qa_clean.parquet',
    'artifacts/split_integrity_report.json', VA_DIR
):
    for rel, local in VA_FILES:
        os.makedirs(os.path.dirname(local), exist_ok=True)
        shutil.copy(os.path.join(VA_DIR, rel), local)
    VA_OK = True
    print('Version A cell-7 artifacts RESTORED from Drive (input hashes verified).')
else:
    print('No Version A cache found - cell 7 will run if selected.')


In [ ]:
# 7) Full Version A experiment (optional; CPU, ~10-20 min)
# HALU_XGB_DEVICE=cpu keeps boosters portable after download.
# RESUMABLE: restores from 7.0 or checkpoints immediately after success.
import datetime, json, os, shutil
os.environ['HALU_XGB_DEVICE'] = 'cpu'
if VA_OK:
    print('Version A cell 7 already complete - skipping training.')
else:
    rc = os.system('python src/models/train_pipeline.py')
    if rc != 0:
        raise RuntimeError(f'Version A cell 7 failed with exit code {rc}')
    os.makedirs(VA_DIR, exist_ok=True)
    for rel, local in VA_FILES:
        dest = os.path.join(VA_DIR, rel)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy(local, dest)
    marker = {
        'schema': 'version-a-checkpoint-v1',
        'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'inputs': {
            'features_full.parquet': sha256_file('data/processed/features_full.parquet'),
            'qa_clean.parquet': sha256_file('data/processed/qa_clean.parquet'),
            'split_integrity_report.json': sha256_file('artifacts/split_integrity_report.json'),
        },
        'artifacts': [rel for rel, _ in VA_FILES],
    }
    with open(VA_MARKER, 'w', encoding='utf-8') as f:
        json.dump(marker, f, indent=2)
    print('Checkpointed Version A cell-7 artifacts to Drive (halurisc_cache/version_a).')


In [ ]:
# 7b.0) Restore B2 artifacts (LOCAL first, then Drive; 7b shortcut)
import os, json, shutil, hashlib
from colab.drive_cache import b2_restore_valid
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
B2_OK = False
B2_FORCE_RESTORE = False  # set True to copy the Drive cache even if the hash differs (prints a warning)

def b2_why_not(cfg_path, parquet_path):
    if not os.path.exists(cfg_path):
        return 'config missing: ' + cfg_path
    if not os.path.isdir(os.path.join(DRIVE_CACHE, 'b2_models')):
        return 'models dir missing: ' + os.path.join(DRIVE_CACHE, 'b2_models')
    if not os.path.exists(parquet_path):
        return 'local features missing: ' + parquet_path
    rec = json.load(open(cfg_path)).get('inputs', {}).get('features_full.parquet', '')
    cur = hashlib.sha256(open(parquet_path, 'rb').read()).hexdigest()
    return 'hash mismatch: config %s.. vs local %s..' % (rec[:16], cur[:16])

if (os.path.exists('artifacts/results/b2/b2_run_config.json')
        and os.path.isdir('artifacts/models/b2')
        and b2_restore_valid('artifacts/results/b2/b2_run_config.json', 'data/processed/features_full.parquet')):
    B2_OK = True
    print('B2 artifacts already present locally (hash verified) - skipping training.')
elif B2_FORCE_RESTORE:
    shutil.copytree(os.path.join(DRIVE_CACHE, 'b2'), 'artifacts/results/b2', dirs_exist_ok=True)
    shutil.copytree(os.path.join(DRIVE_CACHE, 'b2_models'), 'artifacts/models/b2', dirs_exist_ok=True)
    B2_OK = True
    print('WARNING: B2 artifacts FORCE-restored from Drive WITHOUT hash verification.')
else:
    b2_dir = os.path.join(DRIVE_CACHE, 'b2')
    cfg = os.path.join(b2_dir, 'b2_run_config.json')
    if (os.path.exists(cfg) and os.path.isdir(os.path.join(DRIVE_CACHE, 'b2_models'))
            and b2_restore_valid(cfg, 'data/processed/features_full.parquet')):
        shutil.copytree(b2_dir, 'artifacts/results/b2', dirs_exist_ok=True)
        shutil.copytree(os.path.join(DRIVE_CACHE, 'b2_models'), 'artifacts/models/b2', dirs_exist_ok=True)
        B2_OK = True
        print('B2 artifacts RESTORED from Drive (feature hash verified).')
    else:
        print('No valid B2 cache found:', b2_why_not(cfg, 'data/processed/features_full.parquet'))
        print('Cell 7b will train (8-15 min).')


In [ ]:
# 7b) B2: corrected baselines + artifact controls (grouped CV, TF-IDF shortcut checks)
# Runs: majority, overlap heuristic, TF-IDF (all/answer/context), NLI-only, LR, RF, tuned XGBoost
# seeds 42/123/456. Grouped 5-fold CV keyed by item_idx; thresholds: 0.5 / overlap tuned on val.
# ~5-10 min on L4. All outputs under artifacts/{results,models}/b2 (Version A untouched).
# RESUMABLE: skipped automatically when cell 7b.0 restored valid artifacts (B2_OK).
import os
os.environ['HALU_XGB_DEVICE'] = 'cpu'  # portable boosters, see cell 7 note
if B2_OK:
    print('B2 already complete (restored from Drive) - skipping training.')
else:
    rc = os.system('python src/models/run_b2_baselines.py')
    if rc != 0:
        raise RuntimeError(f'B2 run failed with exit code {rc}')


In [ ]:
# 7b.5) Upload B2 artifacts to Drive (crash-safe checkpoint)
import os, shutil
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(os.path.join(DRIVE_CACHE, 'b2'), exist_ok=True)
shutil.copytree('artifacts/results/b2', os.path.join(DRIVE_CACHE, 'b2'), dirs_exist_ok=True)
shutil.copytree('artifacts/models/b2', os.path.join(DRIVE_CACHE, 'b2_models'), dirs_exist_ok=True)
print('Checkpointed B2 artifacts to Drive (halurisc_cache/b2).')


In [ ]:
# 7c) Show B2 headline results (comparison table + leakage-removal impact)
import json, pandas as pd
comp = pd.read_csv('artifacts/results/b2/b2_model_comparison.csv', index_col=0)
cols = [c for c in ['precision_mean','recall_mean','f1_mean','auroc_mean','pr_auc_mean','mcc_mean','ece_mean','threshold'] if c in comp.columns]
print('B2 MODEL COMPARISON (test set, seeds 42/123/456)')
print(comp[cols].round(4).to_string())
print('\nLEAKAGE-REMOVAL IMPACT:')
print(json.dumps(json.load(open('artifacts/results/b2/b2_leakage_comparison.json')), indent=2))


In [ ]:
# 7d) B1: build the unified external dataset (official RAGTruth + FaithBench)
# Required by B3. Downloads RAGTruth response/source files (~35 MB) and FaithBench
# annotation batches (~3 MB). Raw files stay in the Colab VM (never committed;
# FaithBench is CC BY-NC-SA). Deterministic: rerunning gives byte-identical output.
!python src/data/download_ragtruth.py
!python src/data/download_faithbench.py
!python src/data/prepare_unified.py


In [ ]:
# 7d.5) B3 external-feature cache (LOCAL first, then Drive; 7e shortcut)
import os, json, shutil, hashlib
from colab.drive_cache import b3_feature_cache_safe
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')

def unified_sha():
    return hashlib.sha256(open('data/processed/unified_records.parquet', 'rb').read()).hexdigest()

B3_CACHE_OK = False
local_meta = 'data/processed/b3_external_features.meta.json'
local_cache = 'data/processed/b3_external_features.parquet'
if (os.path.exists(local_meta) and os.path.exists(local_cache)
        and json.load(open(local_meta)).get('input_sha256') == unified_sha()
        and json.load(open(local_meta)).get('complete', True)
        and b3_feature_cache_safe(local_cache)):
    B3_CACHE_OK = True
    print('B3 external feature cache already present locally (complete, hash verified).')
else:
    cache_meta = os.path.join(DRIVE_CACHE, 'b3_external_features.meta.json')
    cache_features = os.path.join(DRIVE_CACHE, 'b3_external_features.parquet')
    if (os.path.exists(cache_meta) and os.path.exists(cache_features)
            and b3_feature_cache_safe(cache_features)):
        meta = json.load(open(cache_meta))
        if meta.get('input_sha256') == unified_sha():
            shutil.copy(cache_features, local_cache)
            shutil.copy(cache_meta, local_meta)
            B3_CACHE_OK = bool(meta.get('complete', True)) and b3_feature_cache_safe(cache_features)
            if B3_CACHE_OK:
                print('B3 external feature cache restored from Drive (complete, hash verified).')
            else:
                print('Partial B3 feature cache restored from Drive - cell 7e RESUMES extraction.')
        else:
            print('B3 cache stale (unified parquet changed) - cell 7e will re-extract.')
    else:
        print('No B3 external feature cache found - cell 7e will extract.')


In [ ]:
# 7d.6) B3 RESULTS (LOCAL first, then Drive; 7e shortcut)
import os, json, shutil
from colab.drive_cache import b3_restore_valid, b3_results_safe, b3_predictions_safe
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
B3_OK = False
if (os.path.exists('artifacts/results/b3/b3_run_config.json')
        and os.path.exists('artifacts/results/b3/b3_predictions.parquet')
        and b3_predictions_safe('artifacts/results/b3/b3_predictions.parquet')
        and b3_restore_valid('artifacts/results/b3/b3_run_config.json',
                             'data/processed/unified_records.parquet', 'artifacts/models/b2')):
    B3_OK = True
    print('B3 results already present locally (unified + B2 model hashes verified).')
else:
    b3_dir = os.path.join(DRIVE_CACHE, 'b3')
    figures_dir = os.path.join(DRIVE_CACHE, 'b3_figures')
    cfg = os.path.join(b3_dir, 'b3_run_config.json')
    if (os.path.exists(cfg) and os.path.exists(os.path.join(b3_dir, 'b3_predictions.parquet'))
            and os.path.isdir(figures_dir) and b3_results_safe(b3_dir)
            and b3_predictions_safe(os.path.join(b3_dir, 'b3_predictions.parquet'))
            and b3_restore_valid(cfg, 'data/processed/unified_records.parquet', 'artifacts/models/b2')):
        shutil.copytree(b3_dir, 'artifacts/results/b3', dirs_exist_ok=True)
        shutil.copytree(figures_dir, 'artifacts/figures/b3', dirs_exist_ok=True)
        B3_OK = True
        print('B3 results RESTORED from Drive (unified + B2 model hashes verified).')
    else:
        print('No valid B3 results cache found - cell 7e will run.')


In [ ]:
# 7e) B3: cross-domain zero-shot evaluation (HEAVY: ~18.5K external rows)
# Extracts the 26 features on RAGTruth + FaithBench, then evaluates the B2
# XGBoost models zero-shot (fixed threshold 0.5, source-group bootstrap CIs,
# subgroup + transfer-failure analysis, FaithBench label sensitivity).
# RESUMABLE: skipped when cell 7d.6 restored results (B3_OK); the runner
# reuses a complete local feature cache automatically (self-healing) or
# resumes a partial one. Feature cache uploads to Drive FIRST (most valuable).
import os, shutil
os.environ['HALU_XGB_DEVICE'] = 'cpu'
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
if B3_OK:
    print('B3 already complete (results present) - skipping run.')
else:
    flag = '--skip-features' if B3_CACHE_OK else ''
    cmd = f'python src/models/run_b3_cross_domain.py --device cuda --batch-size {BATCH_SIZE} {flag}'.strip()
    print('Running:', cmd)
    rc = os.system(cmd)
    if rc != 0:
        cache_src = 'data/processed/b3_external_features.parquet'
        if os.path.exists(cache_src):
            shutil.copy(cache_src, os.path.join(DRIVE_CACHE, 'b3_external_features.parquet'))
            shutil.copy('data/processed/b3_external_features.meta.json', os.path.join(DRIVE_CACHE, 'b3_external_features.meta.json'))
            print('B3 failed after feature cache creation; feature cache checkpointed.')
        raise RuntimeError(f'B3 run failed with exit code {rc}')
    # feature cache FIRST (most valuable), then results, then figures
    if os.path.exists('data/processed/b3_external_features.parquet'):
        shutil.copy('data/processed/b3_external_features.parquet', os.path.join(DRIVE_CACHE, 'b3_external_features.parquet'))
        shutil.copy('data/processed/b3_external_features.meta.json', os.path.join(DRIVE_CACHE, 'b3_external_features.meta.json'))
        print('Checkpointed B3 external feature cache to Drive.')
    shutil.copytree('artifacts/results/b3', os.path.join(DRIVE_CACHE, 'b3'), dirs_exist_ok=True)
    print('Checkpointed B3 results to Drive (halurisc_cache/b3).')
    shutil.copytree('artifacts/figures/b3', os.path.join(DRIVE_CACHE, 'b3_figures'), dirs_exist_ok=True)
    print('Checkpointed B3 figures to Drive (halurisc_cache/b3_figures).')


In [ ]:
# 7f) Show B3 headline results (dataset metrics + transfer comparison)
import json, pandas as pd
m = json.load(open('artifacts/results/b3/b3_dataset_metrics.json'))
keep = ('n_rows','n_groups','f1_mean','auroc_mean','ece_mean','predicted_positive_rate','label_positive_rate')
rows = {k: {kk: (round(vv,4) if isinstance(vv,float) else vv) for kk,vv in v.items() if kk in keep} for k,v in m.items() if v}
print('B3 DATASET METRICS (zero-shot, B2 XGBoost, seeds 42/123/456)')
print(pd.DataFrame(rows).T.to_string())
print('\nTRANSFER COMPARISON:')
print(pd.read_csv('artifacts/results/b3/b3_transfer_comparison.csv').to_string(index=False))
print('\nFAITHBENCH LABEL SENSITIVITY:')
print(json.dumps(json.load(open('artifacts/results/b3/b3_label_sensitivity.json')), indent=2))


In [ ]:
# 7g.0) B4 artifacts (LOCAL first, then Drive; 7g shortcut)
import os, json, shutil
from colab.drive_cache import b4_restore_valid, b4_predictions_safe
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
B4_OK = False
if (os.path.exists('artifacts/results/b4/b4_run_config.json')
        and os.path.exists('artifacts/results/b4/b4_predictions.parquet')
        and b4_predictions_safe('artifacts/results/b4/b4_predictions.parquet')
        and b4_restore_valid('artifacts/results/b4/b4_run_config.json',
                             'artifacts/results/b3/b3_predictions.parquet')):
    B4_OK = True
    print('B4 artifacts already present locally (b3 predictions hash verified).')
else:
    b4_dir = os.path.join(DRIVE_CACHE, 'b4')
    cfg = os.path.join(b4_dir, 'b4_run_config.json')
    models_dir = os.path.join(DRIVE_CACHE, 'b4_models')
    figures_dir = os.path.join(DRIVE_CACHE, 'b4_figures')
    if (os.path.exists(cfg) and os.path.exists(os.path.join(b4_dir, 'b4_predictions.parquet'))
            and os.path.isdir(models_dir) and os.path.isdir(figures_dir)
            and b4_predictions_safe(os.path.join(b4_dir, 'b4_predictions.parquet'))
            and b4_restore_valid(cfg, 'artifacts/results/b3/b3_predictions.parquet')):
        shutil.copytree(b4_dir, 'artifacts/results/b4', dirs_exist_ok=True)
        shutil.copytree(models_dir, 'artifacts/models/b4', dirs_exist_ok=True)
        shutil.copytree(figures_dir, 'artifacts/figures/b4', dirs_exist_ok=True)
        B4_OK = True
        print('B4 artifacts RESTORED from Drive (b3 predictions hash verified).')
    else:
        print('No valid B4 cache found - cell 7g will run.')


In [ ]:
# 7g) B4: calibration under distribution shift
# Reuses B3 predictions, cached external features, and B2 models (NO new
# feature extraction, NO retraining). Fits calibrators ONLY on HaluEval
# validation (source) and RAGTruth QA train (target, disjoint source
# groups), then evaluates on HaluEval test / RAGTruth QA test / FaithBench.
# Fast on CPU (2-5 min). All B4 calibrators are pure sklearn -> portable.
# CRASH-SAFE: heavy stages checkpoint to artifacts/results/b4/_stages/ and
# the runner resumes automatically after a kernel kill (OOM/quota) - just
# re-run this cell, it continues instead of restarting. On a Python crash a
# b4_crash.log is written with the full traceback.
import os, shutil
os.environ['HALU_XGB_DEVICE'] = 'cpu'
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
if B4_OK:
    print('B4 already complete (artifacts restored from Drive) - skipping run.')
else:
    rc = os.system('python src/models/run_b4_calibration_shift.py')
    if rc != 0:
        crash = 'artifacts/results/b4/b4_crash.log'
        if os.path.exists(crash):
            shutil.copy(crash, os.path.join(DRIVE_CACHE, 'b4_crash.log'))
            print('B4 crash log checkpointed to Drive.')
        raise RuntimeError(f'B4 run failed with exit code {rc}')
    shutil.copytree('artifacts/results/b4', os.path.join(DRIVE_CACHE, 'b4'), dirs_exist_ok=True)
    shutil.copytree('artifacts/models/b4', os.path.join(DRIVE_CACHE, 'b4_models'), dirs_exist_ok=True)
    shutil.copytree('artifacts/figures/b4', os.path.join(DRIVE_CACHE, 'b4_figures'), dirs_exist_ok=True)
    print('Checkpointed B4 artifacts to Drive (halurisc_cache/b4).')


In [ ]:
# 7h) Show B4 headline results (calibration metrics + target calibration)
import json, pandas as pd
m = json.load(open('artifacts/results/b4/b4_calibration_metrics.json'))
rows = []
for subset, methods in m.items():
    for method, mm in methods.items():
        rows.append({'subset': subset, 'method': method,
                     'ece': mm.get('ece_mean'), 'ace': mm.get('ace_mean'),
                     'brier': mm.get('brier_mean'), 'nll': mm.get('nll_mean'),
                     'slope': mm.get('slope_mean'), 'f1': mm.get('f1_mean')})
df = pd.DataFrame(rows).dropna(subset=['ece']).round(4)
print('B4 CALIBRATION METRICS (seeds 42/123/456; calibrators on HaluEval val only)')
print(df.to_string(index=False))
print('\nTARGET CALIBRATION (RAGTruth QA train -> test):')
t = json.load(open('artifacts/results/b4/b4_target_calibration.json'))
print('n_cal=%d n_test=%d overlap_groups_removed=%d' % (t['n_calibration_rows'], t['n_test_rows'], t['overlapping_groups_removed']))
for k, v in t['methods'].items():
    if 'ece_mean' in v:
        print('  %s: ece=%.4f brier=%.4f' % (k, v['ece_mean'], v['brier_mean']))


In [ ]:
# 7i) Verify every artifact loads on THIS machine BEFORE packaging
# Loads the B2 boosters, B4 calibrators, and all prediction parquet files;
# proves they predict correctly (prevents the old post-download loading error).
# Exit code 0 = safe to package and use locally after download.
import sys
sys.path.insert(0, os.getcwd())
from src.models.verify_artifacts import main
status = main()
if status != 0:
    raise RuntimeError(f'artifact verification failed with exit code {status}')


In [ ]:
# 7j.0) B5 artifacts (LOCAL first, then Drive; 7j shortcut)
import os, json, shutil
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
B5_OK = False
if (os.path.exists('artifacts/results/b5/b5_run_config.json')
        and os.path.exists('artifacts/results/b5/b5_feature_importance.json')):
    B5_OK = True
    print('B5 artifacts already present locally - skipping run.')
else:
    b5_dir = os.path.join(DRIVE_CACHE, 'b5')
    cfg = os.path.join(b5_dir, 'b5_run_config.json')
    if os.path.exists(cfg) and os.path.exists(os.path.join(b5_dir, 'b5_feature_importance.json')):
        shutil.copytree(b5_dir, 'artifacts/results/b5', dirs_exist_ok=True)
        shutil.copytree(os.path.join(DRIVE_CACHE, 'b5_figures'), 'artifacts/figures/b5', dirs_exist_ok=True)
        B5_OK = True
        print('B5 artifacts RESTORED from Drive.')
    else:
        print('No valid B5 cache found - cell 7j will run.')


In [ ]:
# 7j) B5: explanation reliability + error analysis
# SHAP vs permutation vs group ablation, top-k neutralization, text
# perturbations with FULL feature re-extraction, bootstrap CIs, and a
# 40-case reviewer export (10 FP / 10 FN / 20 borderline). CPU-ok;
# pass --device cuda for the perturbation re-extraction speedup.
# CRASH-SAFE: per-sample checkpoint (b5_perturbations.csv), rerun resumes.
import os, shutil
os.environ['HALU_XGB_DEVICE'] = 'cpu'
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
if B5_OK:
    print('B5 already complete - skipping run.')
else:
    rc = os.system('python src/models/run_b5_explanation_reliability.py --device cuda')
    if rc != 0:
        raise RuntimeError(f'B5 run failed with exit code {rc}')
    shutil.copytree('artifacts/results/b5', os.path.join(DRIVE_CACHE, 'b5'), dirs_exist_ok=True)
    shutil.copytree('artifacts/figures/b5', os.path.join(DRIVE_CACHE, 'b5_figures'), dirs_exist_ok=True)
    print('Checkpointed B5 artifacts to Drive (halurisc_cache/b5).')


### CRASH RECOVERY (resume from Drive)

If the runtime dies mid-run, restart it and run cells **1-5**, **5b**, **6/6b**, then **7.0** and **7** if you want Version A. Completed phases restore automatically:

- **7.0** restores Version A cell-7 artifacts; cell **7** skips training.
- **7b.0** restores B2; cell **7b** skips training.
- **7d.5** restores B3 features; **7d.6** restores B3 results; cell **7e** skips extraction/evaluation.
- **7g.0** restores B4; cell **7g** skips calibration.

Continue from the first cell whose checkpoint is missing. Every completed heavy phase is written to `Drive/halurisc_cache/` immediately.


In [ ]:
# 8.0) Restore legacy Version A analyses (cells 8-12) from Drive
# Cell 7 cached the core Version A artifacts; this restores the auxiliary
# analyses (SHAP, RAGTruth, error, latency, optional LLM judge).
import os, json, shutil
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
LEGACY_DIR = os.path.join(DRIVE_CACHE, 'version_a', 'legacy')
LEGACY_FILES = [
    ('models/shap_explainer.joblib', 'artifacts/models/shap_explainer.joblib'),
    ('results/shap_summary.json', 'artifacts/results/shap_summary.json'),
    ('results/ragtruth_results.json', 'artifacts/results/ragtruth_results.json'),
    ('results/error_analysis.json', 'artifacts/results/error_analysis.json'),
    ('results/error_analysis_cases.json', 'artifacts/results/error_analysis_cases.json'),
    ('results/latency_analysis.json', 'artifacts/results/latency_analysis.json'),
    ('results/llm_judge_results.json', 'artifacts/results/llm_judge_results.json'),
]
LEGACY_OK = os.path.exists(os.path.join(LEGACY_DIR, 'marker.json'))
if LEGACY_OK:
    restored = 0
    for rel, local in LEGACY_FILES:
        src = os.path.join(LEGACY_DIR, rel)
        if os.path.exists(src):
            os.makedirs(os.path.dirname(local), exist_ok=True)
            shutil.copy(src, local)
            restored += 1
    fig_dir = os.path.join(LEGACY_DIR, 'figures')
    if os.path.isdir(fig_dir):
        os.makedirs('artifacts/figures', exist_ok=True)
        for fn in os.listdir(fig_dir):
            shutil.copy(os.path.join(fig_dir, fn), os.path.join('artifacts/figures', fn))
    print(f'Legacy Version A analyses restored from Drive ({restored} files).')
else:
    print('No legacy cache yet - cells 8-12 will run once.')


In [ ]:
# 8) SHAP explanations + calibration/ROC/PR figures (skips when already present)
import os
if os.path.exists('artifacts/results/shap_summary.json'):
    print('SHAP analysis already present - skipping.')
else:
    rc = os.system('python src/explain/shap_analysis.py')
    if rc != 0:
        raise RuntimeError(f'SHAP analysis failed with exit code {rc}')


In [ ]:
# 9) RAGTruth zero-shot external validation (skips when already present)
# eval_ragtruth.py self-heals its input from the B1 unified dataset
# (cell 7d) - no separate download step needed.
import os
if os.path.exists('artifacts/results/ragtruth_results.json'):
    print('RAGTruth validation already present - skipping.')
else:
    rc = os.system('python src/models/eval_ragtruth.py')
    if rc != 0:
        raise RuntimeError(f'RAGTruth validation failed with exit code {rc}')


In [ ]:
# 10) Error analysis (10 FP + 10 FN, auto-tagged; skips when already present)
# NOTE: categories are heuristic and must be manually reviewed in
# artifacts/results/error_analysis_cases.json before the paper uses them.
import os
if os.path.exists('artifacts/results/error_analysis_cases.json'):
    print('Error analysis already present - skipping.')
else:
    rc = os.system('python src/models/error_analysis.py')
    if rc != 0:
        raise RuntimeError(f'Error analysis failed with exit code {rc}')


In [ ]:
# 11) Latency / efficiency analysis (skips when already present)
import os
if os.path.exists('artifacts/results/latency_analysis.json'):
    print('Latency analysis already present - skipping.')
else:
    rc = os.system('python src/models/eval_efficiency.py')
    if rc != 0:
        raise RuntimeError(f'Latency analysis failed with exit code {rc}')


In [ ]:
# 12) OPTIONAL: LLM-as-judge comparison (skips when already present or no key)
import os
if os.path.exists('artifacts/results/llm_judge_results.json'):
    print('LLM judge results already present - skipping.')
elif os.environ.get('OPENAI_API_KEY'):
    os.system('python src/models/eval_llm_judge.py')
else:
    print('OPENAI_API_KEY not set - skipping optional LLM-as-judge. You can run it later locally.')


In [ ]:
# 12.5) Checkpoint legacy Version A analyses (cells 8-12) to Drive
import os, json, shutil, datetime
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
LEGACY_DIR = os.path.join(DRIVE_CACHE, 'version_a', 'legacy')
LEGACY_FILES = [
    ('models/shap_explainer.joblib', 'artifacts/models/shap_explainer.joblib'),
    ('results/shap_summary.json', 'artifacts/results/shap_summary.json'),
    ('results/ragtruth_results.json', 'artifacts/results/ragtruth_results.json'),
    ('results/error_analysis.json', 'artifacts/results/error_analysis.json'),
    ('results/error_analysis_cases.json', 'artifacts/results/error_analysis_cases.json'),
    ('results/latency_analysis.json', 'artifacts/results/latency_analysis.json'),
    ('results/llm_judge_results.json', 'artifacts/results/llm_judge_results.json'),
]
os.makedirs(LEGACY_DIR, exist_ok=True)
saved = []
for rel, local in LEGACY_FILES:
    if os.path.exists(local):
        dest = os.path.join(LEGACY_DIR, rel)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy(local, dest)
        saved.append(rel)
fig_dir = os.path.join(LEGACY_DIR, 'figures')
os.makedirs(fig_dir, exist_ok=True)
if os.path.isdir('artifacts/figures'):
    for fn in sorted(os.listdir('artifacts/figures')):
        if fn.startswith('fig_') and os.path.isfile(os.path.join('artifacts/figures', fn)):
            shutil.copy(os.path.join('artifacts/figures', fn), os.path.join(fig_dir, fn))
with open(os.path.join(LEGACY_DIR, 'marker.json'), 'w', encoding='utf-8') as f:
    json.dump({'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
               'files': saved}, f, indent=2)
print(f'Checkpointed legacy Version A analyses to Drive ({len(saved)} files + figures).')


In [ ]:
# 13) Generate the artifact manifest (hashes, versions, hardware, split report)
!python src/models/make_manifest.py
import json
m = json.load(open('artifacts/results/manifest.json'))
print(json.dumps({k: m[k] for k in ['generated_at', 'git_commit', 'model_version', 'nli_model', 'split_report']}, indent=2))


In [ ]:
# 14) Show the headline results (model comparison, calibration, RAGTruth, per-seed rows)
import json, os, pandas as pd
res = json.load(open('artifacts/results/final_results.json'))
rows = {k: v for k, v in res.items() if isinstance(v, dict) and 'f1' in v and 'auroc' in v}
df = pd.DataFrame(rows).T[['precision','recall','f1','auroc','pr_auc','mcc']].round(4)
print('MODEL COMPARISON (test set, mean over seeds 42/123/456)')
print(df.to_string())
print('\nCALIBRATION:')
print(json.dumps(res['calibration'], indent=2))
print('\nABLATION:')
print(pd.DataFrame(res['ablation']).to_string(index=False))
print('\nPER-SEED XGBOOST:')
print(pd.DataFrame(json.load(open('artifacts/results/seed_metrics.json'))['xgboost']).round(4).to_string(index=False))
if os.path.exists('artifacts/results/ragtruth_results.json'):
    rt = json.load(open('artifacts/results/ragtruth_results.json'))
    print('\nRAGTRUTH ZERO-SHOT: f1=%.4f auroc=%.4f ece=%.4f (n=%d)' % (rt['f1'], rt['auroc'], rt['ece'], rt['n_samples']))
else:
    print('\nRAGTRUTH ZERO-SHOT: skipped (cell 9 not run) - not required for the B-run.')


In [ ]:
# 15) Package artifacts to Drive (persists across sessions) and offer a download link
import os, zipfile
from datetime import date
from google.colab import files

stamp = date.today().isoformat()
zip_path = f'{DRIVE_DIR}/halurisc_artifacts_{stamp}.zip'
EXCLUDE_DIRS = {'_stages', '__pycache__'}  # internal checkpoints never ship
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, fnames in os.walk('artifacts'):
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
        for fn in fnames:
            p = os.path.join(root, fn)
            z.write(p, os.path.relpath(p, '.'))
    for fn in ['data/processed/features_full.parquet', 'data/processed/qa_clean.parquet',
               'data/processed/nli_model_used.json', 'data/processed/audit_50_samples.json']:
        if os.path.exists(fn):
            z.write(fn)
print('Saved:', zip_path, f'({os.path.getsize(zip_path)/1e6:.1f} MB)')
print()
print('NEXT: download the zip from your Drive, unzip at the repo root of your laptop.')
print('The API (uvicorn) and web dashboard will then load the corrected artifacts.')
files.download(zip_path)
